## 1. 01 load and check raw data

Source: `01_load_and_check_raw_data.ipynb`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import gc

warnings.filterwarnings("ignore")

# ============================================================
# 1단계: 2023~2025 원천 대여 이력 점검 + parquet 변환
# - notebook 폴더가 아니라 project_2 루트를 기준으로 잡음
# - data/raw/bike_history/2023~2025/*.csv 만 자동 탐색
# - 월별 CSV를 가볍게 parquet으로 변환
# ============================================================

# ------------------------------------------------------------
# 0. 프로젝트 루트 자동 설정
# ------------------------------------------------------------
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name.lower() == "notebook":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
BIKE_RAW_DIR = RAW_DIR / "bike_history"

INTERIM_DIR = DATA_DIR / "interim_2"
RAW_PARQUET_DIR = INTERIM_DIR / "raw_parquet"
BIKE_PARQUET_DIR = RAW_PARQUET_DIR / "bike_history"

TARGET_YEARS = [2023, 2024, 2025]

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
RAW_PARQUET_DIR.mkdir(parents=True, exist_ok=True)
BIKE_PARQUET_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("[경로 확인]")
print("=" * 100)
print("현재 노트북 위치:", CURRENT_DIR)
print("프로젝트 루트:", PROJECT_ROOT)
print("DATA_DIR:", DATA_DIR, "| exists:", DATA_DIR.exists())
print("RAW_DIR:", RAW_DIR, "| exists:", RAW_DIR.exists())
print("BIKE_RAW_DIR:", BIKE_RAW_DIR, "| exists:", BIKE_RAW_DIR.exists())
print("BIKE_PARQUET_DIR:", BIKE_PARQUET_DIR)
print("대상 연도:", TARGET_YEARS)


# ------------------------------------------------------------
# 1. 유틸 함수
# ------------------------------------------------------------
def print_section(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def get_file_size_mb(path):
    path = Path(path)
    if not path.exists():
        return np.nan
    return path.stat().st_size / 1024 / 1024


def load_csv_safely(path, usecols=None, nrows=None):
    path = Path(path)
    try:
        return pd.read_csv(path, encoding="utf-8", usecols=usecols, nrows=nrows, low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp949", usecols=usecols, nrows=nrows, low_memory=False)


def normalize_station_id(series):
    return pd.to_numeric(
        series.astype(str)
              .str.strip()
              .str.replace(".0", "", regex=False),
        errors="coerce"
    ).astype("Int64")


def guess_col(columns, keywords, prefer_keywords=None):
    cols = list(columns)
    candidates = []

    for col in cols:
        col_lower = str(col).lower().replace(" ", "")
        for kw in keywords:
            kw_lower = str(kw).lower().replace(" ", "")
            if kw_lower in col_lower:
                candidates.append(col)
                break

    if prefer_keywords:
        for pref in prefer_keywords:
            pref_lower = str(pref).lower().replace(" ", "")
            for col in candidates:
                col_lower = str(col).lower().replace(" ", "")
                if pref_lower in col_lower:
                    return col, candidates

    selected = candidates[0] if candidates else None
    return selected, candidates


def inspect_csv_columns(path):
    sample = load_csv_safely(path, nrows=5)
    return list(sample.columns), sample


def extract_year_from_path(path):
    path = Path(path)
    for part in path.parts:
        if str(part).isdigit() and len(str(part)) == 4:
            return int(part)
    return None


# ------------------------------------------------------------
# 2. data/raw 파일 구조 확인
# ------------------------------------------------------------
print_section("data/raw 파일 구조 확인")

if not RAW_DIR.exists():
    raise FileNotFoundError(f"RAW_DIR이 없습니다: {RAW_DIR}")

all_raw_files = [p for p in RAW_DIR.rglob("*") if p.is_file()]
print("raw 파일 수:", len(all_raw_files))

for p in all_raw_files[:150]:
    print(p.relative_to(PROJECT_ROOT))

if len(all_raw_files) > 150:
    print("... 생략:", len(all_raw_files) - 150)


# ------------------------------------------------------------
# 3. bike_history 2023~2025 CSV 자동 탐색
# ------------------------------------------------------------
print_section("bike_history 2023~2025 CSV 파일 탐색")

if not BIKE_RAW_DIR.exists():
    raise FileNotFoundError(f"BIKE_RAW_DIR이 없습니다: {BIKE_RAW_DIR}")

all_bike_csv_files = sorted(BIKE_RAW_DIR.rglob("*.csv"))

bike_csv_files = []
for p in all_bike_csv_files:
    y = extract_year_from_path(p)
    if y in TARGET_YEARS:
        bike_csv_files.append(p)

print("전체 bike_history csv 파일 수:", len(all_bike_csv_files))
print("대상 2023~2025 csv 파일 수:", len(bike_csv_files))

if len(bike_csv_files) == 0:
    print("2023~2025 CSV를 찾지 못했습니다.")
    print("확인 경로:", BIKE_RAW_DIR)
    raise FileNotFoundError("data/raw/bike_history/2023~2025/*.csv 구조를 확인해주세요.")

for p in bike_csv_files:
    print(p.relative_to(PROJECT_ROOT))


# ------------------------------------------------------------
# 4. 첫 번째 대상 bike 파일로 컬럼 확인
# ------------------------------------------------------------
print_section("bike_history 컬럼 확인")

first_bike_file = bike_csv_files[0]
bike_columns, bike_sample = inspect_csv_columns(first_bike_file)

print("첫 번째 대상 파일:", first_bike_file.relative_to(PROJECT_ROOT))
print("\n컬럼 목록:")
for c in bike_columns:
    print(" -", c)

print("\n샘플:")
display(bike_sample)

bike_station_col, bike_station_candidates = guess_col(
    bike_columns,
    keywords=[
        "대여대여소번호",
        "대여 대여소번호",
        "대여소번호",
        "station",
        "station_id"
    ],
    prefer_keywords=[
        "대여대여소번호",
        "대여 대여소번호"
    ]
)

bike_datetime_col, bike_datetime_candidates = guess_col(
    bike_columns,
    keywords=[
        "대여일시",
        "대여 일시",
        "rental_datetime",
        "datetime",
        "일시"
    ],
    prefer_keywords=[
        "대여일시",
        "대여 일시"
    ]
)

print("\n대여소번호 후보:", bike_station_candidates)
print("선택된 대여소번호 컬럼:", bike_station_col)

print("\n대여일시 후보:", bike_datetime_candidates)
print("선택된 대여일시 컬럼:", bike_datetime_col)

if bike_station_col is None or bike_datetime_col is None:
    raise ValueError(
        "대여소번호 또는 대여일시 컬럼을 자동으로 찾지 못했습니다. "
        "위 컬럼 목록을 보고 bike_station_col, bike_datetime_col을 직접 지정해야 합니다."
    )


# ------------------------------------------------------------
# 5. 기존 2023~2025 parquet 결과가 있으면 덮어쓰기
# ------------------------------------------------------------
print_section("기존 2023~2025 parquet 정리")

for year in TARGET_YEARS:
    year_dir = BIKE_PARQUET_DIR / str(year)
    year_dir.mkdir(parents=True, exist_ok=True)

    old_files = list(year_dir.glob("*.parquet"))
    print(f"{year} 기존 parquet 파일 수:", len(old_files))

    for old_file in old_files:
        old_file.unlink()

print("기존 대상 연도 parquet 파일 정리 완료")


# ------------------------------------------------------------
# 6. 2023~2025 bike_history 월별 CSV -> parquet 변환
# ------------------------------------------------------------
print_section("2023~2025 bike_history CSV -> parquet 변환 시작")

bike_file_summaries = []
all_station_ids = set()

total_rows_raw = 0
total_rows_clean = 0
total_station_id_missing = 0
total_datetime_missing = 0

global_min_datetime = None
global_max_datetime = None
year_row_counts = {}

for i, csv_path in enumerate(bike_csv_files, start=1):
    year = extract_year_from_path(csv_path)
    out_dir = BIKE_PARQUET_DIR / str(year)
    out_dir.mkdir(parents=True, exist_ok=True)

    out_path = out_dir / (csv_path.stem + ".parquet")

    print(f"\n[{i}/{len(bike_csv_files)}] 처리 중:", csv_path.relative_to(PROJECT_ROOT))

    raw_part = load_csv_safely(
        csv_path,
        usecols=[bike_station_col, bike_datetime_col]
    )

    raw_rows = len(raw_part)
    total_rows_raw += raw_rows

    part = raw_part.rename(columns={
        bike_station_col: "station_id_raw",
        bike_datetime_col: "rental_datetime_raw"
    }).copy()

    part["station_id"] = normalize_station_id(part["station_id_raw"])
    part["datetime"] = pd.to_datetime(part["rental_datetime_raw"], errors="coerce")
    part["datetime"] = part["datetime"].dt.floor("h")

    station_id_missing = part["station_id"].isna().sum()
    datetime_missing = part["datetime"].isna().sum()

    total_station_id_missing += int(station_id_missing)
    total_datetime_missing += int(datetime_missing)

    part_clean = part.dropna(subset=["station_id", "datetime"]).copy()

    # 혹시 파일 안에 다른 연도가 섞여 있어도 2023~2025만 유지
    before_year_filter = len(part_clean)
    part_clean = part_clean[
        part_clean["datetime"].dt.year.isin(TARGET_YEARS)
    ].copy()
    after_year_filter = len(part_clean)

    part_clean = part_clean[["station_id", "datetime"]].copy()
    part_clean["station_id"] = part_clean["station_id"].astype("Int64")

    clean_rows = len(part_clean)
    total_rows_clean += clean_rows

    if clean_rows > 0:
        min_dt = part_clean["datetime"].min()
        max_dt = part_clean["datetime"].max()

        if global_min_datetime is None or min_dt < global_min_datetime:
            global_min_datetime = min_dt

        if global_max_datetime is None or max_dt > global_max_datetime:
            global_max_datetime = max_dt

        all_station_ids.update(part_clean["station_id"].dropna().unique().tolist())

        vc = part_clean["datetime"].dt.year.value_counts().to_dict()
        for y, cnt in vc.items():
            year_row_counts[y] = year_row_counts.get(y, 0) + int(cnt)

    part_clean.to_parquet(out_path, index=False)

    csv_size = get_file_size_mb(csv_path)
    parquet_size = get_file_size_mb(out_path)

    bike_file_summaries.append({
        "year_folder": year,
        "csv_path": str(csv_path.relative_to(PROJECT_ROOT)),
        "parquet_path": str(out_path.relative_to(PROJECT_ROOT)),
        "raw_rows": raw_rows,
        "clean_rows": clean_rows,
        "removed_missing_rows": raw_rows - before_year_filter,
        "removed_out_of_target_year_rows": before_year_filter - after_year_filter,
        "station_id_missing": int(station_id_missing),
        "datetime_missing": int(datetime_missing),
        "min_datetime": part_clean["datetime"].min() if clean_rows > 0 else pd.NaT,
        "max_datetime": part_clean["datetime"].max() if clean_rows > 0 else pd.NaT,
        "station_nunique": part_clean["station_id"].nunique() if clean_rows > 0 else 0,
        "csv_size_mb": csv_size,
        "parquet_size_mb": parquet_size,
        "parquet_over_csv_ratio": parquet_size / csv_size if csv_size > 0 else np.nan
    })

    print("raw rows:", raw_rows)
    print("clean rows:", clean_rows)
    print("removed missing rows:", raw_rows - before_year_filter)
    print("removed out-of-target-year rows:", before_year_filter - after_year_filter)
    print("station_id missing:", station_id_missing)
    print("datetime missing:", datetime_missing)
    print("datetime range:", part_clean["datetime"].min() if clean_rows > 0 else None, "~", part_clean["datetime"].max() if clean_rows > 0 else None)
    print("station nunique:", part_clean["station_id"].nunique() if clean_rows > 0 else 0)
    print("csv MB:", round(csv_size, 2), "| parquet MB:", round(parquet_size, 2))

    del raw_part, part, part_clean
    gc.collect()


# ------------------------------------------------------------
# 7. 변환 요약 저장
# ------------------------------------------------------------
print_section("2023~2025 bike_history parquet 변환 요약")

bike_summary_df = pd.DataFrame(bike_file_summaries)
bike_summary_path = INTERIM_DIR / "step1_bike_history_2023_2025_file_summary.csv"
bike_summary_df.to_csv(bike_summary_path, index=False, encoding="utf-8-sig")

display(bike_summary_df)

csv_total_mb = bike_summary_df["csv_size_mb"].sum()
parquet_total_mb = bike_summary_df["parquet_size_mb"].sum()
parquet_ratio = parquet_total_mb / csv_total_mb if csv_total_mb > 0 else np.nan

print("요약 저장:", bike_summary_path)
print("총 raw rows:", total_rows_raw)
print("총 clean rows:", total_rows_clean)
print("총 제거 rows:", total_rows_raw - total_rows_clean)
print("총 station_id 결측:", total_station_id_missing)
print("총 datetime 결측:", total_datetime_missing)
print("전체 datetime min:", global_min_datetime)
print("전체 datetime max:", global_max_datetime)
print("전체 station 수:", len(all_station_ids))
print("station_id 11 포함 여부:", 11 in all_station_ids)

print("\n연도별 row 수:")
print(pd.Series(year_row_counts).sort_index())

print("\n파일 크기 합계")
print("CSV total MB:", round(csv_total_mb, 2))
print("Parquet total MB:", round(parquet_total_mb, 2))
print("Parquet / CSV ratio:", round(parquet_ratio, 4))


# ------------------------------------------------------------
# 8. parquet 변환본 재검증
# ------------------------------------------------------------
print_section("parquet 변환본 재검증")

bike_parquet_files = []

for year in TARGET_YEARS:
    bike_parquet_files.extend(sorted((BIKE_PARQUET_DIR / str(year)).glob("*.parquet")))

print("대상 parquet 파일 수:", len(bike_parquet_files))

check_rows = 0
check_station_ids = set()
check_min_dt = None
check_max_dt = None
check_year_counts = {}

for p in bike_parquet_files:
    df_part = pd.read_parquet(p)

    check_rows += len(df_part)
    check_station_ids.update(df_part["station_id"].dropna().unique().tolist())

    if len(df_part) > 0:
        min_dt = df_part["datetime"].min()
        max_dt = df_part["datetime"].max()

        if check_min_dt is None or min_dt < check_min_dt:
            check_min_dt = min_dt

        if check_max_dt is None or max_dt > check_max_dt:
            check_max_dt = max_dt

        vc = df_part["datetime"].dt.year.value_counts().to_dict()
        for y, cnt in vc.items():
            check_year_counts[y] = check_year_counts.get(y, 0) + int(cnt)

    del df_part
    gc.collect()

print("parquet 총 rows:", check_rows)
print("parquet station 수:", len(check_station_ids))
print("parquet 기간:", check_min_dt, "~", check_max_dt)
print("parquet 연도별 row 수:")
print(pd.Series(check_year_counts).sort_index())
print("station_id 11 포함 여부:", 11 in check_station_ids)

assert check_rows == total_rows_clean, "저장된 parquet row 수와 clean row 수가 다릅니다."
assert set(check_year_counts.keys()).issubset(set(TARGET_YEARS)), "대상 연도 외 데이터가 parquet에 포함되어 있습니다."


# ------------------------------------------------------------
# 9. station / weather / holiday 후보 탐색
# ------------------------------------------------------------
print_section("station / weather / holiday 파일 후보 탐색")

all_non_bike_files = [
    p for p in RAW_DIR.rglob("*")
    if p.is_file() and "bike_history" not in str(p).replace("\\", "/")
]

print("bike_history 제외 raw 파일 수:", len(all_non_bike_files))

for p in all_non_bike_files[:200]:
    print(p.relative_to(PROJECT_ROOT))

def find_candidates_by_name(files, keywords):
    result = []
    for p in files:
        name = p.name.lower()
        path_text = str(p).lower()
        if any(kw.lower() in name or kw.lower() in path_text for kw in keywords):
            result.append(p)
    return result

station_candidates = find_candidates_by_name(
    all_non_bike_files,
    ["station", "대여소", "공공자전거 대여소", "bike_station"]
)

weather_candidates = find_candidates_by_name(
    all_non_bike_files,
    ["weather", "날씨", "기상", "종관", "asos"]
)

holiday_candidates = find_candidates_by_name(
    all_non_bike_files,
    ["holiday", "공휴", "휴일", "calendar"]
)

print("\n[station 후보]")
for p in station_candidates[:50]:
    print(p.relative_to(PROJECT_ROOT))

print("\n[weather 후보]")
for p in weather_candidates[:50]:
    print(p.relative_to(PROJECT_ROOT))

print("\n[holiday 후보]")
for p in holiday_candidates[:50]:
    print(p.relative_to(PROJECT_ROOT))


# ------------------------------------------------------------
# 10. 1단계 최종 요약 저장
# ------------------------------------------------------------
print_section("1단계 최종 요약")

summary = {
    "project_root": str(PROJECT_ROOT),
    "target_years": TARGET_YEARS,
    "bike_raw_dir": str(BIKE_RAW_DIR),
    "bike_csv_file_count_2023_2025": len(bike_csv_files),
    "bike_parquet_file_count_2023_2025": len(bike_parquet_files),
    "bike_total_raw_rows": int(total_rows_raw),
    "bike_total_clean_rows": int(total_rows_clean),
    "bike_removed_rows": int(total_rows_raw - total_rows_clean),
    "bike_station_id_missing": int(total_station_id_missing),
    "bike_datetime_missing": int(total_datetime_missing),
    "bike_datetime_min": str(global_min_datetime),
    "bike_datetime_max": str(global_max_datetime),
    "bike_station_nunique": int(len(all_station_ids)),
    "station_id_11_in_bike": bool(11 in all_station_ids),
    "bike_csv_total_mb": float(csv_total_mb),
    "bike_parquet_total_mb": float(parquet_total_mb),
    "bike_parquet_over_csv_ratio": float(parquet_ratio),
    "station_candidate_count": int(len(station_candidates)),
    "weather_candidate_count": int(len(weather_candidates)),
    "holiday_candidate_count": int(len(holiday_candidates)),
    "bike_summary_path": str(bike_summary_path),
    "bike_parquet_dir": str(BIKE_PARQUET_DIR),
}

for k, v in summary.items():
    print(f"{k}: {v}")

summary_path = INTERIM_DIR / "step1_summary_2023_2025.txt"
with open(summary_path, "w", encoding="utf-8") as f:
    for k, v in summary.items():
        f.write(f"{k}: {v}\n")

print("\n1단계 요약 저장:", summary_path)

print("\n[다음 단계]")
print("1. 2023~2025 대여 이력 parquet 변환이 완료되었습니다.")
print("2. 다음 단계에서는 station master를 정제하고, 2023~2025 대여 이력 station_id와 정합성을 확인합니다.")
print("3. station_id 11 문제가 station master 누락인지 merge 문제인지 다음 단계에서 확인합니다.")

## 2. 02 clean station master

Source: `02_clean_station_master.ipynb`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import gc

warnings.filterwarnings("ignore")

# ============================================================
# 2단계: station_info.xlsx 정제 + station_id 정합성 검증
# 목적:
# - 모델에 필요한 station 컬럼만 추출
# - station_id / latitude / longitude / rack_count / install_date 정제
# - 2023~2025 bike parquet에 등장한 station_id와 정합성 검증
# - 최종 모델 학습에 부적합한 station을 사전에 식별
#
# 최종 모델용 station 핵심 컬럼:
# - station_id
# - latitude
# - longitude
# - rack_count
# - install_date  -> 이후 station_age_days 계산
#
# 검증/보고서용 보조 컬럼:
# - station_name
# - district
# ============================================================


# ------------------------------------------------------------
# 0. 경로 설정
# ------------------------------------------------------------
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebook" else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"

INTERIM_DIR = DATA_DIR / "interim_2"
RAW_PARQUET_DIR = INTERIM_DIR / "raw_parquet"
BIKE_PARQUET_DIR = RAW_PARQUET_DIR / "bike_history"

PROCESSED_DIR = DATA_DIR / "processed_2"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

TARGET_YEARS = [2023, 2024, 2025]

STATION_PATH = RAW_DIR / "station" / "station_info.xlsx"

STATION_OUTPUT_PATH = PROCESSED_DIR / "station_verified.parquet"
STATION_CONSISTENCY_REPORT_PATH = INTERIM_DIR / "step2_station_consistency_report.csv"
STATION_QUALITY_REPORT_PATH = INTERIM_DIR / "step2_station_quality_report.csv"
STATION_SUMMARY_PATH = INTERIM_DIR / "step2_station_summary.txt"

print("=" * 100)
print("[2단계 경로 확인]")
print("=" * 100)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("STATION_PATH:", STATION_PATH, "| exists:", STATION_PATH.exists())
print("BIKE_PARQUET_DIR:", BIKE_PARQUET_DIR, "| exists:", BIKE_PARQUET_DIR.exists())
print("STATION_OUTPUT_PATH:", STATION_OUTPUT_PATH)

if not STATION_PATH.exists():
    raise FileNotFoundError(f"station_info.xlsx 파일이 없습니다: {STATION_PATH}")

if not BIKE_PARQUET_DIR.exists():
    raise FileNotFoundError(f"1단계 bike parquet 폴더가 없습니다: {BIKE_PARQUET_DIR}")


# ------------------------------------------------------------
# 1. 유틸 함수
# ------------------------------------------------------------
def print_section(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def get_file_size_mb(path):
    path = Path(path)
    if not path.exists():
        return np.nan
    return path.stat().st_size / 1024 / 1024


def normalize_station_id(series):
    """
    station_id를 Int64로 통일.
    예: '00102', '102', '102.0', ' 102 ' -> 102
    """
    return pd.to_numeric(
        series.astype(str)
              .str.strip()
              .str.replace(".0", "", regex=False),
        errors="coerce"
    ).astype("Int64")


def normalize_text(series):
    out = series.astype(str).str.strip()
    out = out.replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
    return out


def check_missing(df, name, top_n=30):
    print_section(f"[{name}] 결측 확인")
    result = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_rate_percent": (df.isna().mean() * 100).round(4)
    }).sort_values("missing_rate_percent", ascending=False)
    display(result.head(top_n))
    return result


def safe_merge(left, right, on, how="left", validate=None, name="merge"):
    print_section(f"[{name}] merge 검증")

    before = len(left)

    result = left.merge(
        right,
        on=on,
        how=how,
        validate=validate
    )

    after = len(result)

    print("before rows:", before)
    print("after rows:", after)
    print("row diff:", after - before)
    print("how:", how)
    print("validate:", validate)

    if how == "left" and before != after:
        raise ValueError(f"{name}: left merge인데 row 수가 변했습니다. before={before}, after={after}")

    return result


# ------------------------------------------------------------
# 2. station_info.xlsx 로드
# ------------------------------------------------------------
print_section("station_info.xlsx 원본 로드")

SHEET_NAME = "대여소현황"

station_raw_no_header = pd.read_excel(
    STATION_PATH,
    sheet_name=SHEET_NAME,
    header=None
)

print("station_raw_no_header shape:", station_raw_no_header.shape)
display(station_raw_no_header.head(10))


# ------------------------------------------------------------
# 3. 실제 데이터 시작 행 자동 탐색
# ------------------------------------------------------------
# 엑셀 구조 기준:
# A열: 대여소번호
# B열: 대여소명
# C열: 자치구
# D열: 상세주소
# E열: 위도
# F열: 경도
# G열: 설치시기
# H열: LCD 거치대수
# I열: QR 거치대수
# J열: 운영방식
#
# 모델에는 station_id, latitude, longitude, rack_count, install_date만 사용.
# station_name, district는 검증/보고서용으로만 보관.

print_section("station_info 실제 데이터 시작 행 탐색")

col_station_id_idx = 0
col_lat_idx = 4
col_lon_idx = 5

station_id_check = pd.to_numeric(station_raw_no_header.iloc[:, col_station_id_idx], errors="coerce")
lat_check = pd.to_numeric(station_raw_no_header.iloc[:, col_lat_idx], errors="coerce")
lon_check = pd.to_numeric(station_raw_no_header.iloc[:, col_lon_idx], errors="coerce")

data_row_mask = (
    station_id_check.notna() &
    lat_check.between(37.3, 37.8) &
    lon_check.between(126.7, 127.3)
)

data_start_candidates = station_raw_no_header.index[data_row_mask].tolist()

if len(data_start_candidates) == 0:
    raise ValueError(
        "station_info.xlsx에서 실제 데이터 시작 행을 찾지 못했습니다. "
        "A열 station_id, E열 위도, F열 경도 구조가 맞는지 확인하세요."
    )

data_start_idx = data_start_candidates[0]

print("실제 데이터 시작 row index:", data_start_idx)
print("시작 행 미리보기:")
display(station_raw_no_header.iloc[data_start_idx:data_start_idx + 5, :10])


# ------------------------------------------------------------
# 4. 필요한 컬럼만 위치 기반으로 추출
# ------------------------------------------------------------
print_section("station_info 필요한 컬럼 추출")

station_data = station_raw_no_header.iloc[data_start_idx:, :10].copy()

station_data.columns = [
    "station_id_raw",
    "station_name_raw",
    "district_raw",
    "address_raw",
    "latitude_raw",
    "longitude_raw",
    "install_date_raw",
    "lcd_rack_count_raw",
    "qr_rack_count_raw",
    "operation_type_raw"
]

station = pd.DataFrame()

# 필수 key
station["station_id"] = normalize_station_id(station_data["station_id_raw"])

# 검증/보고서용
station["station_name"] = normalize_text(station_data["station_name_raw"])
station["district"] = normalize_text(station_data["district_raw"])

# 모델용 핵심 feature
station["latitude"] = pd.to_numeric(station_data["latitude_raw"], errors="coerce")
station["longitude"] = pd.to_numeric(station_data["longitude_raw"], errors="coerce")
station["install_date"] = pd.to_datetime(station_data["install_date_raw"], errors="coerce")

# 거치대 수:
# - LCD/QR 두 컬럼 중 값이 있는 것을 사용
# - 둘 다 있으면 max 사용
# - 최종 모델에는 rack_count 하나만 사용
station["lcd_rack_count"] = pd.to_numeric(station_data["lcd_rack_count_raw"], errors="coerce")
station["qr_rack_count"] = pd.to_numeric(station_data["qr_rack_count_raw"], errors="coerce")
station["rack_count"] = station[["lcd_rack_count", "qr_rack_count"]].max(axis=1)

print("station 추출 후 shape:", station.shape)
display(station.head(20))

check_missing(station, "station_extracted", top_n=20)


# ------------------------------------------------------------
# 5. station_id 결측 제거
# ------------------------------------------------------------
print_section("station_id 결측 제거")

before_station_id_drop = len(station)

station_id_missing_df = station[station["station_id"].isna()].copy()
station = station.dropna(subset=["station_id"]).copy()

after_station_id_drop = len(station)

print("제거 전:", before_station_id_drop)
print("제거 후:", after_station_id_drop)
print("station_id 결측 제거 row:", before_station_id_drop - after_station_id_drop)
print("station_id unique 수:", station["station_id"].nunique())

if len(station_id_missing_df) > 0:
    print("station_id 결측 제거 예시:")
    display(station_id_missing_df.head(30))


# ------------------------------------------------------------
# 6. station_id 중복 정리
# ------------------------------------------------------------
print_section("station_id 중복 확인 및 대표 행 선택")

duplicate_before_count = station["station_id"].duplicated().sum()
print("station_id 중복 row 수:", duplicate_before_count)

if duplicate_before_count > 0:
    print("중복 station 예시:")
    display(
        station[station["station_id"].duplicated(keep=False)]
        .sort_values("station_id")
        .head(100)
    )

# 대표 행 선택 기준:
# 1. latitude 있음
# 2. longitude 있음
# 3. rack_count 있음
# 4. install_date 있음
# 5. station_name 있음
# 평가표용 설명 가능 기준: "정보 완성도가 가장 높은 행 선택"
station["valid_score"] = 0
station["valid_score"] += station["latitude"].notna().astype(int)
station["valid_score"] += station["longitude"].notna().astype(int)
station["valid_score"] += station["rack_count"].notna().astype(int)
station["valid_score"] += station["install_date"].notna().astype(int)
station["valid_score"] += station["station_name"].notna().astype(int)

station = (
    station
    .sort_values(
        ["station_id", "valid_score", "install_date"],
        ascending=[True, False, False],
        na_position="last"
    )
    .drop_duplicates("station_id", keep="first")
    .drop(columns=["valid_score"])
    .copy()
)

duplicate_after_count = station["station_id"].duplicated().sum()

print("중복 정리 후 shape:", station.shape)
print("남은 station_id 중복 수:", duplicate_after_count)

if duplicate_after_count > 0:
    raise ValueError("station_id 중복 제거 후에도 중복이 남아 있습니다.")


# ------------------------------------------------------------
# 7. station 품질 플래그 생성
# ------------------------------------------------------------
print_section("station 품질 플래그 생성")

station_quality = station.copy()

station_quality["missing_latitude"] = station_quality["latitude"].isna()
station_quality["missing_longitude"] = station_quality["longitude"].isna()
station_quality["missing_location"] = (
    station_quality["missing_latitude"] |
    station_quality["missing_longitude"]
)

station_quality["invalid_location_range"] = (
    station_quality["latitude"].notna() &
    station_quality["longitude"].notna() &
    ~(
        station_quality["latitude"].between(37.3, 37.8) &
        station_quality["longitude"].between(126.7, 127.3)
    )
)

station_quality["missing_rack_count"] = station_quality["rack_count"].isna()

station_quality["invalid_rack_count"] = (
    station_quality["rack_count"].notna() &
    ((station_quality["rack_count"] <= 0) | (station_quality["rack_count"] > 200))
)

station_quality["missing_install_date"] = station_quality["install_date"].isna()

# 최종 모델에서 위치 정보가 없는 station은 제거 대상
# rack_count/install_date는 이후 대체 가능하므로 여기서는 사용 불가 판정에 넣지 않음
station_quality["is_station_usable_for_model"] = (
    (~station_quality["missing_location"]) &
    (~station_quality["invalid_location_range"])
)

print("전체 station 수:", len(station_quality))
print("위치 결측 station 수:", int(station_quality["missing_location"].sum()))
print("서울 범위 밖 위치 station 수:", int(station_quality["invalid_location_range"].sum()))
print("거치대수 결측 station 수:", int(station_quality["missing_rack_count"].sum()))
print("비정상 거치대수 station 수:", int(station_quality["invalid_rack_count"].sum()))
print("설치일 결측 station 수:", int(station_quality["missing_install_date"].sum()))
print("모델 사용 가능 station 수:", int(station_quality["is_station_usable_for_model"].sum()))

print("\n위치/거치대수 문제 station 예시:")
display(
    station_quality[
        station_quality["missing_location"] |
        station_quality["invalid_location_range"] |
        station_quality["invalid_rack_count"]
    ].head(100)
)


# ------------------------------------------------------------
# 8. 2023~2025 bike parquet station_id 수집
# ------------------------------------------------------------
print_section("2023~2025 bike parquet station_id 수집")

bike_parquet_files = []

for year in TARGET_YEARS:
    year_dir = BIKE_PARQUET_DIR / str(year)
    if year_dir.exists():
        bike_parquet_files.extend(sorted(year_dir.glob("*.parquet")))

print("bike parquet 파일 수:", len(bike_parquet_files))

if len(bike_parquet_files) == 0:
    raise FileNotFoundError("1단계에서 생성한 2023~2025 bike parquet 파일이 없습니다.")

bike_station_ids = set()
bike_station_row_counts = {}
bike_total_rows = 0
bike_min_dt = None
bike_max_dt = None
bike_year_counts = {}

for p in bike_parquet_files:
    df_part = pd.read_parquet(p, columns=["station_id", "datetime"])

    bike_total_rows += len(df_part)

    vc_station = df_part["station_id"].value_counts().to_dict()
    for sid, cnt in vc_station.items():
        bike_station_row_counts[sid] = bike_station_row_counts.get(sid, 0) + int(cnt)

    bike_station_ids.update(df_part["station_id"].dropna().unique().tolist())

    if len(df_part) > 0:
        min_dt = df_part["datetime"].min()
        max_dt = df_part["datetime"].max()

        if bike_min_dt is None or min_dt < bike_min_dt:
            bike_min_dt = min_dt

        if bike_max_dt is None or max_dt > bike_max_dt:
            bike_max_dt = max_dt

        vc_year = df_part["datetime"].dt.year.value_counts().to_dict()
        for y, cnt in vc_year.items():
            bike_year_counts[y] = bike_year_counts.get(y, 0) + int(cnt)

    del df_part
    gc.collect()

bike_station_usage = pd.DataFrame({
    "station_id": list(bike_station_row_counts.keys()),
    "bike_row_count_2023_2025": list(bike_station_row_counts.values())
})

bike_station_usage["station_id"] = bike_station_usage["station_id"].astype("Int64")

print("bike total rows:", bike_total_rows)
print("bike station unique 수:", len(bike_station_ids))
print("bike 기간:", bike_min_dt, "~", bike_max_dt)
print("bike 연도별 row 수:")
print(pd.Series(bike_year_counts).sort_index())
print("bike에 station_id 11 있음?:", 11 in bike_station_ids)


# ------------------------------------------------------------
# 9. station master vs bike station_id 정합성 확인
# ------------------------------------------------------------
print_section("station master vs bike station_id 정합성 확인")

station_master_ids = set(station["station_id"].dropna().unique().tolist())

missing_in_station = bike_station_ids - station_master_ids
unused_station = station_master_ids - bike_station_ids

print("bike station 수:", len(bike_station_ids))
print("station master station 수:", len(station_master_ids))
print("bike에는 있는데 station master에는 없는 station 수:", len(missing_in_station))
print("station master에는 있는데 bike에는 없는 station 수:", len(unused_station))

print("\nmissing_in_station 예시 100개:")
print(sorted(list(missing_in_station))[:100])

print("\nunused_station 예시 100개:")
print(sorted(list(unused_station))[:100])

print("\nstation_id 11 확인")
print("bike에 11 있음?:", 11 in bike_station_ids)
print("station master에 11 있음?:", 11 in station_master_ids)
print("11이 missing_in_station에 있음?:", 11 in missing_in_station)

if 11 in station_master_ids:
    print("\nstation master station_id 11 정보:")
    display(station[station["station_id"] == 11])


# ------------------------------------------------------------
# 10. station consistency report 생성
# ------------------------------------------------------------
print_section("station consistency report 생성")

bike_station_df = pd.DataFrame({
    "station_id": sorted(list(bike_station_ids))
})

bike_station_df["station_id"] = bike_station_df["station_id"].astype("Int64")

station_report = safe_merge(
    bike_station_df,
    bike_station_usage,
    on="station_id",
    how="left",
    validate="one_to_one",
    name="bike_station_df + bike_station_usage"
)

station_report = safe_merge(
    station_report,
    station_quality,
    on="station_id",
    how="left",
    validate="one_to_one",
    name="station_report + station_quality"
)

station_report["exists_in_station_master"] = (
    station_report["station_name"].notna() |
    station_report["latitude"].notna() |
    station_report["longitude"].notna()
)

station_report["missing_in_station_master"] = ~station_report["exists_in_station_master"]

station_report["missing_location"] = (
    station_report["latitude"].isna() |
    station_report["longitude"].isna()
)

station_report["invalid_location_range"] = (
    station_report["latitude"].notna() &
    station_report["longitude"].notna() &
    ~(
        station_report["latitude"].between(37.3, 37.8) &
        station_report["longitude"].between(126.7, 127.3)
    )
)

station_report["missing_rack_count"] = station_report["rack_count"].isna()

station_report["invalid_rack_count"] = (
    station_report["rack_count"].notna() &
    ((station_report["rack_count"] <= 0) | (station_report["rack_count"] > 200))
)

station_report["missing_install_date"] = station_report["install_date"].isna()

station_report["station_usable_for_final_model"] = (
    (~station_report["missing_in_station_master"]) &
    (~station_report["missing_location"]) &
    (~station_report["invalid_location_range"])
)


def assign_station_issue(row):
    issues = []

    if row["missing_in_station_master"]:
        issues.append("missing_in_station_master")

    if row["missing_location"]:
        issues.append("missing_location")

    if row["invalid_location_range"]:
        issues.append("invalid_location_range")

    if row["missing_rack_count"]:
        issues.append("missing_rack_count")

    if row["invalid_rack_count"]:
        issues.append("invalid_rack_count")

    if row["missing_install_date"]:
        issues.append("missing_install_date")

    return "ok" if not issues else "|".join(issues)


station_report["issue_type"] = station_report.apply(assign_station_issue, axis=1)

station_report.to_csv(STATION_CONSISTENCY_REPORT_PATH, index=False, encoding="utf-8-sig")
station_quality.to_csv(STATION_QUALITY_REPORT_PATH, index=False, encoding="utf-8-sig")

print("station consistency report 저장:", STATION_CONSISTENCY_REPORT_PATH)
print("station quality report 저장:", STATION_QUALITY_REPORT_PATH)

print("\nissue_type 분포:")
print(station_report["issue_type"].value_counts())

print("\n최종 모델 사용 가능 여부:")
print(station_report["station_usable_for_final_model"].value_counts(dropna=False))

problem_station_report = station_report[
    ~station_report["station_usable_for_final_model"]
].copy()

problem_station_row_count = int(problem_station_report["bike_row_count_2023_2025"].fillna(0).sum())

problem_station_row_ratio = (
    problem_station_row_count / bike_total_rows
    if bike_total_rows > 0 else np.nan
)

print("\n문제 station 수:", len(problem_station_report))
print("문제 station의 bike row 수 합계:", problem_station_row_count)
print("전체 bike row 대비 문제 station row 비율:", problem_station_row_ratio)

print("\n문제 station 상위 100개:")
display(
    problem_station_report
    .sort_values("bike_row_count_2023_2025", ascending=False)
    .head(100)
)

print("\nstation_id 11 report:")
display(station_report[station_report["station_id"] == 11])


# ------------------------------------------------------------
# 11. station_verified 저장
# ------------------------------------------------------------
print_section("station_verified 저장")

# station_verified는 최종 merge에 필요한 컬럼 위주로 저장
# station_name, district는 검증/보고서용으로만 유지
station_verified = station[
    [
        "station_id",
        "station_name",
        "district",
        "latitude",
        "longitude",
        "rack_count",
        "install_date"
    ]
].copy()

station_verified["station_id"] = station_verified["station_id"].astype("Int64")
station_verified["latitude"] = station_verified["latitude"].astype("float64")
station_verified["longitude"] = station_verified["longitude"].astype("float64")
station_verified["rack_count"] = station_verified["rack_count"].astype("float64")

# 저장 전 필수 검증
if station_verified["station_id"].isna().sum() > 0:
    raise ValueError("station_verified에 station_id 결측이 남아 있습니다.")

if station_verified["station_id"].duplicated().sum() > 0:
    raise ValueError("station_verified에 station_id 중복이 남아 있습니다.")

station_verified.to_parquet(STATION_OUTPUT_PATH, index=False)

print("저장 완료:", STATION_OUTPUT_PATH)
print("station_verified shape:", station_verified.shape)
print("station_verified size MB:", get_file_size_mb(STATION_OUTPUT_PATH))

display(station_verified.head(20))

check_missing(station_verified, "station_verified", top_n=20)


# ------------------------------------------------------------
# 12. 2단계 최종 요약 저장
# ------------------------------------------------------------
print_section("2단계 최종 요약")

summary = {
    "step": "step2_station_info_validation",
    "purpose": "station_info.xlsx에서 모델에 필요한 station feature 추출 및 bike station_id 정합성 검증",
    "project_root": str(PROJECT_ROOT),
    "station_path": str(STATION_PATH),
    "selected_sheet": str(SHEET_NAME),
    "station_raw_shape": str(station_raw_no_header.shape),
    "station_verified_shape": str(station_verified.shape),

    "station_parse_method": "read_excel_header_none_position_based",
    "data_start_idx": int(data_start_idx),

    "model_station_columns": "station_id, latitude, longitude, rack_count, install_date",
    "report_only_columns": "station_name, district",

    "station_verified_nunique": int(station_verified["station_id"].nunique()),
    "station_duplicate_before_clean": int(duplicate_before_count),
    "station_duplicate_after_clean": int(station_verified["station_id"].duplicated().sum()),
    "station_id_missing_removed_rows": int(before_station_id_drop - after_station_id_drop),

    "station_latitude_missing": int(station_verified["latitude"].isna().sum()),
    "station_longitude_missing": int(station_verified["longitude"].isna().sum()),
    "station_location_missing_count": int(station_quality["missing_location"].sum()),
    "station_invalid_location_count": int(station_quality["invalid_location_range"].sum()),
    "station_rack_count_missing": int(station_verified["rack_count"].isna().sum()),
    "station_invalid_rack_count": int(station_quality["invalid_rack_count"].sum()),
    "station_install_date_missing": int(station_verified["install_date"].isna().sum()),

    "bike_total_rows_2023_2025": int(bike_total_rows),
    "bike_station_nunique_2023_2025": int(len(bike_station_ids)),
    "bike_datetime_min": str(bike_min_dt),
    "bike_datetime_max": str(bike_max_dt),

    "missing_in_station_count": int(len(missing_in_station)),
    "unused_station_count": int(len(unused_station)),

    "problem_station_count_for_final_model": int(len(problem_station_report)),
    "problem_station_bike_row_count": int(problem_station_row_count),
    "problem_station_bike_row_ratio": float(problem_station_row_ratio),

    "station_id_11_in_bike": bool(11 in bike_station_ids),
    "station_id_11_in_station_master": bool(11 in station_master_ids),
    "station_id_11_missing_in_station": bool(11 in missing_in_station),
    "station_id_11_usable_for_final_model": bool(
        station_report.loc[station_report["station_id"] == 11, "station_usable_for_final_model"].iloc[0]
    ) if (station_report["station_id"] == 11).any() else False,

    "station_output_path": str(STATION_OUTPUT_PATH),
    "station_consistency_report_path": str(STATION_CONSISTENCY_REPORT_PATH),
    "station_quality_report_path": str(STATION_QUALITY_REPORT_PATH),

    "preprocessing_report_note": (
        "station_info.xlsx는 병합 셀과 다중 header 구조를 포함하고 있어 header=None으로 로드한 뒤 "
        "엑셀 열 위치를 기준으로 필요한 컬럼만 추출하였다. "
        "모델에는 station_id, latitude, longitude, rack_count, install_date를 사용하고, "
        "station_name과 district는 검증 및 보고서 설명용으로만 유지하였다. "
        "station_id 중복은 정보 완성도 기준으로 대표 행을 선택했으며, "
        "2023~2025 대여 이력에 등장한 station_id와 station master 간 정합성을 검증하였다."
    )
}

for k, v in summary.items():
    print(f"{k}: {v}")

with open(STATION_SUMMARY_PATH, "w", encoding="utf-8") as f:
    for k, v in summary.items():
        f.write(f"{k}: {v}\n")

print("\n2단계 요약 저장:", STATION_SUMMARY_PATH)

print("\n[평가표 반영 포인트]")
print("1. station_info.xlsx의 복잡한 header 구조를 고려하여 필요한 컬럼만 안정적으로 파싱했습니다.")
print("2. station_id를 표준 key로 정의하고, bike history와 station master 정합성을 검증했습니다.")
print("3. 위치 결측/위치 범위 이상/거치대수 이상/설치일 결측을 리포트로 남겼습니다.")
print("4. 최종 모델 학습에 부적합한 station을 사전에 식별했습니다.")
print("5. 최종 학습 데이터에서는 위치 결측/이상 station을 제거하고, rack_count/install_date 관련 파생값은 별도 기준으로 처리합니다.")

## 3. 025 station merge

Source: `025_station_merge.ipynb`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import gc
import re

warnings.filterwarnings("ignore")

# ============================================================
# 2.5단계: station_master.csv로 station_info 누락 station 보완
#
# 목적:
# - station_info.xlsx를 기본 station master로 사용
# - station_info에 없는 bike station_id를 station_master.csv로 위치 보완
# - 최종 학습 데이터에서 latitude/longitude 결측이 생기지 않도록 처리
# - 보완 불가능한 station만 최종 제거 후보로 분리
#
# 입력:
# - data/processed_2/station_verified.parquet
# - data/raw/station/station_master.csv
# - data/interim_2/raw_parquet/bike_history/2023~2025/*.parquet
#
# 출력:
# - data/processed_2/station_verified_enriched.parquet
# - data/interim_2/step25_station_enrichment_report.csv
# - data/interim_2/step25_station_unresolved_report.csv
# - data/interim_2/step25_station_summary.txt
# ============================================================


# ------------------------------------------------------------
# 0. 경로 설정
# ------------------------------------------------------------
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebook" else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim_2"
PROCESSED_DIR = DATA_DIR / "processed_2"

BIKE_PARQUET_DIR = INTERIM_DIR / "raw_parquet" / "bike_history"

STATION_INFO_VERIFIED_PATH = PROCESSED_DIR / "station_verified.parquet"
STATION_MASTER_PATH = RAW_DIR / "station" / "station_master.csv"

STATION_ENRICHED_OUTPUT_PATH = PROCESSED_DIR / "station_verified_enriched.parquet"
STATION_ENRICHMENT_REPORT_PATH = INTERIM_DIR / "step25_station_enrichment_report.csv"
STATION_UNRESOLVED_REPORT_PATH = INTERIM_DIR / "step25_station_unresolved_report.csv"
STATION_SUMMARY_PATH = INTERIM_DIR / "step25_station_summary.txt"

TARGET_YEARS = [2023, 2024, 2025]

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("[2.5단계 경로 확인]")
print("=" * 100)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("STATION_INFO_VERIFIED_PATH:", STATION_INFO_VERIFIED_PATH, "| exists:", STATION_INFO_VERIFIED_PATH.exists())
print("STATION_MASTER_PATH:", STATION_MASTER_PATH, "| exists:", STATION_MASTER_PATH.exists())
print("BIKE_PARQUET_DIR:", BIKE_PARQUET_DIR, "| exists:", BIKE_PARQUET_DIR.exists())
print("STATION_ENRICHED_OUTPUT_PATH:", STATION_ENRICHED_OUTPUT_PATH)

if not STATION_INFO_VERIFIED_PATH.exists():
    raise FileNotFoundError("2단계 결과 station_verified.parquet이 없습니다. 2단계를 먼저 완료하세요.")

if not STATION_MASTER_PATH.exists():
    raise FileNotFoundError("station_master.csv가 없습니다.")

if not BIKE_PARQUET_DIR.exists():
    raise FileNotFoundError("1단계 bike parquet 폴더가 없습니다.")


# ------------------------------------------------------------
# 1. 유틸 함수
# ------------------------------------------------------------
def print_section(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def get_file_size_mb(path):
    path = Path(path)
    if not path.exists():
        return np.nan
    return path.stat().st_size / 1024 / 1024


def normalize_station_id_from_st_code(series):
    """
    station_master.csv의 대여소_ID 예:
    ST-11, ST-999, ST-1234 -> 11, 999, 1234
    """
    s = series.astype(str).str.strip()
    s = s.str.replace("ST-", "", regex=False)
    s = s.str.replace("st-", "", regex=False)
    s = s.str.replace(".0", "", regex=False)
    return pd.to_numeric(s, errors="coerce").astype("Int64")


def normalize_station_id(series):
    return pd.to_numeric(
        series.astype(str).str.strip().str.replace(".0", "", regex=False),
        errors="coerce"
    ).astype("Int64")


def normalize_text(series):
    out = series.astype(str).str.strip()
    out = out.replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
    return out


def load_csv_safely(path):
    try:
        return pd.read_csv(path, encoding="utf-8", low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="cp949", low_memory=False)


def check_missing(df, name, top_n=30):
    print_section(f"[{name}] 결측 확인")
    result = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_rate_percent": (df.isna().mean() * 100).round(4)
    }).sort_values("missing_rate_percent", ascending=False)
    display(result.head(top_n))
    return result


def safe_merge(left, right, on, how="left", validate=None, name="merge"):
    print_section(f"[{name}] merge 검증")

    before = len(left)

    result = left.merge(
        right,
        on=on,
        how=how,
        validate=validate
    )

    after = len(result)

    print("before rows:", before)
    print("after rows:", after)
    print("row diff:", after - before)
    print("how:", how)
    print("validate:", validate)

    if how == "left" and before != after:
        raise ValueError(f"{name}: left merge인데 row 수가 변했습니다. before={before}, after={after}")

    return result


# ------------------------------------------------------------
# 2. station_info 기반 station_verified 로드
# ------------------------------------------------------------
print_section("station_info 기반 station_verified 로드")

station_info = pd.read_parquet(STATION_INFO_VERIFIED_PATH)

print("station_info shape:", station_info.shape)
display(station_info.head())

required_cols = ["station_id", "latitude", "longitude", "rack_count", "install_date"]
missing_cols = [c for c in required_cols if c not in station_info.columns]

if missing_cols:
    raise ValueError(f"station_verified에 필수 컬럼이 없습니다: {missing_cols}")

if station_info["station_id"].isna().sum() > 0:
    raise ValueError("station_info에 station_id 결측이 있습니다.")

if station_info["station_id"].duplicated().sum() > 0:
    raise ValueError("station_info에 station_id 중복이 있습니다.")

station_info["station_id"] = station_info["station_id"].astype("Int64")

station_info_ids = set(station_info["station_id"].dropna().unique().tolist())

print("station_info station 수:", len(station_info_ids))


# ------------------------------------------------------------
# 3. 2023~2025 bike parquet에서 station_id와 사용량 수집
# ------------------------------------------------------------
print_section("2023~2025 bike station_id 수집")

bike_parquet_files = []

for year in TARGET_YEARS:
    year_dir = BIKE_PARQUET_DIR / str(year)
    if year_dir.exists():
        bike_parquet_files.extend(sorted(year_dir.glob("*.parquet")))

print("bike parquet 파일 수:", len(bike_parquet_files))

if len(bike_parquet_files) == 0:
    raise FileNotFoundError("bike parquet 파일이 없습니다. 1단계를 먼저 확인하세요.")

bike_station_row_counts = {}
bike_total_rows = 0
bike_min_dt = None
bike_max_dt = None
bike_year_counts = {}

for p in bike_parquet_files:
    part = pd.read_parquet(p, columns=["station_id", "datetime"])

    bike_total_rows += len(part)

    station_counts = part["station_id"].value_counts().to_dict()
    for sid, cnt in station_counts.items():
        bike_station_row_counts[sid] = bike_station_row_counts.get(sid, 0) + int(cnt)

    if len(part) > 0:
        min_dt = part["datetime"].min()
        max_dt = part["datetime"].max()

        if bike_min_dt is None or min_dt < bike_min_dt:
            bike_min_dt = min_dt

        if bike_max_dt is None or max_dt > bike_max_dt:
            bike_max_dt = max_dt

        vc_year = part["datetime"].dt.year.value_counts().to_dict()
        for y, cnt in vc_year.items():
            bike_year_counts[y] = bike_year_counts.get(y, 0) + int(cnt)

    del part
    gc.collect()

bike_station_usage = pd.DataFrame({
    "station_id": list(bike_station_row_counts.keys()),
    "bike_row_count_2023_2025": list(bike_station_row_counts.values())
})

bike_station_usage["station_id"] = bike_station_usage["station_id"].astype("Int64")

bike_station_ids = set(bike_station_usage["station_id"].dropna().unique().tolist())

print("bike total rows:", bike_total_rows)
print("bike station 수:", len(bike_station_ids))
print("bike 기간:", bike_min_dt, "~", bike_max_dt)
print("bike 연도별 row 수:")
print(pd.Series(bike_year_counts).sort_index())

missing_in_info = bike_station_ids - station_info_ids

print("station_info에 없는 bike station 수:", len(missing_in_info))
print("missing 예시:", sorted(list(missing_in_info))[:100])
print("station_id 11이 missing_in_info인가?:", 11 in missing_in_info)


# ------------------------------------------------------------
# 4. station_master.csv 로드 및 표준화
# ------------------------------------------------------------
print_section("station_master.csv 로드 및 표준화")

station_master_raw = load_csv_safely(STATION_MASTER_PATH)

print("station_master_raw shape:", station_master_raw.shape)
print("columns:", list(station_master_raw.columns))
display(station_master_raw.head())

# 예상 컬럼:
# 대여소_ID, 주소1, 주소2, 위도, 경도
required_master_cols = ["대여소_ID", "위도", "경도"]

missing_master_cols = [c for c in required_master_cols if c not in station_master_raw.columns]

if missing_master_cols:
    raise ValueError(f"station_master.csv에 필수 컬럼이 없습니다: {missing_master_cols}")

station_master = pd.DataFrame()

station_master["station_id"] = normalize_station_id_from_st_code(station_master_raw["대여소_ID"])

if "주소1" in station_master_raw.columns:
    station_master["address1"] = normalize_text(station_master_raw["주소1"])
else:
    station_master["address1"] = pd.NA

if "주소2" in station_master_raw.columns:
    station_master["address2"] = normalize_text(station_master_raw["주소2"])
else:
    station_master["address2"] = pd.NA

station_master["latitude"] = pd.to_numeric(station_master_raw["위도"], errors="coerce")
station_master["longitude"] = pd.to_numeric(station_master_raw["경도"], errors="coerce")

# station_master에는 rack_count/install_date가 없으므로 NaN 유지
station_master["rack_count"] = np.nan
station_master["install_date"] = pd.NaT

# station_name/district도 정확히 없음
# address2를 station_name 후보로 쓰면 대여소명과 정확히 같지 않을 수 있으므로 report용으로만 사용
station_master["station_name"] = station_master["address2"]
station_master["district"] = pd.NA

# 위치 품질 플래그
station_master["missing_location_master"] = (
    station_master["latitude"].isna() |
    station_master["longitude"].isna()
)

station_master["zero_location_master"] = (
    (station_master["latitude"] == 0) |
    (station_master["longitude"] == 0)
)

station_master["valid_location_master"] = (
    station_master["latitude"].between(37.3, 37.8) &
    station_master["longitude"].between(126.7, 127.3)
)

station_master["usable_location_master"] = (
    (~station_master["missing_location_master"]) &
    (~station_master["zero_location_master"]) &
    (station_master["valid_location_master"])
)

print("station_master 표준화 후 shape:", station_master.shape)
display(station_master.head())

check_missing(station_master, "station_master_standardized", top_n=30)

print("\nstation_master 위치 품질:")
print("station_id 결측:", int(station_master["station_id"].isna().sum()))
print("station_id 중복 row:", int(station_master["station_id"].duplicated().sum()))
print("위치 결측:", int(station_master["missing_location_master"].sum()))
print("위도/경도 0:", int(station_master["zero_location_master"].sum()))
print("서울 범위 내 정상 위치:", int(station_master["usable_location_master"].sum()))


# ------------------------------------------------------------
# 5. station_master 중복 정리
# ------------------------------------------------------------
print_section("station_master 중복 정리")

before_master_drop = len(station_master)

station_master = station_master.dropna(subset=["station_id"]).copy()

after_master_drop = len(station_master)

print("station_id 결측 제거 전:", before_master_drop)
print("station_id 결측 제거 후:", after_master_drop)
print("제거 row:", before_master_drop - after_master_drop)

master_dup_before = station_master["station_id"].duplicated().sum()
print("중복 row 수:", master_dup_before)

if master_dup_before > 0:
    print("중복 예시:")
    display(
        station_master[station_master["station_id"].duplicated(keep=False)]
        .sort_values("station_id")
        .head(100)
    )

# 대표 행 선택:
# usable_location_master가 True인 행 우선
station_master["valid_score"] = 0
station_master["valid_score"] += station_master["usable_location_master"].astype(int) * 10
station_master["valid_score"] += station_master["latitude"].notna().astype(int)
station_master["valid_score"] += station_master["longitude"].notna().astype(int)
station_master["valid_score"] += station_master["address1"].notna().astype(int)
station_master["valid_score"] += station_master["address2"].notna().astype(int)

station_master = (
    station_master
    .sort_values(["station_id", "valid_score"], ascending=[True, False])
    .drop_duplicates("station_id", keep="first")
    .drop(columns=["valid_score"])
    .copy()
)

master_dup_after = station_master["station_id"].duplicated().sum()

print("중복 정리 후 shape:", station_master.shape)
print("중복 정리 후 중복 row 수:", master_dup_after)

if master_dup_after > 0:
    raise ValueError("station_master 중복 제거 후에도 station_id 중복이 남아 있습니다.")


# ------------------------------------------------------------
# 6. station_info 누락 station을 station_master로 보완 가능 여부 확인
# ------------------------------------------------------------
print_section("station_info 누락 station 보완 가능 여부 확인")

missing_in_info_df = bike_station_usage[
    bike_station_usage["station_id"].isin(missing_in_info)
].copy()

missing_in_info_df = safe_merge(
    missing_in_info_df,
    station_master[
        [
            "station_id",
            "station_name",
            "district",
            "latitude",
            "longitude",
            "rack_count",
            "install_date",
            "address1",
            "address2",
            "usable_location_master",
            "missing_location_master",
            "zero_location_master",
            "valid_location_master"
        ]
    ],
    on="station_id",
    how="left",
    validate="one_to_one",
    name="missing_in_info + station_master"
)

missing_in_info_df["can_enrich_from_station_master"] = (
    missing_in_info_df["usable_location_master"] == True
)

can_enrich_ids = set(
    missing_in_info_df.loc[
        missing_in_info_df["can_enrich_from_station_master"],
        "station_id"
    ].dropna().tolist()
)

unresolved_ids = set(missing_in_info) - can_enrich_ids

print("station_info에 없는 station 수:", len(missing_in_info))
print("station_master로 위치 보완 가능한 station 수:", len(can_enrich_ids))
print("그래도 보완 불가능한 station 수:", len(unresolved_ids))

print("\n보완 가능 station 예시:")
display(
    missing_in_info_df[
        missing_in_info_df["can_enrich_from_station_master"]
    ].sort_values("bike_row_count_2023_2025", ascending=False).head(100)
)

print("\n보완 불가능 station 예시:")
display(
    missing_in_info_df[
        ~missing_in_info_df["can_enrich_from_station_master"]
    ].sort_values("bike_row_count_2023_2025", ascending=False).head(100)
)

print("\nstation_id 11 보완 확인:")
display(missing_in_info_df[missing_in_info_df["station_id"] == 11])


# ------------------------------------------------------------
# 7. enriched station 생성
# ------------------------------------------------------------
print_section("station_verified_enriched 생성")

# 7-1. station_info 기반 행
station_info_enriched = station_info.copy()

# 혹시 없는 보조 컬럼 보완
if "station_name" not in station_info_enriched.columns:
    station_info_enriched["station_name"] = pd.NA

if "district" not in station_info_enriched.columns:
    station_info_enriched["district"] = pd.NA

station_info_enriched["station_source"] = "station_info"
station_info_enriched["location_source"] = "station_info"
station_info_enriched["needs_rack_count_impute"] = station_info_enriched["rack_count"].isna()
station_info_enriched["needs_install_date_impute"] = station_info_enriched["install_date"].isna()

station_info_enriched = station_info_enriched[
    [
        "station_id",
        "station_name",
        "district",
        "latitude",
        "longitude",
        "rack_count",
        "install_date",
        "station_source",
        "location_source",
        "needs_rack_count_impute",
        "needs_install_date_impute"
    ]
].copy()


# 7-2. station_master로 보완 가능한 행
station_master_enrich = missing_in_info_df[
    missing_in_info_df["can_enrich_from_station_master"]
].copy()

station_master_enrich["station_source"] = "station_master_only"
station_master_enrich["location_source"] = "station_master"

# station_master에는 rack_count/install_date가 없으므로 최종 base에서 대체 필요
station_master_enrich["needs_rack_count_impute"] = True
station_master_enrich["needs_install_date_impute"] = True

station_master_enrich = station_master_enrich[
    [
        "station_id",
        "station_name",
        "district",
        "latitude",
        "longitude",
        "rack_count",
        "install_date",
        "station_source",
        "location_source",
        "needs_rack_count_impute",
        "needs_install_date_impute"
    ]
].copy()


# 7-3. 결합
station_enriched = pd.concat(
    [station_info_enriched, station_master_enrich],
    ignore_index=True
)

station_enriched["station_id"] = station_enriched["station_id"].astype("Int64")
station_enriched["latitude"] = pd.to_numeric(station_enriched["latitude"], errors="coerce")
station_enriched["longitude"] = pd.to_numeric(station_enriched["longitude"], errors="coerce")
station_enriched["rack_count"] = pd.to_numeric(station_enriched["rack_count"], errors="coerce")
station_enriched["install_date"] = pd.to_datetime(station_enriched["install_date"], errors="coerce")

station_enriched["needs_rack_count_impute"] = station_enriched["needs_rack_count_impute"].astype("int8")
station_enriched["needs_install_date_impute"] = station_enriched["needs_install_date_impute"].astype("int8")

print("station_info 기반 행:", len(station_info_enriched))
print("station_master 보완 행:", len(station_master_enrich))
print("station_enriched 총 행:", len(station_enriched))

if station_enriched["station_id"].duplicated().sum() > 0:
    print("중복 station_id:")
    display(
        station_enriched[station_enriched["station_id"].duplicated(keep=False)]
        .sort_values("station_id")
        .head(100)
    )
    raise ValueError("station_enriched에 station_id 중복이 있습니다.")

# 위치 결측/이상 검증
station_enriched["missing_location"] = (
    station_enriched["latitude"].isna() |
    station_enriched["longitude"].isna()
)

station_enriched["invalid_location_range"] = (
    station_enriched["latitude"].notna() &
    station_enriched["longitude"].notna() &
    ~(
        station_enriched["latitude"].between(37.3, 37.8) &
        station_enriched["longitude"].between(126.7, 127.3)
    )
)

print("station_enriched 위치 결측:", int(station_enriched["missing_location"].sum()))
print("station_enriched 위치 범위 이상:", int(station_enriched["invalid_location_range"].sum()))

if station_enriched["missing_location"].sum() > 0 or station_enriched["invalid_location_range"].sum() > 0:
    raise ValueError("station_enriched에 위치 결측 또는 위치 범위 이상이 있습니다.")

# 저장용에서는 검증 임시 컬럼 제거
station_enriched_save = station_enriched.drop(
    columns=["missing_location", "invalid_location_range"],
    errors="ignore"
).copy()

display(station_enriched_save.head())
check_missing(station_enriched_save, "station_enriched_save", top_n=30)


# ------------------------------------------------------------
# 8. enriched station으로 bike station 커버리지 재검증
# ------------------------------------------------------------
print_section("enriched station으로 bike station 커버리지 재검증")

enriched_ids = set(station_enriched_save["station_id"].dropna().unique().tolist())

missing_after_enrichment = bike_station_ids - enriched_ids

resolved_count = len(missing_in_info) - len(missing_after_enrichment)

print("보완 전 missing station 수:", len(missing_in_info))
print("station_master 보완으로 해결된 station 수:", resolved_count)
print("보완 후에도 missing station 수:", len(missing_after_enrichment))
print("보완 후 missing 예시:", sorted(list(missing_after_enrichment))[:100])

print("station_id 11 보완 후 포함 여부:", 11 in enriched_ids)
print("station_id 11 보완 후 missing 여부:", 11 in missing_after_enrichment)

# 보완 후에도 해결 안 된 station 리포트
unresolved_report = bike_station_usage[
    bike_station_usage["station_id"].isin(missing_after_enrichment)
].copy()

unresolved_report = safe_merge(
    unresolved_report,
    missing_in_info_df[
        [
            "station_id",
            "usable_location_master",
            "missing_location_master",
            "zero_location_master",
            "valid_location_master"
        ]
    ],
    on="station_id",
    how="left",
    validate="one_to_one",
    name="unresolved_report + master_flags"
)

unresolved_report = unresolved_report.sort_values(
    "bike_row_count_2023_2025",
    ascending=False
)

unresolved_row_count = int(unresolved_report["bike_row_count_2023_2025"].fillna(0).sum())
unresolved_row_ratio = unresolved_row_count / bike_total_rows if bike_total_rows > 0 else np.nan

print("보완 불가 station 수:", len(unresolved_report))
print("보완 불가 station row 수:", unresolved_row_count)
print("보완 불가 station row 비율:", unresolved_row_ratio)

display(unresolved_report.head(100))


# ------------------------------------------------------------
# 9. 리포트 저장
# ------------------------------------------------------------
print_section("리포트 저장")

# 전체 bike station 기준 enrichment report
bike_station_report = bike_station_usage.copy()

bike_station_report = safe_merge(
    bike_station_report,
    station_enriched_save,
    on="station_id",
    how="left",
    validate="many_to_one",
    name="bike_station_report + station_enriched"
)

bike_station_report["covered_by_enriched_station"] = (
    bike_station_report["latitude"].notna() &
    bike_station_report["longitude"].notna()
)

bike_station_report["will_be_removed_due_to_missing_station"] = ~bike_station_report["covered_by_enriched_station"]

bike_station_report["issue_type"] = np.where(
    bike_station_report["covered_by_enriched_station"],
    "ok",
    "unresolved_missing_station"
)

bike_station_report.to_csv(
    STATION_ENRICHMENT_REPORT_PATH,
    index=False,
    encoding="utf-8-sig"
)

unresolved_report.to_csv(
    STATION_UNRESOLVED_REPORT_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("station enrichment report 저장:", STATION_ENRICHMENT_REPORT_PATH)
print("station unresolved report 저장:", STATION_UNRESOLVED_REPORT_PATH)

print("\ncovered_by_enriched_station 분포:")
print(bike_station_report["covered_by_enriched_station"].value_counts(dropna=False))

print("\nstation_source 분포:")
print(station_enriched_save["station_source"].value_counts(dropna=False))

print("\nlocation_source 분포:")
print(station_enriched_save["location_source"].value_counts(dropna=False))

print("\nneeds_rack_count_impute 분포:")
print(station_enriched_save["needs_rack_count_impute"].value_counts(dropna=False))

print("\nneeds_install_date_impute 분포:")
print(station_enriched_save["needs_install_date_impute"].value_counts(dropna=False))


# ------------------------------------------------------------
# 10. station_verified_enriched 저장
# ------------------------------------------------------------
print_section("station_verified_enriched 저장")

# 저장 전 필수 검증
if station_enriched_save["station_id"].isna().sum() > 0:
    raise ValueError("station_enriched_save에 station_id 결측이 있습니다.")

if station_enriched_save["station_id"].duplicated().sum() > 0:
    raise ValueError("station_enriched_save에 station_id 중복이 있습니다.")

if station_enriched_save["latitude"].isna().sum() > 0:
    raise ValueError("station_enriched_save에 latitude 결측이 있습니다.")

if station_enriched_save["longitude"].isna().sum() > 0:
    raise ValueError("station_enriched_save에 longitude 결측이 있습니다.")

invalid_loc_count = (
    ~(
        station_enriched_save["latitude"].between(37.3, 37.8) &
        station_enriched_save["longitude"].between(126.7, 127.3)
    )
).sum()

if invalid_loc_count > 0:
    raise ValueError(f"station_enriched_save에 서울 범위 밖 위치가 있습니다: {invalid_loc_count}")

station_enriched_save.to_parquet(
    STATION_ENRICHED_OUTPUT_PATH,
    index=False
)

print("저장 완료:", STATION_ENRICHED_OUTPUT_PATH)
print("shape:", station_enriched_save.shape)
print("size MB:", get_file_size_mb(STATION_ENRICHED_OUTPUT_PATH))

display(station_enriched_save.head(20))


# ------------------------------------------------------------
# 11. 최종 요약 저장
# ------------------------------------------------------------
print_section("2.5단계 최종 요약")

summary = {
    "step": "step25_station_enrichment",
    "purpose": "station_info.xlsx 누락 station을 station_master.csv로 위치 보완",
    "project_root": str(PROJECT_ROOT),

    "station_info_verified_path": str(STATION_INFO_VERIFIED_PATH),
    "station_master_path": str(STATION_MASTER_PATH),
    "station_enriched_output_path": str(STATION_ENRICHED_OUTPUT_PATH),

    "bike_total_rows_2023_2025": int(bike_total_rows),
    "bike_station_nunique_2023_2025": int(len(bike_station_ids)),
    "bike_datetime_min": str(bike_min_dt),
    "bike_datetime_max": str(bike_max_dt),

    "station_info_station_count": int(len(station_info_ids)),
    "station_master_station_count": int(station_master["station_id"].nunique()),
    "station_enriched_station_count": int(station_enriched_save["station_id"].nunique()),

    "missing_before_enrichment_station_count": int(len(missing_in_info)),
    "can_enrich_from_station_master_count": int(len(can_enrich_ids)),
    "resolved_by_station_master_count": int(resolved_count),
    "missing_after_enrichment_station_count": int(len(missing_after_enrichment)),

    "missing_before_enrichment_station_ids_sample": str(sorted(list(missing_in_info))[:100]),
    "missing_after_enrichment_station_ids_sample": str(sorted(list(missing_after_enrichment))[:100]),

    "unresolved_station_bike_row_count": int(unresolved_row_count),
    "unresolved_station_bike_row_ratio": float(unresolved_row_ratio),

    "station_id_11_in_bike": bool(11 in bike_station_ids),
    "station_id_11_in_station_info": bool(11 in station_info_ids),
    "station_id_11_in_station_master": bool(11 in set(station_master["station_id"].dropna().tolist())),
    "station_id_11_in_enriched_station": bool(11 in enriched_ids),
    "station_id_11_missing_after_enrichment": bool(11 in missing_after_enrichment),

    "station_enriched_latitude_missing": int(station_enriched_save["latitude"].isna().sum()),
    "station_enriched_longitude_missing": int(station_enriched_save["longitude"].isna().sum()),
    "station_enriched_rack_count_missing": int(station_enriched_save["rack_count"].isna().sum()),
    "station_enriched_install_date_missing": int(station_enriched_save["install_date"].isna().sum()),
    "station_enriched_needs_rack_count_impute_count": int(station_enriched_save["needs_rack_count_impute"].sum()),
    "station_enriched_needs_install_date_impute_count": int(station_enriched_save["needs_install_date_impute"].sum()),

    "station_enrichment_report_path": str(STATION_ENRICHMENT_REPORT_PATH),
    "station_unresolved_report_path": str(STATION_UNRESOLVED_REPORT_PATH),

    "preprocessing_report_note": (
        "station_info.xlsx를 기본 station master로 사용하되, "
        "2023~2025 대여 이력에 등장하지만 station_info에 없는 station_id는 station_master.csv로 보완하였다. "
        "station_master.csv의 위도/경도는 0값과 서울 범위 밖 좌표를 제외하고, "
        "정상 위치인 경우에만 보완에 사용하였다. "
        "station_master로 보완된 대여소는 rack_count와 install_date가 없으므로 "
        "최종 base dataset 생성 단계에서 대체가 필요하도록 플래그를 남겼다."
    )
}

for k, v in summary.items():
    print(f"{k}: {v}")

with open(STATION_SUMMARY_PATH, "w", encoding="utf-8") as f:
    for k, v in summary.items():
        f.write(f"{k}: {v}\n")

print("\n2.5단계 요약 저장:", STATION_SUMMARY_PATH)

print("\n[다음 단계]")
print("1. station_verified_enriched.parquet을 최종 station master로 사용합니다.")
print("2. 다음 단계에서는 bike parquet을 station_id × datetime 기준 hourly rental_count로 집계합니다.")
print("3. 최종 base dataset 생성 단계에서 rack_count 결측은 중앙값 등으로 대체합니다.")
print("4. install_date 결측은 station_age_missing 플래그를 만들고 station_age_days를 대체합니다.")
print("5. 보완 후에도 station 정보가 없는 bike row는 최종 학습 데이터에서 제거합니다.")

## 4. 03 hourly rental count

Source: `03_hourly_rental_count.ipynb`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import gc

warnings.filterwarnings("ignore")

# ============================================================
# 3단계: bike parquet -> hourly rental_count 생성
#
# 목적:
# - 1단계에서 만든 2023~2025 bike parquet을 사용
# - station_id × datetime 기준 시간별 대여량 집계
# - station 정보가 검증되지 않은 58개 station 제거
# - station_verified_enriched.parquet에 없는 station 제거
# - 중복 key / 결측 / 음수 검증 후 저장
#
# 입력:
# - data/interim_2/raw_parquet/bike_history/2023~2025/*.parquet
# - data/processed_2/station_verified_enriched.parquet
#
# 출력:
# - data/processed_2/hourly_rental_verified.parquet
# - data/interim_2/step3_hourly_rental_file_report.csv
# - data/interim_2/step3_hourly_rental_summary.txt
# ============================================================


# ------------------------------------------------------------
# 0. 경로 설정
# ------------------------------------------------------------
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebook" else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim_2"
PROCESSED_DIR = DATA_DIR / "processed_2"

BIKE_PARQUET_DIR = INTERIM_DIR / "raw_parquet" / "bike_history"
STATION_ENRICHED_PATH = PROCESSED_DIR / "station_verified_enriched.parquet"

HOURLY_OUTPUT_PATH = PROCESSED_DIR / "hourly_rental_verified.parquet"
HOURLY_FILE_REPORT_PATH = INTERIM_DIR / "step3_hourly_rental_file_report.csv"
HOURLY_SUMMARY_PATH = INTERIM_DIR / "step3_hourly_rental_summary.txt"

TARGET_YEARS = [2023, 2024, 2025]

# 2.5단계에서 보완 불가로 확정한 station_id 58개
UNRESOLVED_STATION_IDS = [
    601, 1071, 1257, 3520, 3524, 3538, 3541, 3564, 3578,
    3672, 3677, 3711, 3720, 3769, 3810, 3822, 3891, 3898,
    3977, 4003, 4034, 4065, 4093, 4125, 4171, 4239, 4251,
    4262, 4265, 4314, 4335, 4339, 4388, 4392, 4408, 4426,
    4456, 4467, 4516, 4568, 4602, 4605, 4709, 4768, 4778,
    4869, 4876, 4911, 4933, 5056, 5062, 5063, 5094, 5306,
    5770, 5772, 9979, 9980
]

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("[3단계 경로 확인]")
print("=" * 100)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("BIKE_PARQUET_DIR:", BIKE_PARQUET_DIR, "| exists:", BIKE_PARQUET_DIR.exists())
print("STATION_ENRICHED_PATH:", STATION_ENRICHED_PATH, "| exists:", STATION_ENRICHED_PATH.exists())
print("HOURLY_OUTPUT_PATH:", HOURLY_OUTPUT_PATH)

if not BIKE_PARQUET_DIR.exists():
    raise FileNotFoundError(f"bike parquet 폴더가 없습니다: {BIKE_PARQUET_DIR}")

if not STATION_ENRICHED_PATH.exists():
    raise FileNotFoundError(f"station_verified_enriched.parquet이 없습니다: {STATION_ENRICHED_PATH}")


# ------------------------------------------------------------
# 1. 유틸 함수
# ------------------------------------------------------------
def print_section(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def get_file_size_mb(path):
    path = Path(path)
    if not path.exists():
        return np.nan
    return path.stat().st_size / 1024 / 1024


def check_duplicate_key(df, keys, name):
    print_section(f"[{name}] key 중복 확인")
    dup_count = df.duplicated(keys).sum()
    print("keys:", keys)
    print("duplicated rows:", dup_count)

    if dup_count > 0:
        display(df[df.duplicated(keys, keep=False)].sort_values(keys).head(50))
        raise ValueError(f"{name}: key 중복이 있습니다.")

    return dup_count


def check_missing(df, name, top_n=30):
    print_section(f"[{name}] 결측 확인")
    result = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_rate_percent": (df.isna().mean() * 100).round(4)
    }).sort_values("missing_rate_percent", ascending=False)
    display(result.head(top_n))
    return result


def check_datetime_range(df, name):
    print_section(f"[{name}] datetime 범위 확인")
    print("min:", df["datetime"].min())
    print("max:", df["datetime"].max())
    print("연도별 row 수:")
    print(df["datetime"].dt.year.value_counts().sort_index())


# ------------------------------------------------------------
# 2. station_verified_enriched 로드
# ------------------------------------------------------------
print_section("station_verified_enriched 로드")

station_enriched = pd.read_parquet(STATION_ENRICHED_PATH)

print("station_enriched shape:", station_enriched.shape)
display(station_enriched.head())

required_station_cols = ["station_id", "latitude", "longitude"]

missing_cols = [c for c in required_station_cols if c not in station_enriched.columns]
if missing_cols:
    raise ValueError(f"station_enriched에 필수 컬럼이 없습니다: {missing_cols}")

if station_enriched["station_id"].isna().sum() > 0:
    raise ValueError("station_enriched에 station_id 결측이 있습니다.")

if station_enriched["station_id"].duplicated().sum() > 0:
    raise ValueError("station_enriched에 station_id 중복이 있습니다.")

if station_enriched["latitude"].isna().sum() > 0:
    raise ValueError("station_enriched에 latitude 결측이 있습니다.")

if station_enriched["longitude"].isna().sum() > 0:
    raise ValueError("station_enriched에 longitude 결측이 있습니다.")

valid_station_ids = set(station_enriched["station_id"].dropna().astype(int).tolist())
unresolved_station_ids = set(UNRESOLVED_STATION_IDS)

print("검증된 station 수:", len(valid_station_ids))
print("제거 대상 unresolved station 수:", len(unresolved_station_ids))

# station_enriched에 unresolved station이 들어있으면 이상함
unresolved_in_valid = valid_station_ids.intersection(unresolved_station_ids)
print("station_enriched 안에 unresolved station 포함 수:", len(unresolved_in_valid))
if len(unresolved_in_valid) > 0:
    print("예시:", sorted(list(unresolved_in_valid))[:30])
    raise ValueError("station_enriched에 제거 대상 unresolved station이 포함되어 있습니다.")


# ------------------------------------------------------------
# 3. 2023~2025 bike parquet 파일 목록 수집
# ------------------------------------------------------------
print_section("2023~2025 bike parquet 파일 수집")

bike_parquet_files = []

for year in TARGET_YEARS:
    year_dir = BIKE_PARQUET_DIR / str(year)
    if year_dir.exists():
        bike_parquet_files.extend(sorted(year_dir.glob("*.parquet")))

print("bike parquet 파일 수:", len(bike_parquet_files))

if len(bike_parquet_files) == 0:
    raise FileNotFoundError("2023~2025 bike parquet 파일이 없습니다.")

for p in bike_parquet_files[:20]:
    print(p.relative_to(PROJECT_ROOT))

if len(bike_parquet_files) > 20:
    print("... 생략:", len(bike_parquet_files) - 20)


# ------------------------------------------------------------
# 4. 월별 parquet -> 월별 hourly 집계
# ------------------------------------------------------------
print_section("월별 bike parquet -> hourly 집계 시작")

monthly_hourly_list = []
file_reports = []

total_raw_rows = 0
total_after_year_filter_rows = 0
total_removed_unresolved_rows = 0
total_removed_not_in_valid_station_rows = 0
total_used_rows = 0

all_raw_station_ids = set()
all_used_station_ids = set()

global_min_dt = None
global_max_dt = None

for i, path in enumerate(bike_parquet_files, start=1):
    print(f"\n[{i}/{len(bike_parquet_files)}] 처리 중:", path.relative_to(PROJECT_ROOT))

    part = pd.read_parquet(path, columns=["station_id", "datetime"])

    raw_rows = len(part)
    total_raw_rows += raw_rows

    # station_id 타입 정리
    part["station_id"] = part["station_id"].astype("Int64")
    part["datetime"] = pd.to_datetime(part["datetime"], errors="coerce").dt.floor("h")

    # 혹시 모를 결측 제거
    before_core_drop = len(part)
    part = part.dropna(subset=["station_id", "datetime"]).copy()
    removed_core_missing = before_core_drop - len(part)

    # 대상 연도 필터
    before_year_filter = len(part)
    part = part[part["datetime"].dt.year.isin(TARGET_YEARS)].copy()
    after_year_filter = len(part)
    removed_out_of_target_year = before_year_filter - after_year_filter
    total_after_year_filter_rows += after_year_filter

    # raw station 기록
    all_raw_station_ids.update(part["station_id"].dropna().astype(int).unique().tolist())

    # 58개 unresolved station 제거
    before_unresolved_filter = len(part)
    part = part[~part["station_id"].astype(int).isin(unresolved_station_ids)].copy()
    after_unresolved_filter = len(part)
    removed_unresolved = before_unresolved_filter - after_unresolved_filter
    total_removed_unresolved_rows += removed_unresolved

    # station_verified_enriched에 없는 station 제거
    before_valid_station_filter = len(part)
    part = part[part["station_id"].astype(int).isin(valid_station_ids)].copy()
    after_valid_station_filter = len(part)
    removed_not_in_valid_station = before_valid_station_filter - after_valid_station_filter
    total_removed_not_in_valid_station_rows += removed_not_in_valid_station

    used_rows = len(part)
    total_used_rows += used_rows

    if used_rows > 0:
        min_dt = part["datetime"].min()
        max_dt = part["datetime"].max()

        if global_min_dt is None or min_dt < global_min_dt:
            global_min_dt = min_dt

        if global_max_dt is None or max_dt > global_max_dt:
            global_max_dt = max_dt

        all_used_station_ids.update(part["station_id"].dropna().astype(int).unique().tolist())

        # 월별 hourly 집계
        monthly_hourly = (
            part
            .groupby(["station_id", "datetime"], as_index=False)
            .size()
            .rename(columns={"size": "rental_count"})
        )

        monthly_hourly["rental_count"] = monthly_hourly["rental_count"].astype("float32")

        monthly_hourly_list.append(monthly_hourly)

        hourly_rows = len(monthly_hourly)
        hourly_station_nunique = monthly_hourly["station_id"].nunique()
        hourly_min_dt = monthly_hourly["datetime"].min()
        hourly_max_dt = monthly_hourly["datetime"].max()
    else:
        hourly_rows = 0
        hourly_station_nunique = 0
        hourly_min_dt = pd.NaT
        hourly_max_dt = pd.NaT

    file_reports.append({
        "path": str(path.relative_to(PROJECT_ROOT)),
        "raw_rows": int(raw_rows),
        "removed_core_missing": int(removed_core_missing),
        "removed_out_of_target_year": int(removed_out_of_target_year),
        "removed_unresolved_station_rows": int(removed_unresolved),
        "removed_not_in_valid_station_rows": int(removed_not_in_valid_station),
        "used_rows": int(used_rows),
        "hourly_rows": int(hourly_rows),
        "hourly_station_nunique": int(hourly_station_nunique),
        "hourly_min_datetime": str(hourly_min_dt),
        "hourly_max_datetime": str(hourly_max_dt),
        "file_size_mb": float(get_file_size_mb(path))
    })

    print("raw rows:", raw_rows)
    print("removed core missing:", removed_core_missing)
    print("removed out-of-target-year:", removed_out_of_target_year)
    print("removed unresolved station rows:", removed_unresolved)
    print("removed not-in-valid-station rows:", removed_not_in_valid_station)
    print("used rows:", used_rows)
    print("hourly rows:", hourly_rows)
    print("hourly datetime:", hourly_min_dt, "~", hourly_max_dt)

    del part
    gc.collect()


if len(monthly_hourly_list) == 0:
    raise ValueError("hourly 집계 결과가 없습니다.")


# ------------------------------------------------------------
# 5. 월별 hourly를 전체 hourly로 재집계
# ------------------------------------------------------------
print_section("월별 hourly 결합 및 최종 재집계")

hourly_temp = pd.concat(monthly_hourly_list, ignore_index=True)

print("hourly_temp shape:", hourly_temp.shape)

# 월별 파일 경계 또는 중복 파일 가능성 방지용 최종 재집계
hourly = (
    hourly_temp
    .groupby(["station_id", "datetime"], as_index=False)["rental_count"]
    .sum()
)

hourly["station_id"] = hourly["station_id"].astype("Int64")
hourly["datetime"] = pd.to_datetime(hourly["datetime"])
hourly["rental_count"] = hourly["rental_count"].astype("float32")

print("hourly 최종 shape:", hourly.shape)

del hourly_temp, monthly_hourly_list
gc.collect()


# ------------------------------------------------------------
# 6. hourly 검증
# ------------------------------------------------------------
print_section("hourly_rental 검증")

print("전체 원본 row:", total_raw_rows)
print("연도 필터 후 row:", total_after_year_filter_rows)
print("unresolved station 제거 row:", total_removed_unresolved_rows)
print("valid station에 없는 row 제거:", total_removed_not_in_valid_station_rows)
print("최종 hourly 집계에 사용된 row:", total_used_rows)

print("원본 station 수:", len(all_raw_station_ids))
print("사용 station 수:", len(all_used_station_ids))
print("hourly station 수:", hourly["station_id"].nunique())

print("기간:", hourly["datetime"].min(), "~", hourly["datetime"].max())

check_duplicate_key(hourly, ["station_id", "datetime"], "hourly_rental_verified")
check_missing(hourly, "hourly_rental_verified", top_n=20)
check_datetime_range(hourly, "hourly_rental_verified")

# rental_count 검증
if hourly["rental_count"].isna().sum() > 0:
    raise ValueError("hourly rental_count에 결측이 있습니다.")

negative_count = (hourly["rental_count"] < 0).sum()
print("rental_count 음수 개수:", negative_count)
if negative_count > 0:
    raise ValueError("hourly rental_count에 음수가 있습니다.")

zero_or_less_count = (hourly["rental_count"] <= 0).sum()
print("rental_count <= 0 개수:", zero_or_less_count)
if zero_or_less_count > 0:
    raise ValueError("집계된 hourly 데이터에는 rental_count가 1 이상이어야 합니다.")

# 대상 연도 외 데이터 검증
years_in_hourly = set(hourly["datetime"].dt.year.unique().tolist())
print("hourly 포함 연도:", sorted(list(years_in_hourly)))
if not years_in_hourly.issubset(set(TARGET_YEARS)):
    raise ValueError(f"대상 연도 외 데이터가 포함되어 있습니다: {years_in_hourly}")

# unresolved station이 남아있는지 검증
remaining_unresolved = set(hourly["station_id"].astype(int).unique()).intersection(unresolved_station_ids)
print("hourly에 남은 unresolved station 수:", len(remaining_unresolved))
if len(remaining_unresolved) > 0:
    print("남은 unresolved station:", sorted(list(remaining_unresolved))[:100])
    raise ValueError("hourly에 제거 대상 unresolved station이 남아 있습니다.")

# station_enriched에 없는 station이 남아있는지 검증
hourly_station_ids = set(hourly["station_id"].astype(int).unique().tolist())
remaining_not_in_valid = hourly_station_ids - valid_station_ids
print("hourly에 남은 valid station 외 station 수:", len(remaining_not_in_valid))
if len(remaining_not_in_valid) > 0:
    print("예시:", sorted(list(remaining_not_in_valid))[:100])
    raise ValueError("hourly에 station_verified_enriched에 없는 station이 남아 있습니다.")

print("\nrental_count describe:")
print(hourly["rental_count"].describe())

print("\nrental_count quantile:")
print(hourly["rental_count"].quantile([0, 0.5, 0.9, 0.95, 0.99, 0.999, 1]))

print("\n연도별 rental_count 합계:")
print(hourly.groupby(hourly["datetime"].dt.year)["rental_count"].sum())

print("\n연도별 hourly row 수:")
print(hourly.groupby(hourly["datetime"].dt.year).size())

print("\nstation별 총 대여량 요약:")
station_total = hourly.groupby("station_id")["rental_count"].sum()
print(station_total.describe())


# ------------------------------------------------------------
# 7. 리포트 저장
# ------------------------------------------------------------
print_section("3단계 리포트 저장")

file_report_df = pd.DataFrame(file_reports)
file_report_df.to_csv(HOURLY_FILE_REPORT_PATH, index=False, encoding="utf-8-sig")

print("file report 저장:", HOURLY_FILE_REPORT_PATH)
display(file_report_df.head())

print("\n파일별 제거 row 합계:")
print(file_report_df[
    [
        "removed_core_missing",
        "removed_out_of_target_year",
        "removed_unresolved_station_rows",
        "removed_not_in_valid_station_rows",
        "used_rows",
        "hourly_rows"
    ]
].sum())


# ------------------------------------------------------------
# 8. hourly_rental_verified 저장
# ------------------------------------------------------------
print_section("hourly_rental_verified 저장")

hourly.to_parquet(HOURLY_OUTPUT_PATH, index=False)

print("저장 완료:", HOURLY_OUTPUT_PATH)
print("shape:", hourly.shape)
print("size MB:", get_file_size_mb(HOURLY_OUTPUT_PATH))

display(hourly.head(20))


# ------------------------------------------------------------
# 9. 최종 요약 저장
# ------------------------------------------------------------
print_section("3단계 최종 요약")

removed_total_rows = (
    total_raw_rows - total_used_rows
)

removed_total_ratio = removed_total_rows / total_raw_rows if total_raw_rows > 0 else np.nan
removed_unresolved_ratio = total_removed_unresolved_rows / total_raw_rows if total_raw_rows > 0 else np.nan
removed_not_valid_ratio = total_removed_not_in_valid_station_rows / total_raw_rows if total_raw_rows > 0 else np.nan

summary = {
    "step": "step3_make_hourly_rental",
    "purpose": "2023~2025 bike parquet을 station_id × datetime 기준 hourly rental_count로 집계",
    "project_root": str(PROJECT_ROOT),

    "bike_parquet_dir": str(BIKE_PARQUET_DIR),
    "station_enriched_path": str(STATION_ENRICHED_PATH),
    "hourly_output_path": str(HOURLY_OUTPUT_PATH),

    "target_years": str(TARGET_YEARS),
    "bike_parquet_file_count": int(len(bike_parquet_files)),

    "raw_bike_rows": int(total_raw_rows),
    "after_year_filter_rows": int(total_after_year_filter_rows),
    "removed_unresolved_station_rows": int(total_removed_unresolved_rows),
    "removed_not_in_valid_station_rows": int(total_removed_not_in_valid_station_rows),
    "used_bike_rows_for_hourly": int(total_used_rows),
    "removed_total_rows": int(removed_total_rows),
    "removed_total_ratio": float(removed_total_ratio),
    "removed_unresolved_ratio": float(removed_unresolved_ratio),
    "removed_not_valid_ratio": float(removed_not_valid_ratio),

    "raw_station_nunique": int(len(all_raw_station_ids)),
    "used_station_nunique": int(len(all_used_station_ids)),
    "hourly_station_nunique": int(hourly["station_id"].nunique()),

    "hourly_shape": str(hourly.shape),
    "hourly_datetime_min": str(hourly["datetime"].min()),
    "hourly_datetime_max": str(hourly["datetime"].max()),

    "hourly_key_duplicate_count": int(hourly.duplicated(["station_id", "datetime"]).sum()),
    "hourly_rental_count_missing": int(hourly["rental_count"].isna().sum()),
    "hourly_rental_count_negative": int((hourly["rental_count"] < 0).sum()),
    "hourly_unresolved_station_remaining_count": int(len(remaining_unresolved)),
    "hourly_not_in_valid_station_remaining_count": int(len(remaining_not_in_valid)),

    "unresolved_station_ids_removed": str(UNRESOLVED_STATION_IDS),

    "file_report_path": str(HOURLY_FILE_REPORT_PATH),

    "preprocessing_report_note": (
        "2023~2025 대여 이력 parquet에서 station_id와 datetime을 기준으로 시간 단위 대여량을 집계하였다. "
        "station_info.xlsx와 station_master.csv 보완 후에도 위치 정보가 검증되지 않은 58개 station_id는 "
        "최종 학습 데이터 신뢰성을 위해 집계 단계에서 제거하였다. "
        "최종 hourly_rental 데이터는 station_id와 datetime 조합이 유일하며, rental_count 결측 및 음수 값이 없음을 검증하였다."
    )
}

for k, v in summary.items():
    print(f"{k}: {v}")

with open(HOURLY_SUMMARY_PATH, "w", encoding="utf-8") as f:
    for k, v in summary.items():
        f.write(f"{k}: {v}\n")

print("\n3단계 요약 저장:", HOURLY_SUMMARY_PATH)

print("\n[다음 단계]")
print("1. 다음 단계에서는 weather 데이터를 정제합니다.")
print("2. 이후 holiday 데이터를 정제합니다.")
print("3. 그 다음 station별 관측 기간을 기준으로 full hourly grid를 생성합니다.")
print("4. full grid에 hourly rental_count를 merge하고, 없는 시간은 0으로 채웁니다.")

## 5. 04 weather process

Source: `04_weather_process.ipynb`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import gc
import re

warnings.filterwarnings("ignore")

# ============================================================
# 4단계: weather 데이터 정제
#
# 목적:
# - raw weather 파일을 읽어 datetime 기준 시간 단위 weather feature 생성
# - hourly_rental_verified 기간을 기준으로 weather coverage 검증
# - temperature / precipitation / wind_speed / humidity / snowfall 정제
# - 최종 weather_verified.parquet에 결측/중복 datetime이 없도록 저장
#
# 입력:
# - data/raw 아래 weather/날씨/기상/asos 관련 파일
# - data/processed_2/hourly_rental_verified.parquet
#
# 출력:
# - data/processed_2/weather_verified.parquet
# - data/interim_2/step4_weather_raw_report.csv
# - data/interim_2/step4_weather_summary.txt
# ============================================================


# ------------------------------------------------------------
# 0. 경로 설정
# ------------------------------------------------------------
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebook" else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim_2"
PROCESSED_DIR = DATA_DIR / "processed_2"

HOURLY_RENTAL_PATH = PROCESSED_DIR / "hourly_rental_verified.parquet"
WEATHER_OUTPUT_PATH = PROCESSED_DIR / "weather_verified.parquet"
WEATHER_RAW_REPORT_PATH = INTERIM_DIR / "step4_weather_raw_report.csv"
WEATHER_SUMMARY_PATH = INTERIM_DIR / "step4_weather_summary.txt"

TARGET_YEARS = [2023, 2024, 2025]

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("[4단계 경로 확인]")
print("=" * 100)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR, "| exists:", RAW_DIR.exists())
print("HOURLY_RENTAL_PATH:", HOURLY_RENTAL_PATH, "| exists:", HOURLY_RENTAL_PATH.exists())
print("WEATHER_OUTPUT_PATH:", WEATHER_OUTPUT_PATH)

if not RAW_DIR.exists():
    raise FileNotFoundError(f"RAW_DIR이 없습니다: {RAW_DIR}")

if not HOURLY_RENTAL_PATH.exists():
    raise FileNotFoundError("3단계 결과 hourly_rental_verified.parquet이 없습니다. 3단계를 먼저 완료하세요.")


# ------------------------------------------------------------
# 1. 유틸 함수
# ------------------------------------------------------------
def print_section(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def get_file_size_mb(path):
    path = Path(path)
    if not path.exists():
        return np.nan
    return path.stat().st_size / 1024 / 1024


def clean_colname(col):
    col = str(col)
    col = col.replace("\n", "")
    col = col.replace("\r", "")
    col = col.replace("\t", "")
    col = col.strip()
    col = re.sub(r"\s+", "", col)
    return col


def load_table_safely(path):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        try:
            return pd.read_csv(path, encoding="utf-8", low_memory=False)
        except UnicodeDecodeError:
            return pd.read_csv(path, encoding="cp949", low_memory=False)

    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    if suffix == ".parquet":
        return pd.read_parquet(path)

    raise ValueError(f"지원하지 않는 파일 형식입니다: {suffix}")


def find_col_by_keywords(columns, keywords, exclude_keywords=None):
    exclude_keywords = exclude_keywords or []

    for col in columns:
        col_text = clean_colname(col).lower()

        if any(clean_colname(ex).lower() in col_text for ex in exclude_keywords):
            continue

        for kw in keywords:
            kw_text = clean_colname(kw).lower()
            if kw_text in col_text:
                return col

    return None


def to_numeric_safely(series):
    return pd.to_numeric(
        series.astype(str)
              .str.replace(",", "", regex=False)
              .str.strip()
              .replace({"": np.nan, "nan": np.nan, "None": np.nan, "-": np.nan}),
        errors="coerce"
    )


def check_missing(df, name, top_n=30):
    print_section(f"[{name}] 결측 확인")
    result = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_rate_percent": (df.isna().mean() * 100).round(4)
    }).sort_values("missing_rate_percent", ascending=False)
    display(result.head(top_n))
    return result


def check_duplicate_key(df, keys, name):
    print_section(f"[{name}] key 중복 확인")
    dup_count = df.duplicated(keys).sum()
    print("keys:", keys)
    print("duplicated rows:", dup_count)

    if dup_count > 0:
        display(df[df.duplicated(keys, keep=False)].sort_values(keys).head(50))
        raise ValueError(f"{name}: key 중복이 있습니다.")

    return dup_count


# ------------------------------------------------------------
# 2. hourly rental 기간 확인
# ------------------------------------------------------------
print_section("hourly_rental_verified 기간 확인")

hourly_meta = pd.read_parquet(
    HOURLY_RENTAL_PATH,
    columns=["datetime"]
)

hourly_meta["datetime"] = pd.to_datetime(hourly_meta["datetime"])

hourly_start = hourly_meta["datetime"].min().floor("h")
hourly_end = hourly_meta["datetime"].max().floor("h")

print("hourly 기간:", hourly_start, "~", hourly_end)
print("hourly 연도별 row 수:")
print(hourly_meta["datetime"].dt.year.value_counts().sort_index())

del hourly_meta
gc.collect()


# ------------------------------------------------------------
# 3. weather 파일 후보 탐색
# ------------------------------------------------------------
print_section("weather 파일 후보 탐색")

all_raw_files = [p for p in RAW_DIR.rglob("*") if p.is_file()]

weather_candidates = []

for p in all_raw_files:
    path_text = str(p).lower()
    name_text = p.name.lower()

    is_supported = p.suffix.lower() in [".csv", ".xlsx", ".xls", ".parquet"]

    is_weather_like = (
        "weather" in path_text or
        "날씨" in path_text or
        "기상" in path_text or
        "asos" in path_text or
        "종관" in path_text
    )

    is_not_bike = (
        "bike_history" not in path_text and
        "대여이력" not in path_text and
        "station" not in path_text and
        "대여소" not in path_text
    )

    if is_supported and is_weather_like and is_not_bike:
        weather_candidates.append(p)

print("weather 후보 파일 수:", len(weather_candidates))

for i, p in enumerate(weather_candidates, start=1):
    print(f"{i}. {p.relative_to(PROJECT_ROOT)} | size MB: {get_file_size_mb(p):.3f}")

if len(weather_candidates) == 0:
    print("\n[ERROR] weather 후보 파일을 찾지 못했습니다.")
    print("data/raw 아래 파일 목록 일부:")
    for p in all_raw_files[:200]:
        print(p.relative_to(PROJECT_ROOT))
    raise FileNotFoundError("weather/날씨/기상/asos 관련 파일을 찾지 못했습니다.")


# ------------------------------------------------------------
# 4. weather 파일 로드 및 컬럼 탐색
# ------------------------------------------------------------
print_section("weather 파일 로드 및 표준화")

weather_parts = []
weather_raw_reports = []

for i, path in enumerate(weather_candidates, start=1):
    print(f"\n[{i}/{len(weather_candidates)}] 로드:", path.relative_to(PROJECT_ROOT))

    raw = load_table_safely(path)

    original_shape = raw.shape

    # 컬럼명 정리
    original_cols = list(raw.columns)
    cleaned_cols = [clean_colname(c) for c in original_cols]
    raw = raw.rename(columns=dict(zip(original_cols, cleaned_cols)))

    cols = list(raw.columns)

    print("shape:", original_shape)
    print("columns:", cols)

    datetime_col = find_col_by_keywords(
        cols,
        ["일시", "datetime", "date_time", "관측일시", "시간", "날짜"]
    )

    temp_col = find_col_by_keywords(
        cols,
        ["기온", "temperature", "temp"]
    )

    precip_col = find_col_by_keywords(
        cols,
        ["강수량", "precipitation", "rain", "rainfall"]
    )

    wind_col = find_col_by_keywords(
        cols,
        ["풍속", "windspeed", "wind_speed", "wind"]
    )

    humidity_col = find_col_by_keywords(
        cols,
        ["습도", "humidity"]
    )

    snowfall_col = find_col_by_keywords(
        cols,
        ["적설", "snowfall", "snow"]
    )

    print("선택 datetime_col:", datetime_col)
    print("선택 temp_col:", temp_col)
    print("선택 precip_col:", precip_col)
    print("선택 wind_col:", wind_col)
    print("선택 humidity_col:", humidity_col)
    print("선택 snowfall_col:", snowfall_col)

    if datetime_col is None:
        raise ValueError(f"{path}에서 datetime 컬럼을 찾지 못했습니다.")

    if temp_col is None:
        raise ValueError(f"{path}에서 기온 컬럼을 찾지 못했습니다.")

    part = pd.DataFrame()

    part["datetime"] = pd.to_datetime(raw[datetime_col], errors="coerce").dt.floor("h")
    part["temperature"] = to_numeric_safely(raw[temp_col])

    if precip_col is not None:
        part["precipitation"] = to_numeric_safely(raw[precip_col])
    else:
        part["precipitation"] = np.nan

    if wind_col is not None:
        part["wind_speed"] = to_numeric_safely(raw[wind_col])
    else:
        part["wind_speed"] = np.nan

    if humidity_col is not None:
        part["humidity"] = to_numeric_safely(raw[humidity_col])
    else:
        part["humidity"] = np.nan

    if snowfall_col is not None:
        part["snowfall"] = to_numeric_safely(raw[snowfall_col])
    else:
        part["snowfall"] = np.nan

    raw_rows = len(part)

    # datetime 결측 제거
    datetime_missing = part["datetime"].isna().sum()
    part = part.dropna(subset=["datetime"]).copy()

    # 대상 기간 넉넉히 필터
    part = part[
        (part["datetime"] >= hourly_start) &
        (part["datetime"] <= hourly_end)
    ].copy()

    filtered_rows = len(part)

    if filtered_rows > 0:
        weather_parts.append(part)

    weather_raw_reports.append({
        "path": str(path.relative_to(PROJECT_ROOT)),
        "raw_shape": str(original_shape),
        "raw_rows": int(raw_rows),
        "datetime_missing_rows": int(datetime_missing),
        "filtered_rows_in_hourly_period": int(filtered_rows),
        "datetime_col": str(datetime_col),
        "temp_col": str(temp_col),
        "precip_col": str(precip_col),
        "wind_col": str(wind_col),
        "humidity_col": str(humidity_col),
        "snowfall_col": str(snowfall_col),
        "min_datetime": str(part["datetime"].min()) if filtered_rows > 0 else "NaT",
        "max_datetime": str(part["datetime"].max()) if filtered_rows > 0 else "NaT",
        "file_size_mb": float(get_file_size_mb(path))
    })

    display(part.head())

    del raw, part
    gc.collect()


if len(weather_parts) == 0:
    raise ValueError("hourly 기간에 해당하는 weather 데이터가 없습니다.")


# ------------------------------------------------------------
# 5. weather 결합 및 시간 중복 처리
# ------------------------------------------------------------
print_section("weather 결합 및 datetime 중복 처리")

weather_raw = pd.concat(weather_parts, ignore_index=True)

del weather_parts
gc.collect()

print("weather_raw combined shape:", weather_raw.shape)
display(weather_raw.head())

check_missing(weather_raw, "weather_raw_combined", top_n=20)

duplicate_datetime_count = weather_raw["datetime"].duplicated().sum()
print("weather_raw datetime 중복 row 수:", duplicate_datetime_count)

# 같은 시간 데이터가 여러 개 있으면 평균 처리
# 예: 여러 파일 중복, 여러 관측소 중복 가능성
weather_hourly = (
    weather_raw
    .groupby("datetime", as_index=False)
    .agg({
        "temperature": "mean",
        "precipitation": "mean",
        "wind_speed": "mean",
        "humidity": "mean",
        "snowfall": "mean"
    })
)

print("weather_hourly shape:", weather_hourly.shape)
check_duplicate_key(weather_hourly, ["datetime"], "weather_hourly_after_groupby")

del weather_raw
gc.collect()


# ------------------------------------------------------------
# 6. hourly 기간 전체 weather grid 생성
# ------------------------------------------------------------
print_section("hourly 기간 전체 weather grid 생성")

full_weather_grid = pd.DataFrame({
    "datetime": pd.date_range(start=hourly_start, end=hourly_end, freq="h")
})

print("full_weather_grid shape:", full_weather_grid.shape)
print("grid 기간:", full_weather_grid["datetime"].min(), "~", full_weather_grid["datetime"].max())

weather = full_weather_grid.merge(
    weather_hourly,
    on="datetime",
    how="left",
    validate="one_to_one"
)

print("weather merge 후 shape:", weather.shape)

# 결측 flag 생성
weather["weather_row_missing"] = weather[["temperature", "precipitation", "wind_speed", "humidity", "snowfall"]].isna().all(axis=1).astype("int8")
weather["temperature_missing"] = weather["temperature"].isna().astype("int8")
weather["precipitation_missing"] = weather["precipitation"].isna().astype("int8")
weather["wind_speed_missing"] = weather["wind_speed"].isna().astype("int8")
weather["humidity_missing"] = weather["humidity"].isna().astype("int8")
weather["snowfall_missing"] = weather["snowfall"].isna().astype("int8")

print("merge 직후 결측:")
print(weather.isna().sum())

del weather_hourly, full_weather_grid
gc.collect()


# ------------------------------------------------------------
# 7. weather 결측 처리
# ------------------------------------------------------------
print_section("weather 결측 처리")

weather = weather.sort_values("datetime").copy()

# 강수량/적설:
# 기상 데이터에서 빈 값은 일반적으로 강수/적설 없음으로 해석 가능한 경우가 많음.
# 다만 전체 시간 row가 없는 경우도 포함될 수 있으므로 flag를 남긴 상태에서 0으로 대체.
weather["precipitation"] = weather["precipitation"].fillna(0)
weather["snowfall"] = weather["snowfall"].fillna(0)

# 기온/풍속/습도:
# 시간 연속성이 있으므로 선형보간 + 앞뒤 채움
for col in ["temperature", "wind_speed", "humidity"]:
    weather[col] = weather[col].interpolate(method="linear")
    weather[col] = weather[col].ffill().bfill()

# 음수 불가능한 값 보정
weather["precipitation"] = weather["precipitation"].clip(lower=0)
weather["snowfall"] = weather["snowfall"].clip(lower=0)
weather["wind_speed"] = weather["wind_speed"].clip(lower=0)
weather["humidity"] = weather["humidity"].clip(lower=0, upper=100)

# 타입 최적화
float_cols = ["temperature", "precipitation", "wind_speed", "humidity", "snowfall"]
for col in float_cols:
    weather[col] = weather[col].astype("float32")

flag_cols = [
    "weather_row_missing",
    "temperature_missing",
    "precipitation_missing",
    "wind_speed_missing",
    "humidity_missing",
    "snowfall_missing"
]

for col in flag_cols:
    weather[col] = weather[col].astype("int8")

print("결측 처리 후 결측:")
print(weather.isna().sum())

if weather.isna().sum().sum() > 0:
    raise ValueError("weather 결측 처리 후에도 결측이 남아 있습니다.")


# ------------------------------------------------------------
# 8. weather 최종 검증
# ------------------------------------------------------------
print_section("weather_verified 최종 검증")

check_duplicate_key(weather, ["datetime"], "weather_verified")
check_missing(weather, "weather_verified", top_n=30)

print("weather shape:", weather.shape)
print("weather 기간:", weather["datetime"].min(), "~", weather["datetime"].max())

print("\n연도별 row 수:")
print(weather["datetime"].dt.year.value_counts().sort_index())

print("\nweather describe:")
display(weather[float_cols].describe())

# 이상값 간단 점검
invalid_temp = (~weather["temperature"].between(-30, 45)).sum()
invalid_wind = (weather["wind_speed"] > 40).sum()
invalid_humidity = (~weather["humidity"].between(0, 100)).sum()
invalid_precip = (weather["precipitation"] < 0).sum()
invalid_snow = (weather["snowfall"] < 0).sum()

print("\n이상 범위 체크:")
print("temperature -30~45 밖:", int(invalid_temp))
print("wind_speed > 40:", int(invalid_wind))
print("humidity 0~100 밖:", int(invalid_humidity))
print("precipitation < 0:", int(invalid_precip))
print("snowfall < 0:", int(invalid_snow))

if invalid_humidity > 0 or invalid_precip > 0 or invalid_snow > 0:
    raise ValueError("weather에 명확한 이상값이 남아 있습니다.")

# 온도/풍속 극단값은 바로 제거하지 않고 보고만 함
# 서울 기상 데이터에서는 드물지만 관측값 자체일 수 있음


# ------------------------------------------------------------
# 9. 리포트 저장
# ------------------------------------------------------------
print_section("4단계 리포트 저장")

weather_raw_report_df = pd.DataFrame(weather_raw_reports)
weather_raw_report_df.to_csv(WEATHER_RAW_REPORT_PATH, index=False, encoding="utf-8-sig")

print("weather raw report 저장:", WEATHER_RAW_REPORT_PATH)
display(weather_raw_report_df)


# ------------------------------------------------------------
# 10. weather_verified 저장
# ------------------------------------------------------------
print_section("weather_verified 저장")

weather.to_parquet(WEATHER_OUTPUT_PATH, index=False)

print("저장 완료:", WEATHER_OUTPUT_PATH)
print("shape:", weather.shape)
print("size MB:", get_file_size_mb(WEATHER_OUTPUT_PATH))

display(weather.head(20))


# ------------------------------------------------------------
# 11. 최종 요약 저장
# ------------------------------------------------------------
print_section("4단계 최종 요약")

summary = {
    "step": "step4_clean_weather",
    "purpose": "hourly_rental 기간에 맞춘 시간 단위 weather feature 정제",
    "project_root": str(PROJECT_ROOT),

    "hourly_rental_path": str(HOURLY_RENTAL_PATH),
    "weather_output_path": str(WEATHER_OUTPUT_PATH),
    "weather_raw_report_path": str(WEATHER_RAW_REPORT_PATH),

    "weather_candidate_file_count": int(len(weather_candidates)),
    "weather_shape": str(weather.shape),
    "weather_datetime_min": str(weather["datetime"].min()),
    "weather_datetime_max": str(weather["datetime"].max()),

    "weather_duplicate_datetime_count": int(weather.duplicated(["datetime"]).sum()),
    "weather_total_missing_after_impute": int(weather.isna().sum().sum()),

    "weather_row_missing_count_before_impute": int(weather["weather_row_missing"].sum()),
    "temperature_missing_count_before_impute": int(weather["temperature_missing"].sum()),
    "precipitation_missing_count_before_impute": int(weather["precipitation_missing"].sum()),
    "wind_speed_missing_count_before_impute": int(weather["wind_speed_missing"].sum()),
    "humidity_missing_count_before_impute": int(weather["humidity_missing"].sum()),
    "snowfall_missing_count_before_impute": int(weather["snowfall_missing"].sum()),

    "invalid_temperature_range_count": int(invalid_temp),
    "invalid_wind_speed_range_count": int(invalid_wind),
    "invalid_humidity_range_count": int(invalid_humidity),
    "invalid_precipitation_count": int(invalid_precip),
    "invalid_snowfall_count": int(invalid_snow),

    "weather_columns": str(list(weather.columns)),

    "preprocessing_report_note": (
        "기상 데이터는 datetime을 시간 단위로 정규화한 뒤, hourly rental 데이터의 기간과 동일한 시간 grid로 재구성하였다. "
        "동일 datetime 중복 관측값은 평균으로 집계했으며, 기온/풍속/습도 결측은 시간순 선형보간 후 앞뒤 채움으로 처리하였다. "
        "강수량과 적설량 결측은 0으로 대체하되, 원래 결측 여부를 flag 컬럼으로 남겼다. "
        "최종 weather_verified 데이터는 datetime 기준 중복이 없고 결측치가 없도록 검증하였다."
    )
}

for k, v in summary.items():
    print(f"{k}: {v}")

with open(WEATHER_SUMMARY_PATH, "w", encoding="utf-8") as f:
    for k, v in summary.items():
        f.write(f"{k}: {v}\n")

print("\n4단계 요약 저장:", WEATHER_SUMMARY_PATH)

print("\n[다음 단계]")
print("1. 다음 단계에서는 holiday/calendar 데이터를 정제합니다.")
print("2. 이후 station별 관측 기간 기반 full hourly grid를 생성합니다.")
print("3. full grid에 hourly rental_count, station, weather, holiday를 merge합니다.")

## 6. 05 holiday calendar

Source: `05_holiday_calendar.ipynb`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import re
import gc

warnings.filterwarnings("ignore")

# ============================================================
# 5단계: holiday/calendar 데이터 정제
#
# 목적:
# - hourly_rental_verified 기간 기준 날짜 calendar 생성
# - 주말 여부, 공휴일 여부, 쉬는 날 여부 생성
# - raw holiday 파일이 있으면 사용
# - raw holiday 파일이 없으면 Python holidays 패키지 또는 내장 fallback 사용
# - 최종 holiday_verified.parquet 저장
#
# 입력:
# - data/processed_2/hourly_rental_verified.parquet
# - data/raw 아래 holiday/calendar/공휴일 관련 파일
#
# 출력:
# - data/processed_2/holiday_verified.parquet
# - data/interim_2/step5_holiday_summary.txt
# ============================================================


# ------------------------------------------------------------
# 0. 경로 설정
# ------------------------------------------------------------
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebook" else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim_2"
PROCESSED_DIR = DATA_DIR / "processed_2"

HOURLY_RENTAL_PATH = PROCESSED_DIR / "hourly_rental_verified.parquet"
HOLIDAY_OUTPUT_PATH = PROCESSED_DIR / "holiday_verified.parquet"
HOLIDAY_SUMMARY_PATH = INTERIM_DIR / "step5_holiday_summary.txt"
HOLIDAY_RAW_REPORT_PATH = INTERIM_DIR / "step5_holiday_raw_report.csv"

TARGET_YEARS = [2023, 2024, 2025]

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("[5단계 경로 확인]")
print("=" * 100)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_DIR:", RAW_DIR, "| exists:", RAW_DIR.exists())
print("HOURLY_RENTAL_PATH:", HOURLY_RENTAL_PATH, "| exists:", HOURLY_RENTAL_PATH.exists())
print("HOLIDAY_OUTPUT_PATH:", HOLIDAY_OUTPUT_PATH)

if not HOURLY_RENTAL_PATH.exists():
    raise FileNotFoundError("3단계 결과 hourly_rental_verified.parquet이 없습니다.")


# ------------------------------------------------------------
# 1. 유틸 함수
# ------------------------------------------------------------
def print_section(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def clean_colname(col):
    col = str(col)
    col = col.replace("\n", "")
    col = col.replace("\r", "")
    col = col.replace("\t", "")
    col = col.strip()
    col = re.sub(r"\s+", "", col)
    return col


def get_file_size_mb(path):
    path = Path(path)
    if not path.exists():
        return np.nan
    return path.stat().st_size / 1024 / 1024


def load_table_safely(path):
    path = Path(path)
    suffix = path.suffix.lower()

    if suffix == ".csv":
        try:
            return pd.read_csv(path, encoding="utf-8", low_memory=False)
        except UnicodeDecodeError:
            return pd.read_csv(path, encoding="cp949", low_memory=False)

    if suffix in [".xlsx", ".xls"]:
        return pd.read_excel(path)

    if suffix == ".parquet":
        return pd.read_parquet(path)

    raise ValueError(f"지원하지 않는 파일 형식입니다: {suffix}")


def find_col_by_keywords(columns, keywords):
    for col in columns:
        col_text = clean_colname(col).lower()
        for kw in keywords:
            kw_text = clean_colname(kw).lower()
            if kw_text in col_text:
                return col
    return None


def check_missing(df, name, top_n=30):
    print_section(f"[{name}] 결측 확인")
    result = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_rate_percent": (df.isna().mean() * 100).round(4)
    }).sort_values("missing_rate_percent", ascending=False)
    display(result.head(top_n))
    return result


def check_duplicate_key(df, keys, name):
    print_section(f"[{name}] key 중복 확인")
    dup_count = df.duplicated(keys).sum()
    print("keys:", keys)
    print("duplicated rows:", dup_count)

    if dup_count > 0:
        display(df[df.duplicated(keys, keep=False)].sort_values(keys).head(50))
        raise ValueError(f"{name}: key 중복이 있습니다.")

    return dup_count


# ------------------------------------------------------------
# 2. hourly rental 기간 확인
# ------------------------------------------------------------
print_section("hourly_rental_verified 기간 확인")

hourly_dt = pd.read_parquet(HOURLY_RENTAL_PATH, columns=["datetime"])
hourly_dt["datetime"] = pd.to_datetime(hourly_dt["datetime"])

hourly_start = hourly_dt["datetime"].min().floor("h")
hourly_end = hourly_dt["datetime"].max().floor("h")

calendar_start = hourly_start.normalize()
calendar_end = hourly_end.normalize()

print("hourly 기간:", hourly_start, "~", hourly_end)
print("calendar 기간:", calendar_start, "~", calendar_end)
print("연도별 hourly row 수:")
print(hourly_dt["datetime"].dt.year.value_counts().sort_index())

del hourly_dt
gc.collect()


# ------------------------------------------------------------
# 3. 기본 calendar 생성
# ------------------------------------------------------------
print_section("기본 calendar 생성")

calendar = pd.DataFrame({
    "date": pd.date_range(start=calendar_start, end=calendar_end, freq="D")
})

calendar["year"] = calendar["date"].dt.year.astype("int16")
calendar["month"] = calendar["date"].dt.month.astype("int8")
calendar["day"] = calendar["date"].dt.day.astype("int8")
calendar["day_of_week"] = calendar["date"].dt.dayofweek.astype("int8")
calendar["is_weekend"] = calendar["day_of_week"].isin([5, 6]).astype("int8")

print("calendar shape:", calendar.shape)
print("calendar 기간:", calendar["date"].min(), "~", calendar["date"].max())
display(calendar.head())


# ------------------------------------------------------------
# 4. raw holiday 파일 후보 탐색
# ------------------------------------------------------------
print_section("holiday/calendar 파일 후보 탐색")

all_raw_files = [p for p in RAW_DIR.rglob("*") if p.is_file()] if RAW_DIR.exists() else []

holiday_candidates = []

for p in all_raw_files:
    path_text = str(p).lower()

    is_supported = p.suffix.lower() in [".csv", ".xlsx", ".xls", ".parquet"]

    is_holiday_like = (
        "holiday" in path_text or
        "calendar" in path_text or
        "공휴" in path_text or
        "휴일" in path_text or
        "법정" in path_text
    )

    is_not_bike = (
        "bike_history" not in path_text and
        "대여이력" not in path_text and
        "station" not in path_text and
        "대여소" not in path_text and
        "weather" not in path_text and
        "날씨" not in path_text
    )

    if is_supported and is_holiday_like and is_not_bike:
        holiday_candidates.append(p)

print("holiday 후보 파일 수:", len(holiday_candidates))

for i, p in enumerate(holiday_candidates, start=1):
    print(f"{i}. {p.relative_to(PROJECT_ROOT)} | size MB: {get_file_size_mb(p):.3f}")


# ------------------------------------------------------------
# 5. raw holiday 파일이 있으면 로드
# ------------------------------------------------------------
print_section("raw holiday 파일 로드")

holiday_parts = []
holiday_raw_reports = []

for i, path in enumerate(holiday_candidates, start=1):
    print(f"\n[{i}/{len(holiday_candidates)}] 로드:", path.relative_to(PROJECT_ROOT))

    raw = load_table_safely(path)
    original_shape = raw.shape

    original_cols = list(raw.columns)
    cleaned_cols = [clean_colname(c) for c in original_cols]
    raw = raw.rename(columns=dict(zip(original_cols, cleaned_cols)))

    cols = list(raw.columns)

    print("shape:", original_shape)
    print("columns:", cols)

    date_col = find_col_by_keywords(
        cols,
        ["date", "날짜", "일자", "locdate", "기준일", "일시"]
    )

    name_col = find_col_by_keywords(
        cols,
        ["name", "명칭", "공휴일명", "휴일명", "dateName", "이름"]
    )

    holiday_flag_col = find_col_by_keywords(
        cols,
        ["isholiday", "is_holiday", "공휴일여부", "휴일여부", "holiday"]
    )

    print("선택 date_col:", date_col)
    print("선택 name_col:", name_col)
    print("선택 holiday_flag_col:", holiday_flag_col)

    if date_col is None:
        print("[주의] 날짜 컬럼을 찾지 못해 이 파일은 건너뜁니다.")
        continue

    part = pd.DataFrame()
    part["date"] = pd.to_datetime(raw[date_col], errors="coerce").dt.normalize()

    if name_col is not None:
        part["holiday_name"] = raw[name_col].astype(str).str.strip()
        part["holiday_name"] = part["holiday_name"].replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
    else:
        part["holiday_name"] = pd.NA

    if holiday_flag_col is not None:
        flag_raw = raw[holiday_flag_col].astype(str).str.strip().str.lower()
        part["is_holiday"] = flag_raw.isin(["1", "true", "y", "yes", "공휴일", "휴일"]).astype("int8")
        # 만약 flag가 전부 0으로 잡히면, 이 파일 자체가 공휴일 목록이라고 보고 1로 처리
        if part["is_holiday"].sum() == 0:
            part["is_holiday"] = 1
    else:
        part["is_holiday"] = 1

    part = part.dropna(subset=["date"]).copy()

    part = part[
        (part["date"] >= calendar_start) &
        (part["date"] <= calendar_end)
    ].copy()

    if len(part) > 0:
        holiday_parts.append(part)

    holiday_raw_reports.append({
        "path": str(path.relative_to(PROJECT_ROOT)),
        "raw_shape": str(original_shape),
        "date_col": str(date_col),
        "name_col": str(name_col),
        "holiday_flag_col": str(holiday_flag_col),
        "rows_in_calendar_period": int(len(part)),
        "min_date": str(part["date"].min()) if len(part) > 0 else "NaT",
        "max_date": str(part["date"].max()) if len(part) > 0 else "NaT"
    })

    display(part.head())

    del raw, part
    gc.collect()


# ------------------------------------------------------------
# 6. raw holiday가 없거나 부족하면 fallback 생성
# ------------------------------------------------------------
print_section("holiday fallback 생성 여부 판단")

use_fallback = len(holiday_parts) == 0

if use_fallback:
    print("raw holiday 파일에서 사용할 수 있는 공휴일 데이터를 찾지 못했습니다.")
    print("Python holidays 패키지 또는 내장 fallback으로 대한민국 공휴일을 생성합니다.")
else:
    print("raw holiday 파일에서 공휴일 데이터를 찾았습니다. raw holiday를 우선 사용합니다.")


def make_korea_holiday_fallback(start_date, end_date):
    """
    2023~2025 대한민국 공휴일 fallback.
    holidays 패키지가 있으면 우선 사용하고,
    없으면 프로젝트 기간에 필요한 주요 공휴일 날짜를 내장 목록으로 생성.
    """
    dates = []

    try:
        import holidays
        kr_holidays = holidays.KR(years=range(start_date.year, end_date.year + 1))

        for d, name in kr_holidays.items():
            ts = pd.Timestamp(d)
            if start_date <= ts <= end_date:
                dates.append({
                    "date": ts.normalize(),
                    "holiday_name": str(name),
                    "is_holiday": 1,
                    "holiday_source": "python_holidays_KR"
                })

        result = pd.DataFrame(dates)

        if len(result) > 0:
            return result

    except Exception as e:
        print("holidays 패키지 사용 불가 또는 실패:", e)

    # 내장 fallback: 2023~2025 프로젝트 기간용
    # 대체공휴일 포함
    manual_holidays = [
        # 2023
        ("2023-01-01", "신정"),
        ("2023-01-21", "설날연휴"),
        ("2023-01-22", "설날"),
        ("2023-01-23", "설날연휴"),
        ("2023-01-24", "설날 대체공휴일"),
        ("2023-03-01", "삼일절"),
        ("2023-05-05", "어린이날"),
        ("2023-05-27", "부처님오신날"),
        ("2023-05-29", "부처님오신날 대체공휴일"),
        ("2023-06-06", "현충일"),
        ("2023-08-15", "광복절"),
        ("2023-09-28", "추석연휴"),
        ("2023-09-29", "추석"),
        ("2023-09-30", "추석연휴"),
        ("2023-10-03", "개천절"),
        ("2023-10-09", "한글날"),
        ("2023-12-25", "성탄절"),

        # 2024
        ("2024-01-01", "신정"),
        ("2024-02-09", "설날연휴"),
        ("2024-02-10", "설날"),
        ("2024-02-11", "설날연휴"),
        ("2024-02-12", "설날 대체공휴일"),
        ("2024-03-01", "삼일절"),
        ("2024-04-10", "제22대 국회의원 선거일"),
        ("2024-05-05", "어린이날"),
        ("2024-05-06", "어린이날 대체공휴일"),
        ("2024-05-15", "부처님오신날"),
        ("2024-06-06", "현충일"),
        ("2024-08-15", "광복절"),
        ("2024-09-16", "추석연휴"),
        ("2024-09-17", "추석"),
        ("2024-09-18", "추석연휴"),
        ("2024-10-03", "개천절"),
        ("2024-10-09", "한글날"),
        ("2024-12-25", "성탄절"),

        # 2025
        ("2025-01-01", "신정"),
        ("2025-01-28", "설날연휴"),
        ("2025-01-29", "설날"),
        ("2025-01-30", "설날연휴"),
        ("2025-03-01", "삼일절"),
        ("2025-03-03", "삼일절 대체공휴일"),
        ("2025-05-05", "어린이날/부처님오신날"),
        ("2025-05-06", "대체공휴일"),
        ("2025-06-03", "제21대 대통령 선거일"),
        ("2025-06-06", "현충일"),
        ("2025-08-15", "광복절"),
        ("2025-10-03", "개천절"),
        ("2025-10-05", "추석연휴"),
        ("2025-10-06", "추석"),
        ("2025-10-07", "추석연휴"),
        ("2025-10-08", "추석 대체공휴일"),
        ("2025-10-09", "한글날"),
        ("2025-12-25", "성탄절"),
    ]

    result = pd.DataFrame(manual_holidays, columns=["date", "holiday_name"])
    result["date"] = pd.to_datetime(result["date"]).dt.normalize()
    result["is_holiday"] = 1
    result["holiday_source"] = "manual_fallback_2023_2025"

    result = result[
        (result["date"] >= start_date) &
        (result["date"] <= end_date)
    ].copy()

    return result


if use_fallback:
    holiday_raw = make_korea_holiday_fallback(calendar_start, calendar_end)
else:
    holiday_raw = pd.concat(holiday_parts, ignore_index=True)
    holiday_raw["holiday_source"] = "raw_file"

del holiday_parts
gc.collect()

print("holiday_raw shape:", holiday_raw.shape)
display(holiday_raw.head(30))


# ------------------------------------------------------------
# 7. holiday 중복 정리
# ------------------------------------------------------------
print_section("holiday 중복 정리")

holiday_raw["date"] = pd.to_datetime(holiday_raw["date"]).dt.normalize()
holiday_raw["is_holiday"] = holiday_raw["is_holiday"].fillna(1).astype("int8")

if "holiday_name" not in holiday_raw.columns:
    holiday_raw["holiday_name"] = pd.NA

if "holiday_source" not in holiday_raw.columns:
    holiday_raw["holiday_source"] = "unknown"

duplicate_date_count = holiday_raw["date"].duplicated().sum()
print("holiday_raw date 중복 row 수:", duplicate_date_count)

holiday_agg = (
    holiday_raw
    .groupby("date", as_index=False)
    .agg({
        "is_holiday": "max",
        "holiday_name": lambda x: "|".join(sorted(set([str(v) for v in x.dropna() if str(v) != "nan"]))),
        "holiday_source": lambda x: "|".join(sorted(set([str(v) for v in x.dropna()])))
    })
)

holiday_agg["holiday_name"] = holiday_agg["holiday_name"].replace({"": pd.NA})

print("holiday_agg shape:", holiday_agg.shape)
display(holiday_agg.head(30))

check_duplicate_key(holiday_agg, ["date"], "holiday_agg")


# ------------------------------------------------------------
# 8. calendar에 holiday merge
# ------------------------------------------------------------
print_section("calendar + holiday merge")

holiday = calendar.merge(
    holiday_agg,
    on="date",
    how="left",
    validate="one_to_one"
)

holiday["is_holiday"] = holiday["is_holiday"].fillna(0).astype("int8")
holiday["holiday_name"] = holiday["holiday_name"].fillna("")
holiday["holiday_source"] = holiday["holiday_source"].fillna("none")

# 쉬는 날 = 주말 또는 공휴일
holiday["is_day_off"] = (
    (holiday["is_weekend"] == 1) |
    (holiday["is_holiday"] == 1)
).astype("int8")

print("holiday shape:", holiday.shape)
display(holiday.head(20))

check_duplicate_key(holiday, ["date"], "holiday_verified")
check_missing(holiday, "holiday_verified", top_n=30)


# ------------------------------------------------------------
# 9. 최종 검증
# ------------------------------------------------------------
print_section("holiday_verified 최종 검증")

if holiday["date"].isna().sum() > 0:
    raise ValueError("holiday date 결측이 있습니다.")

if holiday.duplicated(["date"]).sum() > 0:
    raise ValueError("holiday date 중복이 있습니다.")

if holiday.isna().sum().sum() > 0:
    raise ValueError("holiday 데이터에 결측이 남아 있습니다.")

years_in_holiday = set(holiday["year"].unique().tolist())
if not years_in_holiday.issubset(set(TARGET_YEARS)):
    raise ValueError(f"대상 연도 외 데이터가 포함되어 있습니다: {years_in_holiday}")

print("기간:", holiday["date"].min(), "~", holiday["date"].max())
print("row 수:", len(holiday))
print("연도별 row 수:")
print(holiday.groupby("year").size())

print("\n연도별 공휴일 수:")
print(holiday.groupby("year")["is_holiday"].sum())

print("\n연도별 주말 수:")
print(holiday.groupby("year")["is_weekend"].sum())

print("\n연도별 쉬는 날 수:")
print(holiday.groupby("year")["is_day_off"].sum())

print("\n공휴일 목록:")
display(
    holiday[holiday["is_holiday"] == 1][
        ["date", "day_of_week", "is_weekend", "is_holiday", "is_day_off", "holiday_name", "holiday_source"]
    ]
)


# ------------------------------------------------------------
# 10. raw report 저장
# ------------------------------------------------------------
print_section("5단계 raw report 저장")

holiday_raw_report_df = pd.DataFrame(holiday_raw_reports)
holiday_raw_report_df.to_csv(HOLIDAY_RAW_REPORT_PATH, index=False, encoding="utf-8-sig")

print("holiday raw report 저장:", HOLIDAY_RAW_REPORT_PATH)

if len(holiday_raw_report_df) > 0:
    display(holiday_raw_report_df)
else:
    print("raw holiday 파일 사용 없음. fallback 사용.")


# ------------------------------------------------------------
# 11. holiday_verified 저장
# ------------------------------------------------------------
print_section("holiday_verified 저장")

holiday.to_parquet(HOLIDAY_OUTPUT_PATH, index=False)

print("저장 완료:", HOLIDAY_OUTPUT_PATH)
print("shape:", holiday.shape)
print("size MB:", get_file_size_mb(HOLIDAY_OUTPUT_PATH))

display(holiday.head(20))


# ------------------------------------------------------------
# 12. 최종 요약 저장
# ------------------------------------------------------------
print_section("5단계 최종 요약")

summary = {
    "step": "step5_clean_holiday_calendar",
    "purpose": "hourly_rental 기간에 맞춘 날짜 단위 holiday/calendar feature 생성",
    "project_root": str(PROJECT_ROOT),

    "hourly_rental_path": str(HOURLY_RENTAL_PATH),
    "holiday_output_path": str(HOLIDAY_OUTPUT_PATH),
    "holiday_raw_report_path": str(HOLIDAY_RAW_REPORT_PATH),

    "holiday_candidate_file_count": int(len(holiday_candidates)),
    "use_fallback": bool(use_fallback),
    "holiday_shape": str(holiday.shape),
    "holiday_date_min": str(holiday["date"].min()),
    "holiday_date_max": str(holiday["date"].max()),

    "holiday_duplicate_date_count": int(holiday.duplicated(["date"]).sum()),
    "holiday_total_missing": int(holiday.isna().sum().sum()),

    "holiday_count_total": int(holiday["is_holiday"].sum()),
    "weekend_count_total": int(holiday["is_weekend"].sum()),
    "day_off_count_total": int(holiday["is_day_off"].sum()),

    "holiday_count_by_year": str(holiday.groupby("year")["is_holiday"].sum().to_dict()),
    "weekend_count_by_year": str(holiday.groupby("year")["is_weekend"].sum().to_dict()),
    "day_off_count_by_year": str(holiday.groupby("year")["is_day_off"].sum().to_dict()),

    "holiday_columns": str(list(holiday.columns)),

    "preprocessing_report_note": (
        "대여량 데이터의 기간에 맞춰 날짜 단위 calendar를 생성하고, 주말 여부와 공휴일 여부를 결합해 is_day_off feature를 생성하였다. "
        "공휴일 데이터는 raw holiday 파일이 있으면 우선 사용하고, 없을 경우 대한민국 공휴일 fallback을 사용하도록 구성하였다. "
        "최종 holiday_verified 데이터는 date 기준 중복과 결측이 없도록 검증하였다."
    )
}

for k, v in summary.items():
    print(f"{k}: {v}")

with open(HOLIDAY_SUMMARY_PATH, "w", encoding="utf-8") as f:
    for k, v in summary.items():
        f.write(f"{k}: {v}\n")

print("\n5단계 요약 저장:", HOLIDAY_SUMMARY_PATH)

print("\n[다음 단계]")
print("1. 다음 단계에서는 station별 관측 기간 기반 full hourly grid를 생성합니다.")
print("2. 이후 full grid에 hourly rental_count를 merge하고 없는 시간은 0으로 채웁니다.")
print("3. 그 다음 station, weather, holiday를 merge해 최종 base dataset을 만듭니다.")

## 7. 06 process nan

Source: `06_process_nan.ipynb`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import gc
import shutil

warnings.filterwarnings("ignore")

# ============================================================
# 6단계: station별 관측 기간 기반 full hourly rental 생성
#
# 목적:
# - hourly_rental_verified.parquet에서 station별 first_seen / last_seen 계산
# - station별 관측 기간 안에서만 시간 단위 full grid 생성
# - 실제 대여가 없는 시간은 rental_count = 0으로 채움
# - station_id + datetime 중복 없음 검증
# - 대용량 처리를 위해 station batch 단위 parquet parts로 저장
#
# 입력:
# - data/processed_2/hourly_rental_verified.parquet
# - data/processed_2/station_verified_enriched.parquet
#
# 출력:
# - data/processed_2/full_hourly_rental_verified_parts/*.parquet
# - data/processed_2/station_life_verified.parquet
# - data/interim_2/step6_full_hourly_rental_part_report.csv
# - data/interim_2/step6_full_hourly_rental_summary.txt
# ============================================================


# ------------------------------------------------------------
# 0. 경로 설정
# ------------------------------------------------------------
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebook" else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim_2"
PROCESSED_DIR = DATA_DIR / "processed_2"

HOURLY_RENTAL_PATH = PROCESSED_DIR / "hourly_rental_verified.parquet"
STATION_ENRICHED_PATH = PROCESSED_DIR / "station_verified_enriched.parquet"

STATION_LIFE_OUTPUT_PATH = PROCESSED_DIR / "station_life_verified.parquet"

FULL_HOURLY_OUTPUT_DIR = PROCESSED_DIR / "full_hourly_rental_verified_parts"
FULL_HOURLY_PART_REPORT_PATH = INTERIM_DIR / "step6_full_hourly_rental_part_report.csv"
FULL_HOURLY_SUMMARY_PATH = INTERIM_DIR / "step6_full_hourly_rental_summary.txt"

TARGET_YEARS = [2023, 2024, 2025]

# 한 번에 처리할 station 수
# 메모리가 부족하면 50으로 줄이고, 여유 있으면 150~200까지 늘려도 됩니다.
STATION_BATCH_SIZE = 100

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("[6단계 경로 확인]")
print("=" * 100)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("HOURLY_RENTAL_PATH:", HOURLY_RENTAL_PATH, "| exists:", HOURLY_RENTAL_PATH.exists())
print("STATION_ENRICHED_PATH:", STATION_ENRICHED_PATH, "| exists:", STATION_ENRICHED_PATH.exists())
print("STATION_LIFE_OUTPUT_PATH:", STATION_LIFE_OUTPUT_PATH)
print("FULL_HOURLY_OUTPUT_DIR:", FULL_HOURLY_OUTPUT_DIR)

if not HOURLY_RENTAL_PATH.exists():
    raise FileNotFoundError("3단계 결과 hourly_rental_verified.parquet이 없습니다.")

if not STATION_ENRICHED_PATH.exists():
    raise FileNotFoundError("2.5단계 결과 station_verified_enriched.parquet이 없습니다.")


# ------------------------------------------------------------
# 1. 유틸 함수
# ------------------------------------------------------------
def print_section(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def get_dir_size_mb(path):
    path = Path(path)
    if not path.exists():
        return np.nan
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) / 1024 / 1024


def get_file_size_mb(path):
    path = Path(path)
    if not path.exists():
        return np.nan
    return path.stat().st_size / 1024 / 1024


def check_duplicate_key(df, keys, name):
    print_section(f"[{name}] key 중복 확인")
    dup_count = df.duplicated(keys).sum()
    print("keys:", keys)
    print("duplicated rows:", dup_count)

    if dup_count > 0:
        display(df[df.duplicated(keys, keep=False)].sort_values(keys).head(50))
        raise ValueError(f"{name}: key 중복이 있습니다.")

    return dup_count


def check_missing(df, name, top_n=30):
    print_section(f"[{name}] 결측 확인")
    result = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_rate_percent": (df.isna().mean() * 100).round(4)
    }).sort_values("missing_rate_percent", ascending=False)
    display(result.head(top_n))
    return result


def make_grid_for_station_batch(station_life_batch):
    grids = []

    for row in station_life_batch.itertuples(index=False):
        station_id = row.station_id
        start = row.grid_start
        end = row.grid_end

        if pd.isna(start) or pd.isna(end) or start > end:
            continue

        dt_range = pd.date_range(start=start, end=end, freq="h")

        temp = pd.DataFrame({
            "station_id": station_id,
            "datetime": dt_range
        })

        grids.append(temp)

    if len(grids) == 0:
        return pd.DataFrame(columns=["station_id", "datetime"])

    return pd.concat(grids, ignore_index=True)


# ------------------------------------------------------------
# 2. hourly_rental 로드
# ------------------------------------------------------------
print_section("hourly_rental_verified 로드")

hourly = pd.read_parquet(HOURLY_RENTAL_PATH)

hourly["station_id"] = hourly["station_id"].astype("Int64")
hourly["datetime"] = pd.to_datetime(hourly["datetime"]).dt.floor("h")
hourly["rental_count"] = pd.to_numeric(hourly["rental_count"], errors="coerce").astype("float32")

print("hourly shape:", hourly.shape)
print("hourly 기간:", hourly["datetime"].min(), "~", hourly["datetime"].max())
print("hourly station 수:", hourly["station_id"].nunique())

check_duplicate_key(hourly, ["station_id", "datetime"], "hourly_rental_verified")
check_missing(hourly, "hourly_rental_verified", top_n=20)

if hourly["rental_count"].isna().sum() > 0:
    raise ValueError("hourly rental_count 결측이 있습니다.")

if (hourly["rental_count"] <= 0).sum() > 0:
    raise ValueError("hourly_rental_verified는 실제 대여 발생만 집계한 데이터이므로 rental_count는 1 이상이어야 합니다.")

if (hourly["rental_count"] < 0).sum() > 0:
    raise ValueError("hourly rental_count 음수가 있습니다.")


# ------------------------------------------------------------
# 3. station_verified_enriched 로드
# ------------------------------------------------------------
print_section("station_verified_enriched 로드")

station = pd.read_parquet(STATION_ENRICHED_PATH)

station["station_id"] = station["station_id"].astype("Int64")

print("station shape:", station.shape)
print("station 수:", station["station_id"].nunique())
display(station.head())

if station["station_id"].isna().sum() > 0:
    raise ValueError("station_enriched station_id 결측이 있습니다.")

if station["station_id"].duplicated().sum() > 0:
    raise ValueError("station_enriched station_id 중복이 있습니다.")

if station["latitude"].isna().sum() > 0 or station["longitude"].isna().sum() > 0:
    raise ValueError("station_enriched 위치 결측이 있습니다.")

valid_station_ids = set(station["station_id"].dropna().astype(int).tolist())

hourly_station_ids = set(hourly["station_id"].dropna().astype(int).unique().tolist())

missing_in_station = hourly_station_ids - valid_station_ids

print("hourly station 수:", len(hourly_station_ids))
print("station_enriched station 수:", len(valid_station_ids))
print("hourly에는 있는데 station_enriched에는 없는 station 수:", len(missing_in_station))

if len(missing_in_station) > 0:
    print("예시:", sorted(list(missing_in_station))[:100])
    raise ValueError("hourly에 station_enriched에 없는 station이 남아 있습니다.")


# ------------------------------------------------------------
# 4. station_life 생성
# ------------------------------------------------------------
print_section("station_life 생성")

station_life = (
    hourly
    .groupby("station_id")["datetime"]
    .agg(first_seen="min", last_seen="max")
    .reset_index()
)

station_life["station_id"] = station_life["station_id"].astype("Int64")

print("station_life shape:", station_life.shape)
print("first_seen 전체 min:", station_life["first_seen"].min())
print("last_seen 전체 max:", station_life["last_seen"].max())

if station_life["station_id"].duplicated().sum() > 0:
    raise ValueError("station_life에 station_id 중복이 있습니다.")

# install_date 붙이기
station_install = station[["station_id", "install_date"]].copy()
station_install["install_date"] = pd.to_datetime(station_install["install_date"], errors="coerce")

station_life = station_life.merge(
    station_install,
    on="station_id",
    how="left",
    validate="one_to_one"
)

# grid_start 원칙:
# 실제 대여 이력이 존재하는 기간을 보존하기 위해 first_seen을 기준으로 사용.
# install_date는 station_age_days 계산용으로만 사용하고,
# full grid 생성 기준에는 사용하지 않는다.
station_life["grid_start"] = pd.to_datetime(station_life["first_seen"]).dt.floor("h")
station_life["grid_end"] = pd.to_datetime(station_life["last_seen"]).dt.floor("h")

# install_date가 first_seen보다 늦은 경우는 데이터 품질 이슈로 기록만 한다.
mask_install_after_first_seen = (
    station_life["install_date"].notna() &
    (station_life["install_date"] > station_life["first_seen"])
)

print("install_date가 first_seen보다 늦은 station 수:", int(mask_install_after_first_seen.sum()))

if mask_install_after_first_seen.sum() > 0:
    display(
        station_life.loc[
            mask_install_after_first_seen,
            ["station_id", "first_seen", "last_seen", "install_date", "grid_start", "grid_end"]
        ].head(50)
    )

station_life["grid_hours"] = (
    (station_life["grid_end"] - station_life["grid_start"]).dt.total_seconds() / 3600 + 1
).astype("int64")

invalid_life = station_life[
    station_life["grid_start"].isna() |
    station_life["grid_end"].isna() |
    (station_life["grid_start"] > station_life["grid_end"])
].copy()

print("install_date가 first_seen보다 늦어서 grid_start 조정된 station 수:", int(mask_install_after_first_seen.sum()))
print("invalid station_life 수:", len(invalid_life))

if len(invalid_life) > 0:
    display(invalid_life.head(50))
    raise ValueError("station_life에 invalid 기간이 있습니다.")

print("\ngrid_hours 요약:")
print(station_life["grid_hours"].describe())

print("\nstation_life 예시:")
display(station_life.head(20))

# 저장
station_life.to_parquet(STATION_LIFE_OUTPUT_PATH, index=False)
print("station_life 저장:", STATION_LIFE_OUTPUT_PATH)
print("station_life size MB:", get_file_size_mb(STATION_LIFE_OUTPUT_PATH))


# ------------------------------------------------------------
# 5. 기존 full hourly output 삭제 후 새로 생성
# ------------------------------------------------------------
print_section("기존 full hourly parts 정리")

if FULL_HOURLY_OUTPUT_DIR.exists():
    shutil.rmtree(FULL_HOURLY_OUTPUT_DIR)

FULL_HOURLY_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("출력 폴더 초기화 완료:", FULL_HOURLY_OUTPUT_DIR)


# ------------------------------------------------------------
# 6. station batch 단위 full grid 생성 + hourly merge
# ------------------------------------------------------------
print_section("station batch 단위 full_hourly 생성 시작")

station_life = station_life.sort_values("station_id").reset_index(drop=True)

station_ids = station_life["station_id"].dropna().tolist()
num_stations = len(station_ids)
num_batches = int(np.ceil(num_stations / STATION_BATCH_SIZE))

print("station 수:", num_stations)
print("batch size:", STATION_BATCH_SIZE)
print("batch 수:", num_batches)

part_reports = []

total_grid_rows = 0
total_rental_sum_from_full = 0.0
total_nonzero_rows = 0
total_zero_rows = 0

for batch_idx in range(num_batches):
    start_idx = batch_idx * STATION_BATCH_SIZE
    end_idx = min((batch_idx + 1) * STATION_BATCH_SIZE, num_stations)

    batch_station_ids = station_ids[start_idx:end_idx]

    print(f"\n[{batch_idx + 1}/{num_batches}] station index {start_idx}~{end_idx - 1}, station 수: {len(batch_station_ids)}")

    station_life_batch = station_life[
        station_life["station_id"].isin(batch_station_ids)
    ].copy()

    # grid 생성
    grid_batch = make_grid_for_station_batch(station_life_batch)

    grid_batch["station_id"] = grid_batch["station_id"].astype("Int64")
    grid_batch["datetime"] = pd.to_datetime(grid_batch["datetime"])

    grid_rows = len(grid_batch)

    if grid_rows == 0:
        print("grid rows 0. skip.")
        continue

    # 해당 station의 hourly만 추출
    hourly_batch = hourly[
        hourly["station_id"].isin(batch_station_ids)
    ][["station_id", "datetime", "rental_count"]].copy()

    hourly_batch["station_id"] = hourly_batch["station_id"].astype("Int64")
    hourly_batch["datetime"] = pd.to_datetime(hourly_batch["datetime"])

    # merge
    full_batch = grid_batch.merge(
        hourly_batch,
        on=["station_id", "datetime"],
        how="left",
        validate="one_to_one"
    )

    if len(full_batch) != grid_rows:
        raise ValueError("full_batch merge 후 row 수가 변했습니다.")

    full_batch["rental_count"] = full_batch["rental_count"].fillna(0).astype("float32")

    # 검증
    dup_count = full_batch.duplicated(["station_id", "datetime"]).sum()
    if dup_count > 0:
        display(full_batch[full_batch.duplicated(["station_id", "datetime"], keep=False)].head(50))
        raise ValueError("full_batch에 key 중복이 있습니다.")

    missing_count = full_batch[["station_id", "datetime", "rental_count"]].isna().sum().sum()
    if missing_count > 0:
        raise ValueError("full_batch에 결측이 있습니다.")

    negative_count = (full_batch["rental_count"] < 0).sum()
    if negative_count > 0:
        raise ValueError("full_batch rental_count 음수가 있습니다.")

    nonzero_rows = int((full_batch["rental_count"] > 0).sum())
    zero_rows = int((full_batch["rental_count"] == 0).sum())
    rental_sum = float(full_batch["rental_count"].sum())

    total_grid_rows += grid_rows
    total_nonzero_rows += nonzero_rows
    total_zero_rows += zero_rows
    total_rental_sum_from_full += rental_sum

    part_path = FULL_HOURLY_OUTPUT_DIR / f"part_{batch_idx + 1:04d}.parquet"

    # 타입 최적화
    full_batch["station_id"] = full_batch["station_id"].astype("Int64")
    full_batch["rental_count"] = full_batch["rental_count"].astype("float32")

    full_batch.to_parquet(part_path, index=False)

    report = {
        "part_index": batch_idx + 1,
        "part_path": str(part_path.relative_to(PROJECT_ROOT)),
        "station_count": int(len(batch_station_ids)),
        "station_id_min": int(min(batch_station_ids)),
        "station_id_max": int(max(batch_station_ids)),
        "grid_rows": int(grid_rows),
        "nonzero_rows": int(nonzero_rows),
        "zero_rows": int(zero_rows),
        "zero_ratio": float(zero_rows / grid_rows if grid_rows > 0 else np.nan),
        "rental_count_sum": float(rental_sum),
        "datetime_min": str(full_batch["datetime"].min()),
        "datetime_max": str(full_batch["datetime"].max()),
        "file_size_mb": float(get_file_size_mb(part_path))
    }

    part_reports.append(report)

    print("grid rows:", grid_rows)
    print("nonzero rows:", nonzero_rows)
    print("zero rows:", zero_rows)
    print("zero ratio:", round(report["zero_ratio"], 4))
    print("rental sum:", rental_sum)
    print("saved:", part_path.name, "| MB:", round(report["file_size_mb"], 3))

    del station_life_batch, grid_batch, hourly_batch, full_batch
    gc.collect()


# ------------------------------------------------------------
# 7. full hourly 전체 검증
# ------------------------------------------------------------
print_section("full_hourly 전체 검증")

part_report_df = pd.DataFrame(part_reports)

if len(part_report_df) == 0:
    raise ValueError("생성된 full hourly part가 없습니다.")

part_report_df.to_csv(FULL_HOURLY_PART_REPORT_PATH, index=False, encoding="utf-8-sig")

hourly_original_rental_sum = float(hourly["rental_count"].sum())
hourly_original_rows = int(len(hourly))

print("part report 저장:", FULL_HOURLY_PART_REPORT_PATH)
display(part_report_df.head())

print("\n전체 full grid rows:", total_grid_rows)
print("전체 nonzero rows:", total_nonzero_rows)
print("전체 zero rows:", total_zero_rows)
print("전체 zero ratio:", total_zero_rows / total_grid_rows if total_grid_rows > 0 else np.nan)
print("full hourly rental sum:", total_rental_sum_from_full)
print("original hourly rental sum:", hourly_original_rental_sum)
print("original hourly rows:", hourly_original_rows)

# rental sum 검증
if not np.isclose(total_rental_sum_from_full, hourly_original_rental_sum):
    raise ValueError(
        f"full hourly rental sum이 원본 hourly 합계와 다릅니다. "
        f"full={total_rental_sum_from_full}, original={hourly_original_rental_sum}"
    )

# nonzero row 수 검증
# 원본 hourly는 실제 대여 발생한 station-hour만 있으므로 full에서 rental_count > 0 row와 같아야 함
if total_nonzero_rows != hourly_original_rows:
    raise ValueError(
        f"full hourly nonzero row 수가 원본 hourly row 수와 다릅니다. "
        f"full_nonzero={total_nonzero_rows}, hourly_rows={hourly_original_rows}"
    )

# part 별 station 범위 확인
print("\npart report 요약:")
display(part_report_df.describe(include="all"))

print("\nfull hourly parts 폴더 크기 MB:", get_dir_size_mb(FULL_HOURLY_OUTPUT_DIR))


# ------------------------------------------------------------
# 8. 샘플 검증
# ------------------------------------------------------------
print_section("저장된 full hourly sample 검증")

sample_part_path = sorted(FULL_HOURLY_OUTPUT_DIR.glob("*.parquet"))[0]
sample = pd.read_parquet(sample_part_path)

print("sample part:", sample_part_path.name)
print("sample shape:", sample.shape)
display(sample.head(20))

check_duplicate_key(sample, ["station_id", "datetime"], "sample_full_hourly_part")
check_missing(sample, "sample_full_hourly_part", top_n=10)

if sample["rental_count"].isna().sum() > 0:
    raise ValueError("sample rental_count 결측이 있습니다.")

if (sample["rental_count"] < 0).sum() > 0:
    raise ValueError("sample rental_count 음수가 있습니다.")


# ------------------------------------------------------------
# 9. 최종 요약 저장
# ------------------------------------------------------------
print_section("6단계 최종 요약")

summary = {
    "step": "step6_make_full_hourly_rental",
    "purpose": "station별 관측 기간 기반 full hourly grid 생성 및 rental_count 0 채우기",
    "project_root": str(PROJECT_ROOT),

    "hourly_rental_path": str(HOURLY_RENTAL_PATH),
    "station_enriched_path": str(STATION_ENRICHED_PATH),
    "station_life_output_path": str(STATION_LIFE_OUTPUT_PATH),
    "full_hourly_output_dir": str(FULL_HOURLY_OUTPUT_DIR),
    "full_hourly_part_report_path": str(FULL_HOURLY_PART_REPORT_PATH),

    "station_batch_size": int(STATION_BATCH_SIZE),
    "station_count": int(num_stations),
    "part_count": int(len(part_report_df)),

    "hourly_original_rows": int(hourly_original_rows),
    "hourly_original_rental_sum": float(hourly_original_rental_sum),

    "full_hourly_grid_rows": int(total_grid_rows),
    "full_hourly_nonzero_rows": int(total_nonzero_rows),
    "full_hourly_zero_rows": int(total_zero_rows),
    "full_hourly_zero_ratio": float(total_zero_rows / total_grid_rows if total_grid_rows > 0 else np.nan),
    "full_hourly_rental_sum": float(total_rental_sum_from_full),

    "full_hourly_datetime_min": str(part_report_df["datetime_min"].min()),
    "full_hourly_datetime_max": str(part_report_df["datetime_max"].max()),

    "station_life_first_seen_min": str(station_life["first_seen"].min()),
    "station_life_last_seen_max": str(station_life["last_seen"].max()),
    "station_life_grid_start_min": str(station_life["grid_start"].min()),
    "station_life_grid_end_max": str(station_life["grid_end"].max()),

    "install_date_after_first_seen_adjusted_station_count": int(mask_install_after_first_seen.sum()),
    "invalid_station_life_count": int(len(invalid_life)),

    "full_hourly_parts_size_mb": float(get_dir_size_mb(FULL_HOURLY_OUTPUT_DIR)),

    "preprocessing_report_note": (
        "hourly_rental_verified는 실제 대여가 발생한 station-hour만 포함하므로, "
        "대여가 없는 시간을 0으로 학습시키기 위해 station별 관측 기간(first_seen~last_seen) 안에서만 full hourly grid를 생성하였다. "
        "전체 station × 전체 기간 방식은 신규/폐쇄 대여소의 운영 전후 시간을 0으로 잘못 포함할 수 있으므로 사용하지 않았다. "
        "full grid에 hourly rental_count를 병합한 뒤 결측 rental_count는 0으로 채웠으며, "
        "원본 hourly rental_count 합계와 full hourly의 rental_count 합계가 일치하는지 검증하였다."
    )
}

for k, v in summary.items():
    print(f"{k}: {v}")

with open(FULL_HOURLY_SUMMARY_PATH, "w", encoding="utf-8") as f:
    for k, v in summary.items():
        f.write(f"{k}: {v}\n")

print("\n6단계 요약 저장:", FULL_HOURLY_SUMMARY_PATH)

print("\n[다음 단계]")
print("1. 다음 단계에서는 full_hourly_rental_verified_parts에 station, weather, holiday를 merge해 base dataset을 만듭니다.")
print("2. base dataset 생성 단계에서 rack_count 결측은 중앙값으로 대체합니다.")
print("3. install_date 결측은 station_age_missing 플래그를 만들고 station_age_days를 대체합니다.")
print("4. 최종 base dataset 저장 전에 결측/중복/이상값 검증을 수행합니다.")

## 8. 07 base dataset

Source: `07_base_dataset.ipynb`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import gc
import shutil

warnings.filterwarnings("ignore")

# ============================================================
# 7단계: 최종 base dataset 생성
#
# 목적:
# - full_hourly_rental_verified_parts에 station/weather/holiday merge
# - 시간 feature 생성
# - rack_count 결측 대체
# - station_age_days 생성 및 결측/음수 처리
# - 최종 학습 데이터에 결측/중복/이상값이 없도록 검증
#
# 입력:
# - data/processed_2/full_hourly_rental_verified_parts/*.parquet
# - data/processed_2/station_verified_enriched.parquet
# - data/processed_2/weather_verified.parquet
# - data/processed_2/holiday_verified.parquet
#
# 출력:
# - data/processed_2/base_dataset_verified_parts/*.parquet
# - data/interim_2/step7_base_dataset_part_report.csv
# - data/interim_2/step7_base_dataset_summary.txt
# ============================================================


# ------------------------------------------------------------
# 0. 경로 설정
# ------------------------------------------------------------
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebook" else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
INTERIM_DIR = DATA_DIR / "interim_2"
PROCESSED_DIR = DATA_DIR / "processed_2"

FULL_HOURLY_DIR = PROCESSED_DIR / "full_hourly_rental_verified_parts"
STATION_ENRICHED_PATH = PROCESSED_DIR / "station_verified_enriched.parquet"
WEATHER_PATH = PROCESSED_DIR / "weather_verified.parquet"
HOLIDAY_PATH = PROCESSED_DIR / "holiday_verified.parquet"

BASE_OUTPUT_DIR = PROCESSED_DIR / "base_dataset_verified_parts"
BASE_PART_REPORT_PATH = INTERIM_DIR / "step7_base_dataset_part_report.csv"
BASE_SUMMARY_PATH = INTERIM_DIR / "step7_base_dataset_summary.txt"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("[7단계 경로 확인]")
print("=" * 100)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("FULL_HOURLY_DIR:", FULL_HOURLY_DIR, "| exists:", FULL_HOURLY_DIR.exists())
print("STATION_ENRICHED_PATH:", STATION_ENRICHED_PATH, "| exists:", STATION_ENRICHED_PATH.exists())
print("WEATHER_PATH:", WEATHER_PATH, "| exists:", WEATHER_PATH.exists())
print("HOLIDAY_PATH:", HOLIDAY_PATH, "| exists:", HOLIDAY_PATH.exists())
print("BASE_OUTPUT_DIR:", BASE_OUTPUT_DIR)

if not FULL_HOURLY_DIR.exists():
    raise FileNotFoundError("6단계 결과 full_hourly_rental_verified_parts 폴더가 없습니다.")

if not STATION_ENRICHED_PATH.exists():
    raise FileNotFoundError("station_verified_enriched.parquet이 없습니다.")

if not WEATHER_PATH.exists():
    raise FileNotFoundError("weather_verified.parquet이 없습니다.")

if not HOLIDAY_PATH.exists():
    raise FileNotFoundError("holiday_verified.parquet이 없습니다.")


# ------------------------------------------------------------
# 1. 유틸 함수
# ------------------------------------------------------------
def print_section(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def get_file_size_mb(path):
    path = Path(path)
    if not path.exists():
        return np.nan
    return path.stat().st_size / 1024 / 1024


def get_dir_size_mb(path):
    path = Path(path)
    if not path.exists():
        return np.nan
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) / 1024 / 1024


def check_duplicate_key(df, keys, name):
    dup_count = df.duplicated(keys).sum()
    if dup_count > 0:
        print(f"[{name}] duplicated rows:", dup_count)
        display(df[df.duplicated(keys, keep=False)].sort_values(keys).head(50))
        raise ValueError(f"{name}: key 중복이 있습니다.")
    return dup_count


def check_no_missing(df, name):
    missing = df.isna().sum()
    missing = missing[missing > 0]
    if len(missing) > 0:
        print(f"[{name}] 결측 컬럼:")
        print(missing.sort_values(ascending=False))
        raise ValueError(f"{name}: 결측치가 남아 있습니다.")
    return 0


def safe_left_merge(left, right, on, validate, name):
    before = len(left)
    out = left.merge(right, on=on, how="left", validate=validate)
    after = len(out)

    if before != after:
        raise ValueError(f"{name}: left merge인데 row 수가 변했습니다. before={before}, after={after}")

    return out


# ------------------------------------------------------------
# 2. 입력 데이터 로드
# ------------------------------------------------------------
print_section("입력 데이터 로드")

full_part_files = sorted(FULL_HOURLY_DIR.glob("*.parquet"))

print("full hourly part 파일 수:", len(full_part_files))

if len(full_part_files) == 0:
    raise FileNotFoundError("full hourly part parquet 파일이 없습니다.")

station = pd.read_parquet(STATION_ENRICHED_PATH)
weather = pd.read_parquet(WEATHER_PATH)
holiday = pd.read_parquet(HOLIDAY_PATH)

print("station shape:", station.shape)
print("weather shape:", weather.shape)
print("holiday shape:", holiday.shape)

display(station.head())
display(weather.head())
display(holiday.head())


# ------------------------------------------------------------
# 3. station feature 정리
# ------------------------------------------------------------
print_section("station feature 정리")

station["station_id"] = station["station_id"].astype("Int64")
station["latitude"] = pd.to_numeric(station["latitude"], errors="coerce")
station["longitude"] = pd.to_numeric(station["longitude"], errors="coerce")
station["rack_count"] = pd.to_numeric(station["rack_count"], errors="coerce")
station["install_date"] = pd.to_datetime(station["install_date"], errors="coerce")

if station["station_id"].isna().sum() > 0:
    raise ValueError("station station_id 결측이 있습니다.")

if station["station_id"].duplicated().sum() > 0:
    raise ValueError("station station_id 중복이 있습니다.")

if station["latitude"].isna().sum() > 0 or station["longitude"].isna().sum() > 0:
    raise ValueError("station 위치 결측이 있습니다.")

invalid_station_location = (
    ~(
        station["latitude"].between(37.3, 37.8) &
        station["longitude"].between(126.7, 127.3)
    )
).sum()

if invalid_station_location > 0:
    raise ValueError(f"station 위치 범위 이상이 있습니다: {invalid_station_location}")

# rack_count는 station_master로 보완된 station에서 결측일 수 있음.
valid_rack = station.loc[station["rack_count"].between(1, 200), "rack_count"]
rack_count_median = float(valid_rack.median())

print("rack_count_median:", rack_count_median)

station["rack_count_missing"] = station["rack_count"].isna().astype("int8")
station["rack_count_invalid"] = (
    station["rack_count"].notna() &
    ~station["rack_count"].between(1, 200)
).astype("int8")

station["rack_count"] = station["rack_count"].where(
    station["rack_count"].between(1, 200),
    np.nan
)

station["rack_count"] = station["rack_count"].fillna(rack_count_median)

# install_date 결측 여부는 station_age_missing으로 반영
station["station_age_missing"] = station["install_date"].isna().astype("int8")

# station_source flag
if "station_source" in station.columns:
    station["is_station_master_only"] = (station["station_source"].astype(str) == "station_master_only").astype("int8")
else:
    station["is_station_master_only"] = 0

if "needs_rack_count_impute" in station.columns:
    station["needs_rack_count_impute"] = station["needs_rack_count_impute"].fillna(0).astype("int8")
else:
    station["needs_rack_count_impute"] = station["rack_count_missing"].astype("int8")

if "needs_install_date_impute" in station.columns:
    station["needs_install_date_impute"] = station["needs_install_date_impute"].fillna(0).astype("int8")
else:
    station["needs_install_date_impute"] = station["station_age_missing"].astype("int8")

station_features = station[
    [
        "station_id",
        "latitude",
        "longitude",
        "rack_count",
        "install_date",
        "rack_count_missing",
        "rack_count_invalid",
        "station_age_missing",
        "is_station_master_only",
        "needs_rack_count_impute",
        "needs_install_date_impute"
    ]
].copy()

print("station_features shape:", station_features.shape)
display(station_features.head())

check_no_missing(
    station_features.drop(columns=["install_date"]),
    "station_features_without_install_date"
)


# ------------------------------------------------------------
# 4. weather feature 정리
# ------------------------------------------------------------
print_section("weather feature 정리")

weather["datetime"] = pd.to_datetime(weather["datetime"]).dt.floor("h")

if weather["datetime"].duplicated().sum() > 0:
    raise ValueError("weather datetime 중복이 있습니다.")

required_weather_cols = [
    "datetime",
    "temperature",
    "precipitation",
    "wind_speed",
    "humidity",
    "snowfall"
]

missing_weather_cols = [c for c in required_weather_cols if c not in weather.columns]
if missing_weather_cols:
    raise ValueError(f"weather 필수 컬럼이 없습니다: {missing_weather_cols}")

# flag 컬럼이 있으면 유지
weather_flag_cols = [
    c for c in [
        "weather_row_missing",
        "temperature_missing",
        "precipitation_missing",
        "wind_speed_missing",
        "humidity_missing",
        "snowfall_missing"
    ]
    if c in weather.columns
]

weather_features = weather[required_weather_cols + weather_flag_cols].copy()

if weather_features.isna().sum().sum() > 0:
    raise ValueError("weather_features에 결측이 있습니다.")

print("weather_features shape:", weather_features.shape)
print("weather 기간:", weather_features["datetime"].min(), "~", weather_features["datetime"].max())
display(weather_features.head())


# ------------------------------------------------------------
# 5. holiday feature 정리
# ------------------------------------------------------------
print_section("holiday feature 정리")

holiday["date"] = pd.to_datetime(holiday["date"]).dt.normalize()

if holiday["date"].duplicated().sum() > 0:
    raise ValueError("holiday date 중복이 있습니다.")

required_holiday_cols = [
    "date",
    "is_weekend",
    "is_holiday",
    "is_day_off"
]

missing_holiday_cols = [c for c in required_holiday_cols if c not in holiday.columns]
if missing_holiday_cols:
    raise ValueError(f"holiday 필수 컬럼이 없습니다: {missing_holiday_cols}")

holiday_features = holiday[required_holiday_cols].copy()

if holiday_features.isna().sum().sum() > 0:
    raise ValueError("holiday_features에 결측이 있습니다.")

for col in ["is_weekend", "is_holiday", "is_day_off"]:
    holiday_features[col] = holiday_features[col].astype("int8")

print("holiday_features shape:", holiday_features.shape)
print("holiday 기간:", holiday_features["date"].min(), "~", holiday_features["date"].max())
display(holiday_features.head())


# ------------------------------------------------------------
# 6. 기존 base output 삭제 후 새로 생성
# ------------------------------------------------------------
print_section("기존 base parts 정리")

if BASE_OUTPUT_DIR.exists():
    shutil.rmtree(BASE_OUTPUT_DIR)

BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("base output 폴더 초기화 완료:", BASE_OUTPUT_DIR)


# ------------------------------------------------------------
# 7. full hourly part별 base dataset 생성
# ------------------------------------------------------------
print_section("part별 base dataset 생성 시작")

part_reports = []

total_rows = 0
total_rental_sum = 0
total_zero_rows = 0
total_nonzero_rows = 0

global_min_dt = None
global_max_dt = None
all_station_ids = set()

for idx, part_path in enumerate(full_part_files, start=1):
    print(f"\n[{idx}/{len(full_part_files)}] 처리 중:", part_path.relative_to(PROJECT_ROOT))

    base = pd.read_parquet(part_path)

    raw_rows = len(base)

    base["station_id"] = base["station_id"].astype("Int64")
    base["datetime"] = pd.to_datetime(base["datetime"]).dt.floor("h")

    # rental_count는 count이므로 정수형으로 변환
    base["rental_count"] = pd.to_numeric(base["rental_count"], errors="coerce")

    if base["rental_count"].isna().sum() > 0:
        raise ValueError(f"{part_path.name}: rental_count 결측이 있습니다.")

    # float32 누적 오차 방지를 위해 round 후 int32
    fractional_count = ((base["rental_count"] % 1) != 0).sum()
    if fractional_count > 0:
        print("fractional rental_count 예시:")
        display(base.loc[(base["rental_count"] % 1) != 0].head())
        raise ValueError(f"{part_path.name}: rental_count에 정수가 아닌 값이 있습니다.")

    base["rental_count"] = base["rental_count"].round().astype("int32")

    if (base["rental_count"] < 0).sum() > 0:
        raise ValueError(f"{part_path.name}: rental_count 음수가 있습니다.")

    check_duplicate_key(base, ["station_id", "datetime"], f"{part_path.name}_before_merge")

    # 시간 feature
    base["date"] = base["datetime"].dt.normalize()
    base["year"] = base["datetime"].dt.year.astype("int16")
    base["month"] = base["datetime"].dt.month.astype("int8")
    base["day"] = base["datetime"].dt.day.astype("int8")
    base["hour"] = base["datetime"].dt.hour.astype("int8")
    base["day_of_week"] = base["datetime"].dt.dayofweek.astype("int8")

    # station merge
    base = safe_left_merge(
        base,
        station_features,
        on="station_id",
        validate="many_to_one",
        name=f"{part_path.name} + station"
    )

    # weather merge
    base = safe_left_merge(
        base,
        weather_features,
        on="datetime",
        validate="many_to_one",
        name=f"{part_path.name} + weather"
    )

    # holiday merge
    base = safe_left_merge(
        base,
        holiday_features,
        on="date",
        validate="many_to_one",
        name=f"{part_path.name} + holiday"
    )

    # station_age_days 계산
    base["station_age_days"] = (
        base["datetime"].dt.normalize() - base["install_date"].dt.normalize()
    ).dt.days

    # install_date 결측이면 station_age_days도 결측 -> indicator 유지 후 중앙값 대체
    # install_date가 first_seen보다 늦은 데이터는 음수가 될 수 있으므로 0으로 clip
    base["station_age_days"] = base["station_age_days"].clip(lower=0)

    # station_age_days median은 해당 part에서 계산하지 않고 전체 station 기준 median으로 계산하는 게 더 안정적
    # 여기서는 base 내 결측 제외 median 사용. 전체적으로 큰 차이는 없음.
    age_median = base["station_age_days"].median()

    if pd.isna(age_median):
        age_median = 0

    base["station_age_days"] = base["station_age_days"].fillna(age_median)

    # install_date는 모델 입력에 직접 쓰지 않으므로 제거
    base = base.drop(columns=["install_date"])

    # 최종 컬럼 정리
    final_cols = [
        "station_id",
        "datetime",
        "date",
        "year",
        "month",
        "day",
        "hour",
        "day_of_week",
        "is_weekend",
        "is_holiday",
        "is_day_off",

        "latitude",
        "longitude",
        "rack_count",
        "rack_count_missing",
        "rack_count_invalid",
        "station_age_days",
        "station_age_missing",
        "is_station_master_only",
        "needs_rack_count_impute",
        "needs_install_date_impute",

        "temperature",
        "precipitation",
        "wind_speed",
        "humidity",
        "snowfall",
    ]

    for col in weather_flag_cols:
        if col not in final_cols:
            final_cols.append(col)

    final_cols.append("rental_count")

    base = base[final_cols].copy()

    # dtype 최적화
    base["station_id"] = base["station_id"].astype("Int64")
    base["latitude"] = base["latitude"].astype("float32")
    base["longitude"] = base["longitude"].astype("float32")
    base["rack_count"] = base["rack_count"].astype("float32")
    base["station_age_days"] = base["station_age_days"].astype("float32")

    for col in [
        "rack_count_missing",
        "rack_count_invalid",
        "station_age_missing",
        "is_station_master_only",
        "needs_rack_count_impute",
        "needs_install_date_impute",
        "is_weekend",
        "is_holiday",
        "is_day_off"
    ] + weather_flag_cols:
        if col in base.columns:
            base[col] = base[col].astype("int8")

    for col in ["temperature", "precipitation", "wind_speed", "humidity", "snowfall"]:
        base[col] = base[col].astype("float32")

    base["rental_count"] = base["rental_count"].astype("int32")

    # 최종 검증
    check_duplicate_key(base, ["station_id", "datetime"], f"{part_path.name}_final_base")
    check_no_missing(base, f"{part_path.name}_final_base")

    if (base["rental_count"] < 0).sum() > 0:
        raise ValueError(f"{part_path.name}: 최종 rental_count 음수")

    invalid_loc = (
        ~(
            base["latitude"].between(37.3, 37.8) &
            base["longitude"].between(126.7, 127.3)
        )
    ).sum()

    if invalid_loc > 0:
        raise ValueError(f"{part_path.name}: 위치 범위 이상 row 존재 {invalid_loc}")

    output_path = BASE_OUTPUT_DIR / part_path.name
    base.to_parquet(output_path, index=False)

    part_rows = len(base)
    rental_sum = int(base["rental_count"].sum())
    zero_rows = int((base["rental_count"] == 0).sum())
    nonzero_rows = int((base["rental_count"] > 0).sum())

    total_rows += part_rows
    total_rental_sum += rental_sum
    total_zero_rows += zero_rows
    total_nonzero_rows += nonzero_rows

    min_dt = base["datetime"].min()
    max_dt = base["datetime"].max()

    if global_min_dt is None or min_dt < global_min_dt:
        global_min_dt = min_dt

    if global_max_dt is None or max_dt > global_max_dt:
        global_max_dt = max_dt

    all_station_ids.update(base["station_id"].dropna().astype(int).unique().tolist())

    part_reports.append({
        "part_index": idx,
        "input_part": str(part_path.relative_to(PROJECT_ROOT)),
        "output_part": str(output_path.relative_to(PROJECT_ROOT)),
        "rows": int(part_rows),
        "station_nunique": int(base["station_id"].nunique()),
        "datetime_min": str(min_dt),
        "datetime_max": str(max_dt),
        "rental_count_sum": int(rental_sum),
        "zero_rows": int(zero_rows),
        "nonzero_rows": int(nonzero_rows),
        "zero_ratio": float(zero_rows / part_rows if part_rows > 0 else np.nan),
        "missing_total": int(base.isna().sum().sum()),
        "duplicate_key_count": int(base.duplicated(["station_id", "datetime"]).sum()),
        "file_size_mb": float(get_file_size_mb(output_path))
    })

    print("rows:", part_rows)
    print("rental sum:", rental_sum)
    print("zero rows:", zero_rows)
    print("nonzero rows:", nonzero_rows)
    print("saved:", output_path.name, "| MB:", round(get_file_size_mb(output_path), 3))

    del base
    gc.collect()


# ------------------------------------------------------------
# 8. 전체 리포트 저장 및 검증
# ------------------------------------------------------------
print_section("base dataset 전체 검증")

part_report_df = pd.DataFrame(part_reports)
part_report_df.to_csv(BASE_PART_REPORT_PATH, index=False, encoding="utf-8-sig")

print("part report 저장:", BASE_PART_REPORT_PATH)
display(part_report_df.head())

print("전체 rows:", total_rows)
print("전체 rental_count sum:", total_rental_sum)
print("전체 zero rows:", total_zero_rows)
print("전체 nonzero rows:", total_nonzero_rows)
print("전체 zero ratio:", total_zero_rows / total_rows if total_rows > 0 else np.nan)
print("전체 station 수:", len(all_station_ids))
print("전체 기간:", global_min_dt, "~", global_max_dt)
print("base parts size MB:", get_dir_size_mb(BASE_OUTPUT_DIR))

print("\npart별 missing 합계:", int(part_report_df["missing_total"].sum()))
print("part별 duplicate key 합계:", int(part_report_df["duplicate_key_count"].sum()))

if part_report_df["missing_total"].sum() > 0:
    raise ValueError("base parts에 결측이 남아 있습니다.")

if part_report_df["duplicate_key_count"].sum() > 0:
    raise ValueError("base parts에 key 중복이 남아 있습니다.")


# ------------------------------------------------------------
# 9. 샘플 검증
# ------------------------------------------------------------
print_section("base dataset sample 검증")

sample_path = sorted(BASE_OUTPUT_DIR.glob("*.parquet"))[0]
sample = pd.read_parquet(sample_path)

print("sample path:", sample_path.name)
print("sample shape:", sample.shape)
display(sample.head(20))

check_duplicate_key(sample, ["station_id", "datetime"], "sample_base")
check_no_missing(sample, "sample_base")

print("sample columns:")
print(list(sample.columns))

del sample
gc.collect()


# ------------------------------------------------------------
# 10. 최종 요약 저장
# ------------------------------------------------------------
print_section("7단계 최종 요약")

summary = {
    "step": "step7_make_base_dataset",
    "purpose": "full hourly rental에 station/weather/holiday를 merge하여 최종 학습용 base dataset 생성",
    "project_root": str(PROJECT_ROOT),

    "full_hourly_dir": str(FULL_HOURLY_DIR),
    "station_enriched_path": str(STATION_ENRICHED_PATH),
    "weather_path": str(WEATHER_PATH),
    "holiday_path": str(HOLIDAY_PATH),
    "base_output_dir": str(BASE_OUTPUT_DIR),
    "base_part_report_path": str(BASE_PART_REPORT_PATH),

    "part_count": int(len(part_report_df)),
    "base_total_rows": int(total_rows),
    "base_station_nunique": int(len(all_station_ids)),
    "base_datetime_min": str(global_min_dt),
    "base_datetime_max": str(global_max_dt),

    "base_rental_count_sum": int(total_rental_sum),
    "base_zero_rows": int(total_zero_rows),
    "base_nonzero_rows": int(total_nonzero_rows),
    "base_zero_ratio": float(total_zero_rows / total_rows if total_rows > 0 else np.nan),

    "base_missing_total": int(part_report_df["missing_total"].sum()),
    "base_duplicate_key_total": int(part_report_df["duplicate_key_count"].sum()),
    "base_parts_size_mb": float(get_dir_size_mb(BASE_OUTPUT_DIR)),

    "rack_count_median_used_for_impute": float(rack_count_median),

    "base_columns": str(list(pd.read_parquet(sorted(BASE_OUTPUT_DIR.glob('*.parquet'))[0], nrows=1).columns)) if False else "see sample output",

    "preprocessing_report_note": (
        "full hourly rental 데이터에 station, weather, holiday 데이터를 병합하여 최종 학습용 base dataset을 생성하였다. "
        "station_id와 datetime을 핵심 key로 유지하고, 모든 merge는 many_to_one 기준으로 검증하였다. "
        "station_master로 보완된 대여소는 rack_count와 install_date가 없을 수 있어 rack_count는 중앙값으로 대체하고, "
        "station_age_days는 station_age_missing indicator와 함께 결측을 대체하였다. "
        "최종 base dataset은 결측치, 음수 target, 위치 범위 이상, station_id-datetime 중복이 없도록 검증하였다."
    )
}

for k, v in summary.items():
    print(f"{k}: {v}")

with open(BASE_SUMMARY_PATH, "w", encoding="utf-8") as f:
    for k, v in summary.items():
        f.write(f"{k}: {v}\n")

print("\n7단계 요약 저장:", BASE_SUMMARY_PATH)

print("\n[다음 단계]")
print("1. 다음 단계에서는 base_dataset_verified_parts를 train/valid/test로 시간 기준 분리합니다.")
print("2. 권장 분리: train=2023, valid=2024, test=2025")
print("3. 분리 후 baseline 모델과 LightGBM 모델을 다시 학습합니다.")

## 9. 08 train valid test

Source: `08_train_valid_test.ipynb`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import shutil
import gc

warnings.filterwarnings("ignore")

# ============================================================
# 8단계: train / valid / test 데이터 생성
#
# 목적:
# - base_dataset_verified_parts를 연도 기준으로 분리
# - Train: 2023
# - Valid: 2024
# - Test : 2025
# - 대용량 처리를 위해 각 split을 part parquet 형태로 저장
#
# 입력:
# - data/processed_2/base_dataset_verified_parts/*.parquet
#
# 출력:
# - data/processed_2/model_data/train_2023_parts/*.parquet
# - data/processed_2/model_data/valid_2024_parts/*.parquet
# - data/processed_2/model_data/test_2025_parts/*.parquet
# - data/interim_2/step8_split_part_report.csv
# - data/interim_2/step8_split_summary.txt
# ============================================================


# ------------------------------------------------------------
# 0. 경로 설정
# ------------------------------------------------------------
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebook" else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed_2"
INTERIM_DIR = DATA_DIR / "interim_2"

BASE_PARTS_DIR = PROCESSED_DIR / "base_dataset_verified_parts"

MODEL_DATA_DIR = PROCESSED_DIR / "model_data"

TRAIN_DIR = MODEL_DATA_DIR / "train_2023_parts"
VALID_DIR = MODEL_DATA_DIR / "valid_2024_parts"
TEST_DIR = MODEL_DATA_DIR / "test_2025_parts"

SPLIT_PART_REPORT_PATH = INTERIM_DIR / "step8_split_part_report.csv"
SPLIT_SUMMARY_PATH = INTERIM_DIR / "step8_split_summary.txt"

INTERIM_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 100)
print("[8단계 경로 확인]")
print("=" * 100)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("BASE_PARTS_DIR:", BASE_PARTS_DIR, "| exists:", BASE_PARTS_DIR.exists())
print("MODEL_DATA_DIR:", MODEL_DATA_DIR)

if not BASE_PARTS_DIR.exists():
    raise FileNotFoundError("base_dataset_verified_parts 폴더가 없습니다. 7단계를 먼저 완료하세요.")


# ------------------------------------------------------------
# 1. 유틸 함수
# ------------------------------------------------------------
def print_section(title):
    print("\n" + "=" * 100)
    print(title)
    print("=" * 100)


def get_dir_size_mb(path):
    path = Path(path)
    if not path.exists():
        return 0
    return sum(p.stat().st_size for p in path.rglob("*") if p.is_file()) / 1024 / 1024


def get_file_size_mb(path):
    path = Path(path)
    if not path.exists():
        return np.nan
    return path.stat().st_size / 1024 / 1024


def reset_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def check_split_part(df, name):
    # 결측 검증
    missing_total = int(df.isna().sum().sum())
    if missing_total > 0:
        print(f"[{name}] 결측 컬럼:")
        print(df.isna().sum()[df.isna().sum() > 0].sort_values(ascending=False))
        raise ValueError(f"{name}: 결측치가 있습니다.")

    # key 중복 검증
    dup_count = int(df.duplicated(["station_id", "datetime"]).sum())
    if dup_count > 0:
        display(
            df[df.duplicated(["station_id", "datetime"], keep=False)]
            .sort_values(["station_id", "datetime"])
            .head(50)
        )
        raise ValueError(f"{name}: station_id + datetime 중복이 있습니다.")

    # target 검증
    negative_target_count = int((df["rental_count"] < 0).sum())
    if negative_target_count > 0:
        raise ValueError(f"{name}: rental_count 음수가 있습니다.")

    # 위치 검증
    invalid_location = int(
        (
            ~(
                df["latitude"].between(37.3, 37.8) &
                df["longitude"].between(126.7, 127.3)
            )
        ).sum()
    )

    if invalid_location > 0:
        raise ValueError(f"{name}: 위치 범위 이상 row가 있습니다. count={invalid_location}")

    return {
        "missing_total": missing_total,
        "duplicate_key_count": dup_count,
        "negative_target_count": negative_target_count,
        "invalid_location_count": invalid_location
    }


# ------------------------------------------------------------
# 2. 출력 폴더 초기화
# ------------------------------------------------------------
print_section("출력 폴더 초기화")

reset_dir(TRAIN_DIR)
reset_dir(VALID_DIR)
reset_dir(TEST_DIR)

print("TRAIN_DIR:", TRAIN_DIR)
print("VALID_DIR:", VALID_DIR)
print("TEST_DIR:", TEST_DIR)


# ------------------------------------------------------------
# 3. base part 파일 확인
# ------------------------------------------------------------
print_section("base part 파일 확인")

base_part_files = sorted(BASE_PARTS_DIR.glob("*.parquet"))

print("base part 파일 수:", len(base_part_files))

if len(base_part_files) == 0:
    raise FileNotFoundError("base_dataset_verified_parts 안에 parquet 파일이 없습니다.")

for p in base_part_files[:10]:
    print(p.relative_to(PROJECT_ROOT))

if len(base_part_files) > 10:
    print("... 생략:", len(base_part_files) - 10)


# ------------------------------------------------------------
# 4. 연도별 split 생성
# ------------------------------------------------------------
print_section("연도별 train/valid/test split 생성 시작")

split_configs = {
    2023: {
        "name": "train",
        "dir": TRAIN_DIR,
        "prefix": "train_2023"
    },
    2024: {
        "name": "valid",
        "dir": VALID_DIR,
        "prefix": "valid_2024"
    },
    2025: {
        "name": "test",
        "dir": TEST_DIR,
        "prefix": "test_2025"
    }
}

split_stats = {
    2023: {
        "rows": 0,
        "station_ids": set(),
        "rental_sum": 0,
        "zero_rows": 0,
        "nonzero_rows": 0,
        "min_datetime": None,
        "max_datetime": None,
        "part_count": 0
    },
    2024: {
        "rows": 0,
        "station_ids": set(),
        "rental_sum": 0,
        "zero_rows": 0,
        "nonzero_rows": 0,
        "min_datetime": None,
        "max_datetime": None,
        "part_count": 0
    },
    2025: {
        "rows": 0,
        "station_ids": set(),
        "rental_sum": 0,
        "zero_rows": 0,
        "nonzero_rows": 0,
        "min_datetime": None,
        "max_datetime": None,
        "part_count": 0
    }
}

part_reports = []

total_base_rows = 0
total_split_rows = 0
total_base_rental_sum = 0
total_split_rental_sum = 0

for idx, base_path in enumerate(base_part_files, start=1):
    print(f"\n[{idx}/{len(base_part_files)}] 처리 중:", base_path.relative_to(PROJECT_ROOT))

    base = pd.read_parquet(base_path)

    # 기본 타입 정리
    base["datetime"] = pd.to_datetime(base["datetime"])
    base["year"] = base["year"].astype("int16")
    base["rental_count"] = pd.to_numeric(base["rental_count"], errors="coerce").astype("int32")

    base_rows = len(base)
    base_rental_sum = int(base["rental_count"].sum())

    total_base_rows += base_rows
    total_base_rental_sum += base_rental_sum

    years_in_part = sorted(base["year"].dropna().unique().tolist())

    print("base rows:", base_rows)
    print("years in part:", years_in_part)
    print("base rental sum:", base_rental_sum)

    for year, cfg in split_configs.items():
        split_df = base[base["year"] == year].copy()

        split_rows = len(split_df)

        if split_rows == 0:
            continue

        check_result = check_split_part(split_df, f"{cfg['prefix']}_from_{base_path.name}")

        output_path = cfg["dir"] / f"{cfg['prefix']}_part_{idx:04d}.parquet"

        split_df.to_parquet(output_path, index=False)

        rental_sum = int(split_df["rental_count"].sum())
        zero_rows = int((split_df["rental_count"] == 0).sum())
        nonzero_rows = int((split_df["rental_count"] > 0).sum())

        min_dt = split_df["datetime"].min()
        max_dt = split_df["datetime"].max()

        split_stats[year]["rows"] += split_rows
        split_stats[year]["station_ids"].update(split_df["station_id"].dropna().astype(int).unique().tolist())
        split_stats[year]["rental_sum"] += rental_sum
        split_stats[year]["zero_rows"] += zero_rows
        split_stats[year]["nonzero_rows"] += nonzero_rows
        split_stats[year]["part_count"] += 1

        if split_stats[year]["min_datetime"] is None or min_dt < split_stats[year]["min_datetime"]:
            split_stats[year]["min_datetime"] = min_dt

        if split_stats[year]["max_datetime"] is None or max_dt > split_stats[year]["max_datetime"]:
            split_stats[year]["max_datetime"] = max_dt

        total_split_rows += split_rows
        total_split_rental_sum += rental_sum

        part_reports.append({
            "input_base_part": str(base_path.relative_to(PROJECT_ROOT)),
            "split": cfg["name"],
            "year": int(year),
            "output_part": str(output_path.relative_to(PROJECT_ROOT)),
            "rows": int(split_rows),
            "station_nunique": int(split_df["station_id"].nunique()),
            "datetime_min": str(min_dt),
            "datetime_max": str(max_dt),
            "rental_count_sum": int(rental_sum),
            "zero_rows": int(zero_rows),
            "nonzero_rows": int(nonzero_rows),
            "zero_ratio": float(zero_rows / split_rows if split_rows > 0 else np.nan),
            "missing_total": int(check_result["missing_total"]),
            "duplicate_key_count": int(check_result["duplicate_key_count"]),
            "negative_target_count": int(check_result["negative_target_count"]),
            "invalid_location_count": int(check_result["invalid_location_count"]),
            "file_size_mb": float(get_file_size_mb(output_path))
        })

        print(f"  -> {cfg['name']} {year}: rows={split_rows:,}, rental_sum={rental_sum:,}, saved={output_path.name}")

        del split_df
        gc.collect()

    del base
    gc.collect()


# ------------------------------------------------------------
# 5. split report 저장
# ------------------------------------------------------------
print_section("split part report 저장")

part_report_df = pd.DataFrame(part_reports)
part_report_df.to_csv(SPLIT_PART_REPORT_PATH, index=False, encoding="utf-8-sig")

print("split part report 저장:", SPLIT_PART_REPORT_PATH)
display(part_report_df.head())

print("\npart report split별 요약:")
display(
    part_report_df
    .groupby(["split", "year"])
    .agg(
        rows=("rows", "sum"),
        station_nunique=("station_nunique", "max"),
        rental_count_sum=("rental_count_sum", "sum"),
        zero_rows=("zero_rows", "sum"),
        nonzero_rows=("nonzero_rows", "sum"),
        missing_total=("missing_total", "sum"),
        duplicate_key_count=("duplicate_key_count", "sum"),
        negative_target_count=("negative_target_count", "sum"),
        invalid_location_count=("invalid_location_count", "sum"),
        part_count=("output_part", "count"),
        file_size_mb=("file_size_mb", "sum")
    )
    .reset_index()
)


# ------------------------------------------------------------
# 6. 전체 split 검증
# ------------------------------------------------------------
print_section("전체 split 검증")

print("total_base_rows:", total_base_rows)
print("total_split_rows:", total_split_rows)
print("row diff:", total_base_rows - total_split_rows)

print("total_base_rental_sum:", total_base_rental_sum)
print("total_split_rental_sum:", total_split_rental_sum)
print("rental sum diff:", total_base_rental_sum - total_split_rental_sum)

if total_base_rows != total_split_rows:
    raise ValueError("base 전체 row 수와 split row 수 합계가 다릅니다.")

if total_base_rental_sum != total_split_rental_sum:
    raise ValueError("base rental_count 합계와 split rental_count 합계가 다릅니다.")

if part_report_df["missing_total"].sum() > 0:
    raise ValueError("split 데이터에 결측이 있습니다.")

if part_report_df["duplicate_key_count"].sum() > 0:
    raise ValueError("split 데이터에 중복 key가 있습니다.")

if part_report_df["negative_target_count"].sum() > 0:
    raise ValueError("split 데이터에 음수 target이 있습니다.")

if part_report_df["invalid_location_count"].sum() > 0:
    raise ValueError("split 데이터에 위치 범위 이상값이 있습니다.")

for year, cfg in split_configs.items():
    stats = split_stats[year]

    print("\n" + "-" * 80)
    print(f"{cfg['name'].upper()} {year}")
    print("-" * 80)
    print("rows:", stats["rows"])
    print("station_nunique:", len(stats["station_ids"]))
    print("rental_sum:", stats["rental_sum"])
    print("zero_rows:", stats["zero_rows"])
    print("nonzero_rows:", stats["nonzero_rows"])
    print("zero_ratio:", stats["zero_rows"] / stats["rows"] if stats["rows"] > 0 else np.nan)
    print("datetime_min:", stats["min_datetime"])
    print("datetime_max:", stats["max_datetime"])
    print("part_count:", stats["part_count"])
    print("dir_size_mb:", get_dir_size_mb(cfg["dir"]))

    if stats["rows"] == 0:
        raise ValueError(f"{cfg['name']} {year} split row 수가 0입니다.")


# ------------------------------------------------------------
# 7. 샘플 로드 검증
# ------------------------------------------------------------
print_section("샘플 로드 검증")

for year, cfg in split_configs.items():
    files = sorted(cfg["dir"].glob("*.parquet"))

    if len(files) == 0:
        raise FileNotFoundError(f"{cfg['name']} {year} part 파일이 없습니다.")

    sample = pd.read_parquet(files[0])

    print(f"\n[{cfg['name']} {year}] sample file:", files[0].name)
    print("shape:", sample.shape)
    print("datetime:", sample["datetime"].min(), "~", sample["datetime"].max())
    print("year unique:", sorted(sample["year"].unique().tolist()))
    print("missing total:", sample.isna().sum().sum())
    print("duplicate key:", sample.duplicated(["station_id", "datetime"]).sum())
    display(sample.head())

    if set(sample["year"].unique().tolist()) != {year}:
        raise ValueError(f"{cfg['name']} sample에 다른 연도가 포함되어 있습니다.")

    del sample
    gc.collect()


# ------------------------------------------------------------
# 8. 최종 요약 저장
# ------------------------------------------------------------
print_section("8단계 최종 요약")

summary = {
    "step": "step8_split_train_valid_test",
    "purpose": "base_dataset_verified_parts를 연도 기준 train/valid/test로 분리",
    "project_root": str(PROJECT_ROOT),

    "base_parts_dir": str(BASE_PARTS_DIR),
    "model_data_dir": str(MODEL_DATA_DIR),
    "train_dir": str(TRAIN_DIR),
    "valid_dir": str(VALID_DIR),
    "test_dir": str(TEST_DIR),
    "split_part_report_path": str(SPLIT_PART_REPORT_PATH),

    "split_rule": "train=2023, valid=2024, test=2025",

    "total_base_rows": int(total_base_rows),
    "total_split_rows": int(total_split_rows),
    "total_base_rental_sum": int(total_base_rental_sum),
    "total_split_rental_sum": int(total_split_rental_sum),

    "train_2023_rows": int(split_stats[2023]["rows"]),
    "train_2023_station_nunique": int(len(split_stats[2023]["station_ids"])),
    "train_2023_rental_sum": int(split_stats[2023]["rental_sum"]),
    "train_2023_zero_ratio": float(split_stats[2023]["zero_rows"] / split_stats[2023]["rows"]),
    "train_2023_datetime_min": str(split_stats[2023]["min_datetime"]),
    "train_2023_datetime_max": str(split_stats[2023]["max_datetime"]),
    "train_2023_part_count": int(split_stats[2023]["part_count"]),

    "valid_2024_rows": int(split_stats[2024]["rows"]),
    "valid_2024_station_nunique": int(len(split_stats[2024]["station_ids"])),
    "valid_2024_rental_sum": int(split_stats[2024]["rental_sum"]),
    "valid_2024_zero_ratio": float(split_stats[2024]["zero_rows"] / split_stats[2024]["rows"]),
    "valid_2024_datetime_min": str(split_stats[2024]["min_datetime"]),
    "valid_2024_datetime_max": str(split_stats[2024]["max_datetime"]),
    "valid_2024_part_count": int(split_stats[2024]["part_count"]),

    "test_2025_rows": int(split_stats[2025]["rows"]),
    "test_2025_station_nunique": int(len(split_stats[2025]["station_ids"])),
    "test_2025_rental_sum": int(split_stats[2025]["rental_sum"]),
    "test_2025_zero_ratio": float(split_stats[2025]["zero_rows"] / split_stats[2025]["rows"]),
    "test_2025_datetime_min": str(split_stats[2025]["min_datetime"]),
    "test_2025_datetime_max": str(split_stats[2025]["max_datetime"]),
    "test_2025_part_count": int(split_stats[2025]["part_count"]),

    "split_missing_total": int(part_report_df["missing_total"].sum()),
    "split_duplicate_key_total": int(part_report_df["duplicate_key_count"].sum()),
    "split_negative_target_total": int(part_report_df["negative_target_count"].sum()),
    "split_invalid_location_total": int(part_report_df["invalid_location_count"].sum()),

    "train_dir_size_mb": float(get_dir_size_mb(TRAIN_DIR)),
    "valid_dir_size_mb": float(get_dir_size_mb(VALID_DIR)),
    "test_dir_size_mb": float(get_dir_size_mb(TEST_DIR)),

    "preprocessing_report_note": (
        "최종 base dataset을 시간 누수 방지를 위해 연도 기준으로 분리하였다. "
        "2023년은 train, 2024년은 validation, 2025년은 test로 사용한다. "
        "각 split은 station_id와 datetime 조합 중복, 결측치, 음수 target, 위치 범위 이상값이 없도록 검증하였다."
    )
}

for k, v in summary.items():
    print(f"{k}: {v}")

with open(SPLIT_SUMMARY_PATH, "w", encoding="utf-8") as f:
    for k, v in summary.items():
        f.write(f"{k}: {v}\n")

print("\n8단계 요약 저장:", SPLIT_SUMMARY_PATH)

print("\n[다음 단계]")
print("1. 이제 baseline 모델을 다시 학습합니다.")
print("2. 먼저 train_2023 기준 평균 baseline을 만들고 valid_2024/test_2025에서 평가합니다.")
print("3. 그 다음 LightGBM 모델을 학습합니다.")

## 10. 13 data process

Source: `13_data_process.ipynb`

In [ ]:
# ============================================================
# 최종 학습용 parquet 데이터셋 생성 코드
#
# Input:
#   data/processed_2/model_data/train_2023_parts/*.parquet
#   data/processed_2/model_data/valid_2024_parts/*.parquet
#   data/processed_2/model_data/test_2025_parts/*.parquet
#
# Output:
#   data/processed_3/final_model_data/train_2023_parts/*.parquet
#   data/processed_3/final_model_data/valid_2024_parts/*.parquet
#   data/processed_3/final_model_data/test_2025_parts/*.parquet
#
# 최종 저장 컬럼:
#   28개 feature + rental_count
# ============================================================

import gc
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm


# ============================================================
# 0. 경로 설정
# ============================================================

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebook" else CURRENT_DIR

DATA_DIR = PROJECT_ROOT / "data"

INPUT_BASE_DIR = DATA_DIR / "processed_2" / "model_data"
OUTPUT_BASE_DIR = DATA_DIR / "processed_3" / "final_model_data"

TRAIN_DIR = INPUT_BASE_DIR / "train_2023_parts"
VALID_DIR = INPUT_BASE_DIR / "valid_2024_parts"
TEST_DIR = INPUT_BASE_DIR / "test_2025_parts"

OUT_TRAIN_DIR = OUTPUT_BASE_DIR / "train_2023_parts"
OUT_VALID_DIR = OUTPUT_BASE_DIR / "valid_2024_parts"
OUT_TEST_DIR = OUTPUT_BASE_DIR / "test_2025_parts"

TARGET = "rental_count"
MAX_LAG = 168


# ============================================================
# 1. 최종 피처 정의
# ============================================================

FINAL_FEATURE_COLS = [
    # station
    "station_id",
    "latitude",
    "longitude",
    "rack_count",
    "station_age_days",

    # time
    "month",
    "day",
    "hour",
    "dayofweek",
    "is_day_off",

    # cyclic
    "hour_sin",
    "hour_cos",

    # weather
    "temperature",
    "humidity",
    "wind_speed",
    "precipitation",

    # lag
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_24",
    "lag_48",
    "lag_168",

    # rolling
    "rolling_mean_3",
    "rolling_mean_6",
    "rolling_mean_24",
    "rolling_std_24",

    # diff
    "diff_1",
    "diff_24",
]

FINAL_SAVE_COLS = FINAL_FEATURE_COLS + [TARGET]


# 기존 parquet에서 읽어야 하는 원본 컬럼
# dayofweek는 datetime에서 새로 만들기 때문에 원본에서 안 읽어도 됨
RAW_COLS = [
    "station_id",
    "datetime",

    "latitude",
    "longitude",
    "rack_count",
    "station_age_days",

    "is_day_off",

    "temperature",
    "humidity",
    "wind_speed",
    "precipitation",

    TARGET,
]


# ============================================================
# 2. 폴더/로드 유틸
# ============================================================

def reset_output_dir(path: Path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def get_existing_columns_from_parquet(folder: Path):
    files = sorted(folder.glob("*.parquet"))

    if not files:
        raise FileNotFoundError(f"parquet 파일이 없습니다: {folder}")

    sample = pd.read_parquet(files[0])
    return sample.columns.tolist()


def load_parquet_parts(folder: Path, columns: list, name: str) -> pd.DataFrame:
    files = sorted(folder.glob("*.parquet"))

    if not files:
        raise FileNotFoundError(f"parquet 파일이 없습니다: {folder}")

    dfs = []

    for p in tqdm(files, desc=f"{name} 로드 중"):
        part = pd.read_parquet(p, columns=columns)
        dfs.append(part)

    df = pd.concat(dfs, ignore_index=True)

    del dfs
    gc.collect()

    print(f"{name} shape:", df.shape)

    return df


# ============================================================
# 3. 타입 최적화
# ============================================================

def optimize_raw_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["station_id"] = pd.to_numeric(df["station_id"], errors="coerce").astype("int32")
    df["datetime"] = pd.to_datetime(df["datetime"])

    int8_cols = [
        "is_day_off",
    ]

    float32_cols = [
        "latitude",
        "longitude",
        "rack_count",
        "station_age_days",
        "temperature",
        "humidity",
        "wind_speed",
        "precipitation",
    ]

    for col in int8_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int8")

    for col in float32_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")

    df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce").astype("float32")

    return df


# ============================================================
# 4. 최종 피처 생성
# ============================================================

def add_final_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    최종 피처 생성 함수.

    핵심 규칙:
    - station_id, datetime 기준 정렬
    - lag/rolling은 현재 시점 rental_count가 섞이지 않도록 shift(1) 사용
    - precipitation_missing은 사용하지 않음
    """

    df = df.copy()

    df["datetime"] = pd.to_datetime(df["datetime"])
    df = df.sort_values(["station_id", "datetime"]).reset_index(drop=True)

    # ----------------------------
    # 시간 피처 생성
    # ----------------------------
    df["month"] = df["datetime"].dt.month.astype("int8")
    df["day"] = df["datetime"].dt.day.astype("int8")
    df["hour"] = df["datetime"].dt.hour.astype("int8")
    df["dayofweek"] = df["datetime"].dt.dayofweek.astype("int8")

    if "is_day_off" not in df.columns:
        raise ValueError("is_day_off 컬럼이 없습니다. 기존 전처리에서 생성되어 있어야 합니다.")

    # ----------------------------
    # 시간 주기성 피처
    # ----------------------------
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24).astype("float32")
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24).astype("float32")

    # ----------------------------
    # 강수량 결측 처리
    # precipitation_missing은 제외하고,
    # precipitation 결측은 0으로 대체
    # ----------------------------
    df["precipitation"] = pd.to_numeric(df["precipitation"], errors="coerce").fillna(0).astype("float32")

    # ----------------------------
    # Lag / Rolling 피처
    # ----------------------------
    g = df.groupby("station_id", sort=False)[TARGET]

    df["lag_1"] = g.shift(1).astype("float32")
    df["lag_2"] = g.shift(2).astype("float32")
    df["lag_3"] = g.shift(3).astype("float32")
    df["lag_24"] = g.shift(24).astype("float32")
    df["lag_48"] = g.shift(48).astype("float32")
    df["lag_168"] = g.shift(168).astype("float32")

    shifted = g.shift(1)

    df["rolling_mean_3"] = (
        shifted.groupby(df["station_id"], sort=False)
        .rolling(3)
        .mean()
        .reset_index(level=0, drop=True)
        .astype("float32")
    )

    df["rolling_mean_6"] = (
        shifted.groupby(df["station_id"], sort=False)
        .rolling(6)
        .mean()
        .reset_index(level=0, drop=True)
        .astype("float32")
    )

    df["rolling_mean_24"] = (
        shifted.groupby(df["station_id"], sort=False)
        .rolling(24)
        .mean()
        .reset_index(level=0, drop=True)
        .astype("float32")
    )

    df["rolling_std_24"] = (
        shifted.groupby(df["station_id"], sort=False)
        .rolling(24)
        .std()
        .reset_index(level=0, drop=True)
        .astype("float32")
    )

    # ----------------------------
    # 변화량 피처
    # ----------------------------
    df["diff_1"] = (df["lag_1"] - df["lag_2"]).astype("float32")
    df["diff_24"] = (df["lag_1"] - df["lag_24"]).astype("float32")

    # ----------------------------
    # 최종 결측 처리
    # ----------------------------
    for col in FINAL_FEATURE_COLS:
        if col not in df.columns:
            raise ValueError(f"최종 피처가 생성되지 않았습니다: {col}")

    df[FINAL_FEATURE_COLS] = df[FINAL_FEATURE_COLS].fillna(0)

    return df


# ============================================================
# 5. valid/test lag 계산용 history 처리
# ============================================================

def get_history_tail(df: pd.DataFrame, n_tail: int = 168) -> pd.DataFrame:
    """
    valid/test 초반 lag 계산용 context.
    각 station_id별 마지막 168행만 사용.
    전체 train+valid+test를 합치지 않기 때문에 메모리 절약 가능.
    """

    hist = (
        df.sort_values(["station_id", "datetime"])
        .groupby("station_id", sort=False)
        .tail(n_tail)
        .copy()
    )

    return hist


def make_features_with_history(current_df: pd.DataFrame, history_df: pd.DataFrame = None) -> pd.DataFrame:
    """
    current_df의 lag 계산을 위해 history_df를 앞에 붙여 피처 생성.
    피처 생성 후에는 current_df 행만 반환.
    """

    current_df = current_df.copy()
    current_df["__is_current"] = np.uint8(1)

    if history_df is not None and len(history_df) > 0:
        history_df = history_df.copy()
        history_df["__is_current"] = np.uint8(0)

        temp = pd.concat([history_df, current_df], ignore_index=True)
    else:
        temp = current_df

    temp = add_final_features(temp)

    result = temp[temp["__is_current"] == 1].copy()
    result = result.drop(columns=["__is_current"])

    del temp
    gc.collect()

    return result


# ============================================================
# 6. parquet 저장
# ============================================================

def save_partitioned_parquet(df: pd.DataFrame, out_dir: Path, prefix: str, n_parts: int = 29):
    out_dir.mkdir(parents=True, exist_ok=True)

    df = df[FINAL_SAVE_COLS].copy()

    # 최종 타입 최적화
    int32_cols = ["station_id"]
    int8_cols = [
        "month",
        "day",
        "hour",
        "dayofweek",
        "is_day_off",
    ]

    for col in int32_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int32")

    for col in int8_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("int8")

    for col in FINAL_FEATURE_COLS:
        if col not in int32_cols + int8_cols:
            df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).astype("float32")

    df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce").fillna(0).astype("float32")

    splits = np.array_split(df, n_parts)

    for i, part in enumerate(tqdm(splits, desc=f"{prefix} 저장 중"), start=1):
        out_path = out_dir / f"{prefix}_part_{i:04d}.parquet"
        part.to_parquet(out_path, index=False)

    print(f"{prefix} 저장 완료:", out_dir)
    print(f"{prefix} shape:", df.shape)


# ============================================================
# 7. 실행
# ============================================================

print("=" * 100)
print("[최종 학습용 parquet 생성 시작]")
print("=" * 100)

print("INPUT_BASE_DIR :", INPUT_BASE_DIR)
print("OUTPUT_BASE_DIR:", OUTPUT_BASE_DIR)

if not TRAIN_DIR.exists():
    raise FileNotFoundError(f"TRAIN_DIR이 없습니다: {TRAIN_DIR}")

if not VALID_DIR.exists():
    raise FileNotFoundError(f"VALID_DIR이 없습니다: {VALID_DIR}")

if not TEST_DIR.exists():
    raise FileNotFoundError(f"TEST_DIR이 없습니다: {TEST_DIR}")

reset_output_dir(OUT_TRAIN_DIR)
reset_output_dir(OUT_VALID_DIR)
reset_output_dir(OUT_TEST_DIR)

# 실제 존재하는 컬럼 확인
existing_cols = get_existing_columns_from_parquet(TRAIN_DIR)
load_cols = [c for c in RAW_COLS if c in existing_cols]

required_cols = [
    "station_id",
    "datetime",
    "latitude",
    "longitude",
    "rack_count",
    "station_age_days",
    "is_day_off",
    "temperature",
    "humidity",
    "wind_speed",
    "precipitation",
    TARGET,
]

missing_required = [c for c in required_cols if c not in load_cols]

if missing_required:
    raise ValueError(f"원본 데이터에 필수 컬럼이 없습니다: {missing_required}")

print("\n[최종 피처]")
for c in FINAL_FEATURE_COLS:
    print(" -", c)

print("\n[로드 컬럼]")
for c in load_cols:
    print(" -", c)

print("\n최종 피처 수:", len(FINAL_FEATURE_COLS))
print("최종 저장 컬럼 수:", len(FINAL_SAVE_COLS))


# ----------------------------
# train 생성
# ----------------------------
print("\n" + "=" * 100)
print("[1] train_2023 생성")
print("=" * 100)

train_df = load_parquet_parts(TRAIN_DIR, load_cols, "train_2023")
train_df = optimize_raw_dtypes(train_df)

train_feat = make_features_with_history(train_df, history_df=None)

save_partitioned_parquet(
    train_feat,
    OUT_TRAIN_DIR,
    prefix="train_2023",
    n_parts=29
)

train_tail = get_history_tail(train_df, MAX_LAG)

del train_feat
gc.collect()


# ----------------------------
# valid 생성
# ----------------------------
print("\n" + "=" * 100)
print("[2] valid_2024 생성")
print("=" * 100)

valid_df = load_parquet_parts(VALID_DIR, load_cols, "valid_2024")
valid_df = optimize_raw_dtypes(valid_df)

valid_feat = make_features_with_history(valid_df, history_df=train_tail)

save_partitioned_parquet(
    valid_feat,
    OUT_VALID_DIR,
    prefix="valid_2024",
    n_parts=29
)

valid_tail = get_history_tail(valid_df, MAX_LAG)

del train_df, train_tail, valid_feat
gc.collect()


# ----------------------------
# test 생성
# ----------------------------
print("\n" + "=" * 100)
print("[3] test_2025 생성")
print("=" * 100)

test_df = load_parquet_parts(TEST_DIR, load_cols, "test_2025")
test_df = optimize_raw_dtypes(test_df)

test_feat = make_features_with_history(test_df, history_df=valid_tail)

save_partitioned_parquet(
    test_feat,
    OUT_TEST_DIR,
    prefix="test_2025",
    n_parts=29
)

del valid_df, valid_tail, test_df, test_feat
gc.collect()

print("\n" + "=" * 100)
print("[최종 학습용 parquet 생성 완료]")
print("=" * 100)

print("저장 위치:", OUTPUT_BASE_DIR)
print("최종 피처 수:", len(FINAL_FEATURE_COLS))
print("타겟:", TARGET)
print("최종 저장 컬럼 수:", len(FINAL_SAVE_COLS))